# Avaliação formal de desempenho do Sentinela

Este notebook avalia quantitativamente o desempenho do algoritmo do Sentinela utilizando um ground truth independente das regras estatísticas do próprio sistema.

## Objetivo

Comparar os sinais produzidos pelo Sentinela com evidências epidemiológicas externas e independentes.

A unidade de avaliação será:

`região × ano × semana epidemiológica`

## Ground truth

Cada semana poderá receber uma das seguintes classificações:

- `1` — evento epidemiológico confirmado por fonte externa;
- `0` — ausência de evento sustentada por evidência epidemiológica externa;
- `NA` — evidência insuficiente para classificação.

Semanas `NA` não serão utilizadas no cálculo das métricas de classificação.

## Princípio fundamental

O ground truth não utilizará:

- Q75 do Sentinela;
- mediana histórica do Sentinela;
- nowcast do Sentinela;
- estado `SIGNAL` ou `ALERT` do Sentinela.

Essas informações serão incorporadas somente após a construção independente dos rótulos externos.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

In [44]:
signal_sp_2026_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2026_sp_signal_evaluation.csv"
)

assert signal_sp_2026_path.exists(), (
    f"Arquivo não encontrado: {signal_sp_2026_path}"
)

signal_evaluation_sp_2026 = pd.read_csv(
    signal_sp_2026_path,
    sep=";",
)

print(
    "Arquivo carregado:",
    signal_sp_2026_path.name,
)

print(
    "Linhas:",
    len(signal_evaluation_sp_2026),
)

display(
    signal_evaluation_sp_2026.head()
)

Arquivo carregado: srag_2026_sp_signal_evaluation.csv
Linhas: 52


,epi_week,epi_year,week_start,week_end,record_count,region,calendar_status,data_status,cumulative_record_count,historical_median,q25,q75,historical_years,has_minimum_history,median_threshold,ratio_to_median,above_q75,above_factor_threshold,signal_status,signal_reason
0,1,2026,2026-01-04,2026-01-10,647.0,SP,encerrada,observado,647.0,927.0,423.5,5111.0,7,True,1390.5,0.697950,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
1,2,2026,2026-01-11,2026-01-17,541.0,SP,encerrada,observado,1188.0,817.0,338.0,5707.5,7,True,1225.5,0.662179,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
2,3,2026,2026-01-18,2026-01-24,565.0,SP,encerrada,observado,1753.0,812.0,351.0,5273.0,7,True,1218.0,0.695813,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
3,4,2026,2026-01-25,2026-01-31,491.0,SP,encerrada,observado,2244.0,791.0,394.0,4530.0,7,True,1186.5,0.620733,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
4,5,2026,2026-02-01,2026-02-07,703.0,SP,encerrada,observado,2947.0,913.0,401.0,3836.0,7,True,1369.5,0.769989,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.


In [47]:
signal_mg_2025_path = (
    project_root
    / "data"
    / "processed"
    / "srag_2025_mg_signal_evaluation.csv"
)

assert signal_mg_2025_path.exists(), (
    f"Arquivo não encontrado: {signal_mg_2025_path}"
)

signal_evaluation_2025 = pd.read_csv(
    signal_mg_2025_path,
    sep=";",
)

print(
    "Arquivo carregado:",
    signal_mg_2025_path.name,
)

print(
    "Linhas:",
    len(signal_evaluation_2025),
)

display(
    signal_evaluation_2025.head()
)

Arquivo carregado: srag_2025_mg_signal_evaluation.csv
Linhas: 53


,epi_week,epi_year,week_start,week_end,record_count,cumulative_record_count,calendar_status,data_status,historical_median,q25,q75,historical_years,has_minimum_history,region,median_threshold,ratio_to_median,above_q75,above_factor_threshold,signal_status,signal_reason
0,1,2025,2024-12-29,2025-01-04,397,397,encerrada,observado,586.0,111.00,2993.75,6.0,True,MG,879.00,0.677474,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
1,2,2025,2025-01-05,2025-01-11,422,819,encerrada,observado,471.5,105.00,3472.00,6.0,True,MG,707.25,0.895016,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
2,3,2025,2025-01-12,2025-01-18,413,1232,encerrada,observado,427.0,129.75,3170.75,6.0,True,MG,640.50,0.967213,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
3,4,2025,2025-01-19,2025-01-25,412,1644,encerrada,observado,363.0,105.25,2947.25,6.0,True,MG,544.50,1.134986,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.
4,5,2025,2025-01-26,2025-02-01,359,2003,encerrada,observado,433.5,130.50,2613.75,6.0,True,MG,650.25,0.828143,False,False,NO_SIGNAL,Critérios de aumento incomum não atendidos.


In [14]:
def build_epi_calendar(
    region: str,
    year: int,
    first_week_start: str,
    weeks: int = 52,
) -> pd.DataFrame:
    calendar = pd.DataFrame(
        {
            "epi_week": range(1, weeks + 1),
        }
    )

    calendar["region"] = region
    calendar["year"] = year

    first_start = pd.Timestamp(
        first_week_start
    )

    calendar["week_start"] = (
        first_start
        + pd.to_timedelta(
            (calendar["epi_week"] - 1) * 7,
            unit="D",
        )
    )

    calendar["week_end"] = (
        calendar["week_start"]
        + pd.Timedelta(days=6)
    )

    return calendar[
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]

In [15]:
ground_truth_base = pd.concat(
    [
        build_epi_calendar(
            region="MG",
            year=2025,
            first_week_start="2024-12-29",
            weeks=52,
        ),
        build_epi_calendar(
            region="SP",
            year=2026,
            first_week_start="2026-01-04",
            weeks=52,
        ),
    ],
    ignore_index=True,
)

display(ground_truth_base.head())

,region,year,epi_week,week_start,week_end
0,MG,2025,1,2024-12-29,2025-01-04
1,MG,2025,2,2025-01-05,2025-01-11
2,MG,2025,3,2025-01-12,2025-01-18
3,MG,2025,4,2025-01-19,2025-01-25
4,MG,2025,5,2025-01-26,2025-02-01


In [16]:
ground_truth = (
    ground_truth_base.copy()
)

ground_truth["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth.index,
    dtype="Int64",
)

ground_truth["evidence_level"] = pd.NA

ground_truth["event_type"] = pd.NA

ground_truth["evidence_summary"] = pd.NA

ground_truth["source_count"] = 0

ground_truth["source_ids"] = pd.NA

In [17]:
EVIDENCE_LEVELS = {
    "STRONG",
    "MODERATE",
    "WEAK",
    "UNLABELED",
}

In [18]:
external_events = pd.DataFrame(
    [
        {
            "event_id": "MG2025_001",
            "region": "MG",
            "event_date": "2025-05-02",
            "event_type": "STATE_EMERGENCY",
            "source_type": "government",
            "source_name": "Governo de Minas Gerais",
            "description": (
                "Situação de emergência em saúde pública "
                "diante do aumento de doenças respiratórias."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_002",
            "region": "MG",
            "event_date": "2025-05-02",
            "event_type": "LEGAL_DECREE",
            "source_type": "legislation",
            "source_name": (
                "Assembleia Legislativa de Minas Gerais"
            ),
            "description": (
                "Decreto com Numeração Especial nº 411."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_003",
            "region": "MG",
            "event_date": "2025-05-06",
            "event_type": "MUNICIPAL_EMERGENCY",
            "source_type": "government",
            "source_name": "Prefeitura de Uberlândia",
            "description": (
                "Situação de emergência devido ao aumento "
                "de casos de SRAG."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_004",
            "region": "MG",
            "event_date": "2025-05-08",
            "event_type": "EMERGENCY_OPERATIONS",
            "source_type": "government",
            "source_name": "Governo de Minas Gerais",
            "description": (
                "Implementação do Centro de Operações "
                "de Emergência em Saúde por SRAG."
            ),
            "evidence_strength": "STRONG",
        },
        {
            "event_id": "MG2025_005",
            "region": "MG",
            "event_date": "2025-05-21",
            "event_type": "RESPONSE_PLAN",
            "source_type": "government",
            "source_name": "Prefeitura de Uberlândia",
            "description": (
                "Plano de ação para enfrentamento "
                "das síndromes respiratórias."
            ),
            "evidence_strength": "MODERATE",
        },
    ]
)

external_events["event_date"] = pd.to_datetime(
    external_events["event_date"]
)

display(external_events)

,event_id,region,event_date,event_type,source_type,source_name,description,evidence_strength
0,MG2025_001,MG,2025-05-02,STATE_EMERGENCY,government,Governo de Minas Gerais,Situação de emergência em saúde pública diante...,STRONG
1,MG2025_002,MG,2025-05-02,LEGAL_DECREE,legislation,Assembleia Legislativa de Minas Gerais,Decreto com Numeração Especial nº 411.,STRONG
2,MG2025_003,MG,2025-05-06,MUNICIPAL_EMERGENCY,government,Prefeitura de Uberlândia,Situação de emergência devido ao aumento de ca...,STRONG
3,MG2025_004,MG,2025-05-08,EMERGENCY_OPERATIONS,government,Governo de Minas Gerais,Implementação do Centro de Operações de Emergê...,STRONG
4,MG2025_005,MG,2025-05-21,RESPONSE_PLAN,government,Prefeitura de Uberlândia,Plano de ação para enfrentamento das síndromes...,MODERATE


In [21]:
def assign_epi_week(
    event_row: pd.Series,
    calendar: pd.DataFrame,
):
    candidates = calendar.loc[
        calendar["region"].eq(
            event_row["region"]
        )
        & (
            calendar["week_start"]
            <= event_row["event_date"]
        )
        & (
            calendar["week_end"]
            >= event_row["event_date"]
        )
    ]

    if candidates.empty:
        return pd.NA

    return int(
        candidates.iloc[0]["epi_week"]
    )

In [22]:
external_events["epi_week"] = (
    external_events.apply(
        lambda row: assign_epi_week(
            row,
            ground_truth_base,
        ),
        axis=1,
    )
)

display(
    external_events[
        [
            "event_id",
            "event_date",
            "region",
            "epi_week",
            "event_type",
        ]
    ]
)

,event_id,event_date,region,epi_week,event_type
0,MG2025_001,2025-05-02,MG,18,STATE_EMERGENCY
1,MG2025_002,2025-05-02,MG,18,LEGAL_DECREE
2,MG2025_003,2025-05-06,MG,19,MUNICIPAL_EMERGENCY
3,MG2025_004,2025-05-08,MG,19,EMERGENCY_OPERATIONS
4,MG2025_005,2025-05-21,MG,21,RESPONSE_PLAN


In [23]:
event_group_map = {
    "MG2025_001": "MG2025_EMERGENCY_STATE",
    "MG2025_002": "MG2025_EMERGENCY_STATE",
    "MG2025_003": "MG2025_UBERLANDIA_EMERGENCY",
    "MG2025_004": "MG2025_COE_ACTIVATION",
    "MG2025_005": "MG2025_UBERLANDIA_RESPONSE_PLAN",
}

external_events["event_group_id"] = (
    external_events["event_id"]
    .map(event_group_map)
)

In [24]:
assert (
    external_events["event_group_id"]
    .notna()
    .all()
)

display(
    external_events[
        [
            "event_id",
            "event_group_id",
            "epi_week",
            "event_type",
            "evidence_strength",
        ]
    ]
)

,event_id,event_group_id,epi_week,event_type,evidence_strength
0,MG2025_001,MG2025_EMERGENCY_STATE,18,STATE_EMERGENCY,STRONG
1,MG2025_002,MG2025_EMERGENCY_STATE,18,LEGAL_DECREE,STRONG
2,MG2025_003,MG2025_UBERLANDIA_EMERGENCY,19,MUNICIPAL_EMERGENCY,STRONG
3,MG2025_004,MG2025_COE_ACTIVATION,19,EMERGENCY_OPERATIONS,STRONG
4,MG2025_005,MG2025_UBERLANDIA_RESPONSE_PLAN,21,RESPONSE_PLAN,MODERATE


In [25]:
evidence_rank = {
    "WEAK": 1,
    "MODERATE": 2,
    "STRONG": 3,
}

external_events["evidence_rank"] = (
    external_events["evidence_strength"]
    .map(evidence_rank)
)

In [26]:
weekly_external_evidence = (
    external_events
    .groupby(
        [
            "region",
            "epi_week",
        ],
        as_index=False,
    )
    .agg(
        independent_events=(
            "event_group_id",
            "nunique",
        ),
        source_count=(
            "event_id",
            "nunique",
        ),
        max_evidence_rank=(
            "evidence_rank",
            "max",
        ),
        source_ids=(
            "event_id",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
        event_group_ids=(
            "event_group_id",
            lambda values: ",".join(
                sorted(set(values))
            ),
        ),
        evidence_summary=(
            "description",
            lambda values: " | ".join(
                dict.fromkeys(values)
            ),
        ),
    )
)

In [27]:
rank_to_evidence = {
    1: "WEAK",
    2: "MODERATE",
    3: "STRONG",
}

weekly_external_evidence[
    "evidence_level"
] = (
    weekly_external_evidence[
        "max_evidence_rank"
    ]
    .map(rank_to_evidence)
)

In [28]:
display(
    weekly_external_evidence[
        [
            "region",
            "epi_week",
            "independent_events",
            "source_count",
            "evidence_level",
            "event_group_ids",
        ]
    ]
)

,region,epi_week,independent_events,source_count,evidence_level,event_group_ids
0,MG,18,1,2,STRONG,MG2025_EMERGENCY_STATE
1,MG,19,2,2,STRONG,"MG2025_COE_ACTIVATION,MG2025_UBERLANDIA_EMERGENCY"
2,MG,21,1,1,MODERATE,MG2025_UBERLANDIA_RESPONSE_PLAN


In [29]:
ground_truth = (
    ground_truth_base
    .merge(
        weekly_external_evidence,
        on=[
           "region",
           "epi_week", 
        ],
        how="left",
    )
)

In [30]:
ground_truth["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth.index,
    dtype="Int64",
)

In [31]:
positive_mask = (
    ground_truth["evidence_level"]
    .isin(
        [
           "STRONG",
           "MODERATE", 
        ]
    )
)

ground_truth.loc[
    positive_mask,
    "ground_truth"
] = 1

In [32]:
display(
    ground_truth.loc[
        ground_truth[
            "ground_truth"
        ].eq(1)
    ][
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
            "ground_truth",
            "evidence_level",
            "independent_events",
            "source_count",
        ]
    ]
)

,region,year,epi_week,week_start,week_end,ground_truth,evidence_level,independent_events,source_count
17,MG,2025,18,2025-04-27,2025-05-03,1,STRONG,1.0,2.0
18,MG,2025,19,2025-05-04,2025-05-10,1,STRONG,2.0,2.0
20,MG,2025,21,2025-05-18,2025-05-24,1,MODERATE,1.0,1.0


In [33]:
epidemiological_episodes = pd.DataFrame(
    [
        {
            "episode_id": "MG2025_RESP_001",
            "region": "MG",
            "year": 2025,
            "start_epi_week": 18,
            "end_epi_week": 25,
            "ground_truth": 1,
            "evidence_level": "STRONG",
            "description": (
                "Episódio estadual de aumento de SRAG com "
                "decreto de emergência, ativação do COE, "
                "expansão municipal e ampliação assistencial."
            ),
        },
    ]
)

In [34]:
for episode in epidemiological_episodes.itertuples(
    index=False
):
    mask = (
        ground_truth["region"].eq(
            episode.region
        )
        & ground_truth["year"].eq(
            episode.year
        )
        & ground_truth["epi_week"].between(
            episode.start_epi_week,
            episode.end_epi_week,
        )
    )

    ground_truth.loc[
        mask,
        "ground_truth",
    ] = episode.ground_truth

    ground_truth.loc[
        mask,
        "evidence_level",
    ] = episode.evidence_level

In [35]:
display(
    ground_truth.loc[
        ground_truth["ground_truth"].eq(1),
        [
            "region",
            "year",
            "epi_week",
            "week_start",
            "week_end",
            "ground_truth",
            "evidence_level",
        ],
    ]
)

,region,year,epi_week,week_start,week_end,ground_truth,evidence_level
17,MG,2025,18,2025-04-27,2025-05-03,1,STRONG
18,MG,2025,19,2025-05-04,2025-05-10,1,STRONG
19,MG,2025,20,2025-05-11,2025-05-17,1,STRONG
20,MG,2025,21,2025-05-18,2025-05-24,1,STRONG
21,MG,2025,22,2025-05-25,2025-05-31,1,STRONG
22,MG,2025,23,2025-06-01,2025-06-07,1,STRONG
23,MG,2025,24,2025-06-08,2025-06-14,1,STRONG
24,MG,2025,25,2025-06-15,2025-06-21,1,STRONG


## 11. Ground truth independente — SP 2026

A revisão de fontes externas mostrou que São Paulo não permaneceu em situação epidemiológica estável durante todo o período analisado.

Até a SE 16, o Ministério da Saúde/InfoGripe indicava São Paulo entre as UFs que não apresentavam incidência de SRAG em nível de alerta, risco ou alto risco nas duas semanas anteriores.

Entretanto, até a SE 20, São Paulo já apresentava:

- incidência de SRAG em nível de alerta, risco ou alto risco;
- sinal de crescimento na tendência de longo prazo;
- aumento das hospitalizações por Influenza A;
- aumento de SRAG associado ao VSR.

Por esse motivo, SP/2026 não será tratado como um caso totalmente negativo.

A rotulagem inicial será:

- SE 15–16: evidência forte de ausência de evento;
- SE 17–19: período de transição, mantido como `NA`;
- SE 20 em diante: requer reconstrução da duração do episódio com boletins subsequentes.

In [36]:
sp_negative_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(15, 16)
)

ground_truth.loc[
    sp_negative_mask,
    "ground_truth",
] = 0

ground_truth.loc[
    sp_negative_mask,
    "evidence_level",
] = "STRONG"

In [37]:
sp_transition_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(17, 19)
)

ground_truth.loc[
    sp_transition_mask,
    "ground_truth",
] = pd.NA

ground_truth.loc[
    sp_transition_mask,
    "evidence_level",
] = "UNLABELED"

In [38]:
sp_positive_episode = pd.DataFrame(
    [
        {
            "episode_id": "SP2026_RESP_001",
            "region": "SP",
            "year": 2026,
            "start_epi_week": 20,
            "end_epi_week": pd.NA,
            "ground_truth": 1,
            "evidence_level": "STRONG",
            "description": (
                "Crescimento de SRAG documentado pelo InfoGripe, "
                "com aumento de hospitalizações por Influenza A e VSR."
            ),
        }
    ]
)

### Janela epidemiológica externa — SP 2026

A revisão dos informes do Ministério da Saúde/InfoGripe indica:

- SE 20: São Paulo apresentava incidência de SRAG em nível de alerta, risco ou alto risco e sinal de crescimento na tendência de longo prazo;
- SE 28: a incidência permanecia elevada, porém sem crescimento na tendência de longo prazo;
- SE 32: São Paulo já não figurava entre as UFs com incidência geral de SRAG em alerta, risco ou alto risco;
- SE 36: permanecia fora desse grupo, embora houvesse leve aumento de SRAG por Covid-19 em nível baixo de incidência.

Para reduzir viés de classificação, as semanas de transição não são rotuladas automaticamente.

Rotulagem adotada:

- SE 15–16 → negativo;
- SE 17–19 → não rotulado;
- SE 20–28 → positivo;
- SE 29–31 → não rotulado;
- SE 32–37 → negativo;
- SE 38 → não rotulado por imaturidade operacional.

In [40]:
mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(15, 16)
)

ground_truth.loc[mask, "ground_truth"] = 0
ground_truth.loc[mask, "evidence_level"] = "STRONG"


mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(20, 28)
)

ground_truth.loc[mask, "ground_truth"] = 1
ground_truth.loc[mask, "evidence_level"] = "STRONG"

mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].between(32, 37)
)

ground_truth.loc[mask, "ground_truth"] = 0
ground_truth.loc[mask, "evidence_level"] = "STRONG"

transition_mask = (
    ground_truth["region"].eq("SP")
    & ground_truth["year"].eq(2026)
    & ground_truth["epi_week"].isin(
        [17, 18, 19, 29, 30, 31, 38]
    )
)

ground_truth.loc[
    transition_mask,
    "ground_truth",
] = pd.NA

ground_truth.loc[
    transition_mask,
    "evidence_level",
] = "UNLABELED"

In [41]:
display(
    ground_truth.loc[
        ground_truth["region"].eq("SP")
        & ground_truth["year"].eq(2026)
        & ground_truth["epi_week"].between(14, 38),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "evidence_level",
        ],
    ]
)

,region,year,epi_week,ground_truth,evidence_level
65,SP,2026,14,<NA>,NaN
66,SP,2026,15,0,STRONG
67,SP,2026,16,0,STRONG
68,SP,2026,17,<NA>,UNLABELED
69,SP,2026,18,<NA>,UNLABELED
70,SP,2026,19,<NA>,UNLABELED
71,SP,2026,20,1,STRONG
72,SP,2026,21,1,STRONG
73,SP,2026,22,1,STRONG
74,SP,2026,23,1,STRONG


## 12. Matriz de confusão — primeira avaliação formal

O ground truth externo será comparado com a saída do Sentinela.

Semanas não rotuladas (`NA`) serão excluídas da avaliação.

A classificação utilizará:

- `ground_truth = 1` → evento epidemiológico real;
- `ground_truth = 0` → ausência de evento;
- `sentinela_positive = True` → semana sinalizada pelo Sentinela;
- `sentinela_positive = False` → semana sem sinal.

A partir dessa comparação serão calculados:

- verdadeiro positivo (`TP`);
- falso positivo (`FP`);
- falso negativo (`FN`);
- verdadeiro negativo (`TN`);
- sensibilidade;
- especificidade;
- valor preditivo positivo;
- valor preditivo negativo;
- F1-score.

In [45]:
sentinela_sp_2026 = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "signal_status",
        ]
    ]
    .copy()
)

sentinela_sp_2026["sentinela_positive"] = (
    sentinela_sp_2026[
        "signal_status"
    ].eq("SIGNAL")
)

sentinela_sp_2026["region"] = "SP"
sentinela_sp_2026["year"] = 2026

display(sentinela_sp_2026.head())

,epi_week,signal_status,sentinela_positive,region,year
0,1,NO_SIGNAL,False,SP,2026
1,2,NO_SIGNAL,False,SP,2026
2,3,NO_SIGNAL,False,SP,2026
3,4,NO_SIGNAL,False,SP,2026
4,5,NO_SIGNAL,False,SP,2026


In [48]:
sentinela_mg_2025 = (
    signal_evaluation_2025[
        [
            "epi_week",
            "signal_status",
        ]
    ]
    .copy()
)

sentinela_mg_2025["sentinela_positive"] = (
    sentinela_mg_2025[
        "signal_status"
    ].eq("SIGNAL")
)

sentinela_mg_2025["region"] = "MG"
sentinela_mg_2025["year"] = 2025

In [49]:
sentinela_predictions = pd.concat(
    [
        sentinela_mg_2025,
        sentinela_sp_2026,
    ],
    ignore_index=True,
)

In [50]:
evaluation = (
    ground_truth
    .merge(
        sentinela_predictions[
            [
                "region",
                "year",
                "epi_week",
                "sentinela_positive",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="left",
    )
)

In [51]:
evaluation_labeled = (
    evaluation.loc[
        evaluation[
            "ground_truth"
        ].notna()
    ]
    .copy()
)

evaluation_labeled[
    "ground_truth"
] = (
    evaluation_labeled[
        "ground_truth"
    ].astype(int)
)

evaluation_labeled[
    "sentinela_positive"
] = (
    evaluation_labeled[
        "sentinela_positive"
    ]
    .fillna(False)
    .astype(bool)
)

In [52]:
def classify_outcome(row):
    truth = row["ground_truth"]
    pred = row["sentinela_positive"]

    if truth == 1 and pred:
        return "TP"

    if truth == 0 and pred:
        return "FP"

    if truth == 1 and not pred:
        return "FN"

    if truth == 0 and not pred:
        return "TN"

    raise ValueError("Combinação inválida.")

In [53]:
evaluation_labeled[
    "outcome"
] = (
    evaluation_labeled.apply(
        classify_outcome,
        axis=1
    )
)

In [54]:
display(
    evaluation_labeled[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "sentinela_positive",
            "outcome",
        ]
    ]
)

,region,year,epi_week,ground_truth,sentinela_positive,outcome
17,MG,2025,18,1,True,TP
18,MG,2025,19,1,True,TP
19,MG,2025,20,1,True,TP
20,MG,2025,21,1,True,TP
21,MG,2025,22,1,True,TP
22,MG,2025,23,1,False,FN
23,MG,2025,24,1,False,FN
24,MG,2025,25,1,False,FN
66,SP,2026,15,0,False,TN
67,SP,2026,16,0,False,TN


In [55]:
confusion_counts = (
    evaluation_labeled[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(confusion_counts)

outcome
TP     5
FP     0
FN    12
TN     8
Name: count, dtype: int64

In [56]:
TP = int(confusion_counts["TP"])
FP = int(confusion_counts["FP"])
FN = int(confusion_counts["FN"])
TN = int(confusion_counts["TN"])

sensitivity = (
    TP / (TP + FN)
    if (TP + FN) > 0
    else float("nan")
)

specificity = (
    TN / (TN + FP)
    if (TN + FP) > 0
    else float("nan")
)

ppv = (
    TP / (TP + FP)
    if (TP + FP) > 0
    else float("nan")
)

npv = (
    TN / (TN + FN)
    if (TN + FN) > 0
    else float("nan")
)

f1 = (
    2 * TP / (2 * TP + FP + FN)
    if (2 * TP + FP + FN) > 0
    else float("nan")
)

In [57]:
metrics = pd.DataFrame(
    [
        {
            "metric": "Sensitivity",
            "value": sensitivity,
        },
        {
            "metric": "Specificity",
            "value": specificity,
        },
        {
            "metric": "PPV",
            "value": ppv,
        },
        {
            "metric": "NPV",
            "value": npv,
        },
        {
            "metric": "F1",
            "value": f1,
        },
    ]
)

display(metrics)

,metric,value
0,Sensitivity,0.294118
1,Specificity,1.000000
2,PPV,1.000000
3,NPV,0.400000
4,F1,0.454545


## 13. Otimização de parâmetros

A primeira avaliação formal mostrou que o algoritmo atual apresenta alta especificidade, porém baixa sensibilidade.

Resultados iniciais:

- Sensibilidade: 29,4%
- Especificidade: 100%
- PPV: 100%
- NPV: 40%
- F1-score: 45,5%

Esse padrão indica um classificador conservador, com poucos falsos positivos, porém elevado número de falsos negativos.

A próxima etapa consiste em avaliar sistematicamente diferentes combinações de limiares, sem utilizar o ground truth para definir manualmente uma regra específica para uma região.

Serão avaliados:

- quantis históricos;
- fatores relativos à mediana;
- operadores de combinação entre os critérios.

O objetivo é encontrar configurações que aumentem a sensibilidade sem provocar crescimento excessivo de falsos positivos.

In [58]:
quantiles = [
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

median_factors = [
    1.10,
    1.20,
    1.25,
    1.30,
    1.40,
    1.50,
]

In [60]:
historical_mg_path = (
    project_root
    / "data"
    / "processed"
    / "srag_mg_historical_weekly_2019_2026.csv"
)

historical_sp_path = (
    project_root
    / "data"
    / "processed"
    / "srag_sp_historical_weekly_2019_2026.csv"
)

historical_mg = pd.read_csv(
    historical_mg_path,
    sep=";",
)

historical_sp = pd.read_csv(
    historical_sp_path,
    sep=";",
)

In [61]:
def build_reference_for_parameters(
    historical_weekly: pd.DataFrame,
    target_year: int,
    quantile: float,
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly["epi_year"] < target_year
    ].copy()

    history = history.loc[
        history["epi_week"].between(1, 52)
    ].copy()


    return (
        history
        .groupby("epi_week")
        .agg(
            historical_median=(
                "record_count",
                "median",
            ),
            historical_quantile=(
                "record_count",
                lambda values: values.quantile(
                    quantile
                ),
            ),
        )
        .reset_index()
    )

In [62]:
def evaluate_parameters(
    weekly_observed: pd.DataFrame,
    historical_weekly: pd.DataFrame,
    ground_truth_subset: pd.DataFrame,
    target_year: int,
    quantile: float,
    median_factor: float, 
) -> dict:
    reference =build_reference_for_parameters(
        historical_weekly=historical_weekly,
        target_year=target_year,
        quantile=quantile
    )

    evaluated = (
        weeklly_observed[
            [
                "epi_week",
                "record_count", 
            ]
        ]
        .merge(
            reference,
            on="epi_week",
            how="left",
        )
    )

    evaluated["prediction"] =(
        (
            evaluated["record_count"]
            > evaluated[
                "historical_quantile"
            ]
        )
        & (
            evaluated["record_count"]
            >= (
                evaluated[
                    "historical_median"
                ]
                * median_factor
            )
        )
    )

    labeled = (
        ground_truth_subset[
            ground_truth_subset[
                "ground_truth"
            ].notna()
        ]
        .merge(
            evaluated[
                [
                    "epi_week",
                    "prediction",
                ]
            ],
            on="epi_week",
            how="inner",
        )
    )

    truth = labeled[
        "ground_truth"
    ].astype(int)

    pred = labeled[
        "prediction"
    ].astype(bool)

    TP = int(
        ((truth == 1) & pred).sum()
    )

    FP = int(
        ((truth == 0) & pred).sum()
    )

    FN = int(
        ((truth == 1) & ~pred).sum()
    )

    TN = int(
        ((truth == 0) & ~pred).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if TP + FN
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if TN + FP
        else float("nan")
    )

    f1 = (
        2 * TP
        / (2 * TP + FP + FN)
        if (2 * TP + FP + FN)
        else float("nan")
    )

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "f1": f1,
    }



In [63]:
evaluation_cases = [
    {
        "region": "MG",
        "year": 2025,
        "weekly_observed": signal_evaluation_2025,
        "historical_weekly": historical_mg,
    },
    {
        "region": "SP",
        "year": 2026,
        "weekly_observed": signal_evaluation_sp_2026,
        "historical_weekly": historical_sp,
    },
]

In [70]:
def evaluate_combined_parameters(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    quantile: float,
    median_factor: float,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly_observed = case[
            "weekly_observed"
        ].copy()

        historical_weekly = case[
            "historical_weekly"
        ].copy()

        reference = (
            build_reference_for_parameters(
                historical_weekly=historical_weekly,
                target_year=year,
                quantile=quantile,
            )
        )

        evaluated = (
            weekly_observed[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        evaluated["sentinela_positive"] = (
            (
                evaluated["record_count"]
                > evaluated[
                    "historical_quantile"
                ]
            )
            & (
                evaluated["record_count"]
                >= (
                    evaluated[
                        "historical_median"
                    ]
                    * median_factor
                )
            )
        )

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive", 
                ]
            ]
        )
    
    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )
    
    labeled = (
        ground_truth.loc[
            ground_truth[
                "ground_truth"
            ].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = (
        labeled["ground_truth"]
        .astype(int)
    )

    prediction = (
        labeled["sentinela_positive"]
        .astype(bool)
    )

    TP = int(
        (
            (truth == 1)
            & prediction
        ).sum()
    )

    FP = int(
        (
            (truth == 0)
            & prediction
        ).sum()
    )

    FN = int(
        (
            (truth == 1)
            & ~prediction
        ).sum()
    )

    TN = int(
        (
            (truth == 0)
            & ~prediction
        ).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP
        / (
            2 * TP
            + FP
            + FN
        )
        if (
            2 * TP
            + FP
            + FN
        ) > 0
        else float("nan")
    )

    balanced_accuracy = (
        (
            sensitivity
            + specificity
        )
        / 2
    )

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy":
            balanced_accuracy,
    }

In [71]:
optimization_results = []

for quantile in quantiles:
    for median_factor in median_factors:
        result = (
            evaluate_combined_parameters(
                cases=evaluation_cases,
                ground_truth=ground_truth,
                quantile=quantile,
                median_factor=median_factor,
            )
        )

        optimization_results.append(
            result
        )

optimization_df = pd.DataFrame(
    optimization_results
)

In [72]:
optimization_ranked = (
    optimization_df
    .sort_values(
        [
            "f1",
            "balanced_accuracy",
            "sensitivity",
            "specificity",
        ],
        ascending=[
            False,
            False,
            False,
            False,
        ],
    )
    .reset_index(drop=True)
)

display(
    optimization_ranked.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [73]:
high_specificity = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ] >= 0.90
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    high_specificity.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [74]:
perfect_specificity = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ].eq(1.0)
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    perfect_specificity.head(15)
)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
1,0.65,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
2,0.65,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
3,0.65,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
4,0.65,1.40,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
5,0.65,1.50,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
6,0.70,1.10,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
7,0.70,1.20,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
8,0.70,1.25,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471
9,0.70,1.30,6,0,11,8,0.352941,1.0,1.0,0.421053,0.521739,0.676471


In [75]:
current_configuration = (
    optimization_df.loc[
        optimization_df[
            "quantile"
        ].eq(0.75)
        & optimization_df[
            "median_factor"
        ].eq(1.50)
    ]
)

display(current_configuration)

,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
17,0.75,1.5,5,0,12,8,0.294118,1.0,1.0,0.4,0.454545,0.647059


In [76]:
best_configuration = (
    optimization_ranked.iloc[0]
)

print("Configuração atual:")
display(current_configuration)

print("\nMelhor configuração encontrada:")
display(
    best_configuration.to_frame().T
)

Configuração atual:


,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
17,0.75,1.5,5,0,12,8,0.294118,1.0,1.0,0.4,0.454545,0.647059



Melhor configuração encontrada:


,quantile,median_factor,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.65,1.1,6.0,0.0,11.0,8.0,0.352941,1.0,1.0,0.421053,0.521739,0.676471


## 14. Avaliação da regra de combinação dos limiares

A otimização dos valores de quantil e fator da mediana produziu apenas
ganho modesto de sensibilidade.

Isso sugere que a baixa sensibilidade pode não depender apenas dos valores
dos limiares, mas também da regra lógica utilizada para combiná-los.

A regra atual exige simultaneamente:

`valor > quantil histórico AND valor >= fator × mediana`

Nesta etapa serão comparadas:

- `AND`: ambos os critérios precisam ser atendidos;
- `OR`: pelo menos um dos critérios precisa ser atendido.

A análise será realizada sobre o mesmo ground truth externo, mantendo
separação entre otimização e posterior validação.

In [77]:
def evaluate_combined_parameters(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    quantile: float,
    median_factor: float,
    combination: str,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly_observed = case[
            "weekly_observed"
        ].copy()

        historical_weekly = case[
            "historical_weekly"
        ].copy()

        reference = build_reference_for_parameters(
            historical_weekly=historical_weekly,
            target_year=year,
            quantile=quantile,
        )

        evaluated = (
            weekly_observed[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        above_quantile = (
            evaluated["record_count"]
            > evaluated["historical_quantile"]
        )

        above_median_factor = (
            evaluated["record_count"]
            >= (
                evaluated["historical_median"]
                * median_factor
            )
        )

        if combination == "AND":
            prediction = (
                above_quantile
                & above_median_factor
            )

        elif combination == "OR":
            prediction = (
                above_quantile
                | above_median_factor
            )

        else:
            raise ValueError(
                f"Combinação inválida: {combination}"
            )

        evaluated[
            "sentinela_positive"
        ] = prediction

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive",
                ]
            ]
        )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    labeled = (
        ground_truth.loc[
            ground_truth[
                "ground_truth"
            ].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = labeled[
        "ground_truth"
    ].astype(int)

    pred = labeled[
        "sentinela_positive"
    ].astype(bool)

    TP = int(
        ((truth == 1) & pred).sum()
    )
    FP = int(
        ((truth == 0) & pred).sum()
    )
    FN = int(
        ((truth == 1) & ~pred).sum()
    )
    TN = int(
        ((truth == 0) & ~pred).sum()
    )

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP / (2 * TP + FP + FN)
        if (2 * TP + FP + FN) > 0
        else float("nan")
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    return {
        "quantile": quantile,
        "median_factor": median_factor,
        "combination": combination,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy":
            balanced_accuracy,
    }

In [78]:
quantiles = [
    0.60,
    0.65,
    0.70,
    0.75,
    0.80,
    0.85,
    0.90,
]

median_factors = [
    1.00,
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
    1.40,
    1.50,
]

combinations = [
    "AND",
    "OR",
]

In [79]:
optimization_results = []

for combination in combinations:
    for quantile in quantiles:
        for median_factor in median_factors:
            optimization_results.append(
                evaluate_combined_parameters(
                    cases=evaluation_cases,
                    ground_truth=ground_truth,
                    quantile=quantile,
                    median_factor=median_factor,
                    combination=combination,
                )
            )

optimization_df = pd.DataFrame(
    optimization_results
)

In [80]:
candidate_models = (
    optimization_df.loc[
        optimization_df[
            "specificity"
        ] >= 0.90
    ]
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "sensitivity",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    candidate_models.head(20)
)

,quantile,median_factor,combination,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,0.60,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
1,0.60,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
2,0.65,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
3,0.65,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
4,0.70,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
5,0.70,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
6,0.75,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
7,0.75,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
8,0.80,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
9,0.80,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706


In [81]:
display(
    optimization_df.loc[
        optimization_df[
            "specificity"
        ].eq(1.0)
    ]
    .sort_values(
        [
            "sensitivity",
            "f1",
        ],
        ascending=False,
    )
    .head(20)
)

,quantile,median_factor,combination,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
63,0.60,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
64,0.60,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
72,0.65,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
73,0.65,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
81,0.70,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
82,0.70,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
90,0.75,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
91,0.75,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
99,0.80,1.00,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706
100,0.80,1.05,OR,9,0,8,8,0.529412,1.0,1.0,0.500000,0.692308,0.764706


In [82]:
pd.crosstab(
    evaluation_labeled["region"],
    evaluation_labeled["outcome"],
)

outcome,FN,TN,TP
region,,,
MG,3,0,5
SP,9,8,0


In [83]:
BEST_QUANTILE = 0.60
BEST_MEDIAN_FACTOR = 1.00
BEST_COMBINATION = "OR"

In [84]:
def predict_configuration(
    cases: list[dict],
    quantile: float,
    median_factor: float,
    combination: str,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        weekly = case[
            "weekly_observed"
        ].copy()

        historical = case[
            "historical_weekly"
        ].copy()

        reference = (
            build_reference_for_parameters(
                historical_weekly=historical,
                target_year=year,
                quantile=quantile,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference,
                on="epi_week",
                how="left",
            )
        )

        evaluated[
            "ratio_to_median"
        ] = (
            evaluated["record_count"]
            / evaluated[
                "historical_median"
            ]
        )

        evaluated[
            "above_quantile"
        ] = (
            evaluated["record_count"]
            > evaluated[
                "historical_quantile"
            ]
        )

        evaluated[
            "above_median_factor"
        ] = (
            evaluated["record_count"]
            >= (
                evaluated[
                    "historical_median"
                ]
                * median_factor
            )
        )

        if combination == "AND":
            prediction = (
                evaluated["above_quantile"]
                & evaluated[
                    "above_median_factor"
                ]
            )

        elif combination == "OR":
            prediction = (
                evaluated["above_quantile"]
                | evaluated[
                    "above_median_factor"
                ]
            )

        else:
            raise ValueError(
                f"Combinação inválida: {combination}"
            )

        evaluated[
            "sentinela_positive"
        ] = prediction

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [85]:
candidate_predictions = predict_configuration(
    cases=evaluation_cases,
    quantile=0.60,
    median_factor=1.00,
    combination="OR",
)

In [86]:
candidate_evaluation = (
    ground_truth.loc[
        ground_truth[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions,
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

candidate_evaluation[
    "ground_truth"
] = (
    candidate_evaluation[
        "ground_truth"
    ].astype(int)
)

In [87]:
candidate_evaluation[
    "outcome"
] = candidate_evaluation.apply(
    classify_outcome,
    axis=1,
)

In [88]:
candidate_fn = (
    candidate_evaluation.loc[
        candidate_evaluation[
            "outcome"
        ].eq("FN")
    ][
        [
            "region",
            "year",
            "epi_week",
            "record_count",
            "historical_median",
            "historical_quantile",
            "ratio_to_median",
            "above_quantile",
            "above_median_factor",
        ]
    ]
    .sort_values(
        [
            "region",
            "epi_week",
        ]
    )
)

display(candidate_fn)

,region,year,epi_week,record_count,historical_median,historical_quantile,ratio_to_median,above_quantile,above_median_factor
10,SP,2026,20,2345.0,3053.0,3060.2,0.768097,False,False
11,SP,2026,21,2271.0,3141.0,3604.2,0.723018,False,False
12,SP,2026,22,2477.0,2777.0,3674.0,0.891970,False,False
13,SP,2026,23,2232.0,2746.0,3624.4,0.812819,False,False
14,SP,2026,24,2217.0,2323.0,3412.0,0.954369,False,False
15,SP,2026,25,2071.0,2399.0,3568.4,0.863276,False,False
17,SP,2026,27,1704.0,1768.0,2772.4,0.963801,False,False
18,SP,2026,28,1224.0,1555.0,2410.0,0.787138,False,False


In [89]:
display(
    pd.crosstab(
        candidate_evaluation[
            "region"
        ],
        candidate_evaluation[
            "outcome"
        ],
    )
)

outcome,FN,TN,TP
region,,,
MG,0,0,8
SP,8,8,1


In [90]:
def build_reference_selected_years(
    historical_weekly: pd.DataFrame,
    years: list[int],
    quantile: float,
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly[
            "epi_year"
        ].isin(years)
        & historical_weekly[
            "epi_week"
        ].between(1, 52)
    ].copy()

    reference = (
        history
        .groupby("epi_week")
        .agg(
            historical_median=(
                "record_count",
                "median",
            ),
            historical_quantile=(
                "record_count",
                lambda values: (
                    values.quantile(
                        quantile
                    )
                ),
            ),
            historical_years=(
                "epi_year",
                "nunique",
            ),
        )
        .reset_index()
    )

    return reference

In [91]:
reference_sp_without_pandemic = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
            2025,
        ],
        quantile=0.75,
    )
)

In [92]:
comparison_sp_reference = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "record_count",
        ]
    ]
    .merge(
        build_reference_for_parameters(
            historical_weekly=historical_sp,
            target_year=2026,
            quantile=0.75,
        ).rename(
            columns={
                "historical_median":
                    "median_all",
                "historical_quantile":
                    "q75_all",
            }
        ),
        on="epi_week",
    )
    .merge(
        reference_sp_without_pandemic.rename(
            columns={
                "historical_median":
                    "median_without_pandemic",
                "historical_quantile":
                    "q75_without_pandemic",
            }
        ),
        on="epi_week",
    )
)

display(
    comparison_sp_reference.loc[
        comparison_sp_reference[
            "epi_week"
        ].between(20, 28)
    ]
)

,epi_week,record_count,median_all,q75_all,median_without_pandemic,q75_without_pandemic,historical_years
19,20,2345.0,3053.0,7284.5,1869.0,3053.0,5
20,21,2271.0,3141.0,7292.0,1872.0,3141.0,5
21,22,2477.0,2777.0,7213.0,1692.0,2777.0,5
22,23,2232.0,2746.0,7843.0,1574.0,2746.0,5
23,24,2217.0,2323.0,7378.0,1629.0,2323.0,5
24,25,2071.0,2399.0,7482.5,1637.0,2399.0,5
25,26,2052.0,1891.0,6932.5,1527.0,1891.0,5
26,27,1704.0,1768.0,6239.5,1457.0,1768.0,5
27,28,1224.0,1555.0,5192.0,1206.0,1555.0,5


## 15. Detecção por desvio robusto

A análise de SP/2026 mostrou que o episódio epidemiológico apresentou
elevação sustentada em relação à mediana histórica, porém sem ultrapassar
frequentemente os quantis superiores.

Isso sugere que uma regra baseada exclusivamente em Q75 pode ter baixa
sensibilidade para episódios de crescimento moderado e persistente.

Nesta etapa será avaliado um escore robusto baseado na mediana e no
Median Absolute Deviation (MAD).

O método reduz a influência de anos extremos e mede o afastamento da
contagem semanal em relação ao comportamento histórico típico.

In [93]:
import numpy as np

def build_robust_reference(
    historical_weekly: pd.DataFrame,
    years: list[int],
) -> pd.DataFrame:
    history = historical_weekly.loc[
        historical_weekly["epi_year"].isin(years)
        & historical_weekly["epi_week"].between(1, 52)
    ].copy()

    rows = []

    for epi_week, group in history.groupby("epi_week"):
        values = group["record_count"].astype(float)

        median = values.median()

        mad = np.median(
            np.abs(values - median)
        )

        rows.append(
            {
                "epi_week": int(epi_week),
                "historical_median": median,
                "mad": mad,
                "historical_years": (
                    group["epi_year"].nunique()
                ),
            }
        )

    return pd.DataFrame(rows)

In [94]:
robust_reference_sp = build_robust_reference(
    historical_weekly=historical_sp,
    years=[
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
)

display(
    robust_reference_sp.head()
)

,epi_week,historical_median,mad,historical_years
0,1,927.0,778.0,5
1,2,817.0,579.0,5
2,3,812.0,520.0,5
3,4,791.0,416.0,5
4,5,913.0,568.0,5


In [95]:
robust_sp_2026 = (
    signal_evaluation_sp_2026[
        [
            "epi_week",
            "record_count",
        ]
    ]
    .merge(
        robust_reference_sp,
        on="epi_week",
        how="left",
    )
)

In [96]:
robust_sp_2026[
    "robust_z"
] = np.where(
    robust_sp_2026["mad"] > 0,
    (
        0.6745
        * (
            robust_sp_2026["record_count"]
            - robust_sp_2026[
                "historical_median"
            ]
        )
        / robust_sp_2026["mad"]
    ),
    np.nan,
)

In [97]:
display(
    robust_sp_2026.loc[
        robust_sp_2026[
            "epi_week"
        ].between(15, 37),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "mad",
            "robust_z",
        ],
    ]
)

,epi_week,record_count,historical_median,mad,robust_z
14,15,1599.0,1870.0,115.0,-1.589474
15,16,1759.0,1909.0,226.0,-0.447677
16,17,1736.0,1814.0,243.0,-0.216506
17,18,1780.0,1813.0,550.0,-0.040470
18,19,2186.0,1688.0,968.0,0.347005
19,20,2345.0,1869.0,1184.0,0.271167
20,21,2271.0,1872.0,1269.0,0.212077
21,22,2477.0,1692.0,1085.0,0.488002
22,23,2232.0,1574.0,1015.0,0.437262
23,24,2217.0,1629.0,694.0,0.571478


In [98]:
robust_thresholds = [
    0.5,
    1.0,
    1.5,
    2.0,
    2.5,
    3.0,
]

In [99]:
for threshold in robust_thresholds:
    robust_sp_2026[
        f"signal_z_{threshold}"
    ] = (
        robust_sp_2026[
            "robust_z"
        ]
        >= threshold
    )

In [100]:
baseline_sets = {
    "all_history": [
        2019,
        2020,
        2021,
        2022,
        2023,
        2024,
        2025,
    ],
    "without_pandemic": [
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
    "recent_post_pandemic": [
        2022,
        2023,
        2024,
        2025,
    ],
}

## 16. Detecção por excesso relativo persistente

A avaliação por MAD mostrou que o episódio de SP/2026 não apresentou
desvios semanais extremos em relação à dispersão histórica.

Entretanto, entre as SE 20 e 28, observou-se uma sequência prolongada
de semanas acima da mediana histórica.

Isso sugere que episódios epidemiológicos podem se manifestar de duas formas:

1. aumento abrupto de grande magnitude;
2. aumento moderado, porém persistente.

Nesta etapa será avaliado um indicador de excesso relativo combinado
com persistência temporal.

In [101]:
robust_sp_2026["ratio_to_median"] = (
    robust_sp_2026["record_count"]
    / robust_sp_2026["historical_median"]
)

robust_sp_2026["relative_excess"] = (
    robust_sp_2026["ratio_to_median"] - 1
)

In [102]:
display(
    robust_sp_2026.loc[
        robust_sp_2026["epi_week"].between(15, 37),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "relative_excess",
        ],
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,relative_excess
14,15,1599.0,1870.0,0.855080,-0.144920
15,16,1759.0,1909.0,0.921425,-0.078575
16,17,1736.0,1814.0,0.957001,-0.042999
17,18,1780.0,1813.0,0.981798,-0.018202
18,19,2186.0,1688.0,1.295024,0.295024
19,20,2345.0,1869.0,1.254682,0.254682
20,21,2271.0,1872.0,1.213141,0.213141
21,22,2477.0,1692.0,1.463948,0.463948
22,23,2232.0,1574.0,1.418043,0.418043
23,24,2217.0,1629.0,1.360958,0.360958


In [103]:
ratio_thresholds = [
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
]

persistence_windows = [
    1,
    2,
    3,
    4,
]

In [104]:
def apply_persistent_signal(
    df: pd.DataFrame,
    ratio_threshold: float,
    persistence: int,
) -> pd.DataFrame:
    result = (
        df.sort_values("epi_week")
        .copy()
    )

    result["above_ratio"] = (
        result["ratio_to_median"]
        >= ratio_threshold
    )

    result["persistent_signal"] = (
        result["above_ratio"]
        .rolling(
            window=persistence,
            min_periods=persistence,
        )
        .sum()
        .eq(persistence)
    )

    return result

In [105]:
test_sp = apply_persistent_signal(
    robust_sp_2026,
    ratio_threshold=1.15,
    persistence=2,
)

display(
    test_sp.loc[
        test_sp["epi_week"].between(15, 37),
        [
            "epi_week",
            "ratio_to_median",
            "above_ratio",
            "persistent_signal",
        ],
    ]
)

,epi_week,ratio_to_median,above_ratio,persistent_signal
14,15,0.855080,False,False
15,16,0.921425,False,False
16,17,0.957001,False,False
17,18,0.981798,False,False
18,19,1.295024,True,False
19,20,1.254682,True,True
20,21,1.213141,True,True
21,22,1.463948,True,True
22,23,1.418043,True,True
23,24,1.360958,True,True


In [107]:
def evaluate_persistent_ratio(
    cases: list[dict],
    ground_truth: pd.DataFrame,
    baseline_years_by_case: dict,
    ratio_threshold: float,
    persistence: int,
) -> dict:
    prediction_frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]

        key = (region, year)

        historical = case["historical_weekly"]

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=historical,
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["above_ratio"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["sentinela_positive"] = (
            evaluated["above_ratio"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        prediction_frames.append(
            evaluated[
                [
                    "region",
                    "year",
                    "epi_week",
                    "sentinela_positive",
                ]
            ]
        )

    predictions = pd.concat(
        prediction_frames,
        ignore_index=True,
    )

    labeled = (
        ground_truth.loc[
            ground_truth["ground_truth"].notna()
        ][
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
            ]
        ]
        .merge(
            predictions,
            on=[
                "region",
                "year",
                "epi_week",
            ],
            how="inner",
        )
    )

    truth = labeled["ground_truth"].astype(int)
    pred = labeled["sentinela_positive"].astype(bool)

    TP = int(((truth == 1) & pred).sum())
    FP = int(((truth == 0) & pred).sum())
    FN = int(((truth == 1) & ~pred).sum())
    TN = int(((truth == 0) & ~pred).sum())

    sensitivity = (
        TP / (TP + FN)
        if (TP + FN) > 0
        else float("nan")
    )

    specificity = (
        TN / (TN + FP)
        if (TN + FP) > 0
        else float("nan")
    )

    ppv = (
        TP / (TP + FP)
        if (TP + FP) > 0
        else float("nan")
    )

    npv = (
        TN / (TN + FN)
        if (TN + FN) > 0
        else float("nan")
    )

    f1 = (
        2 * TP / (2 * TP + FP + FN)
        if (2 * TP + FP + FN) > 0
        else float("nan")
    )

    balanced_accuracy = (
        sensitivity + specificity
    ) / 2

    return {
        "ratio_threshold": ratio_threshold,
        "persistence": persistence,
        "TP": TP,
        "FP": FP,
        "FN": FN,
        "TN": TN,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "ppv": ppv,
        "npv": npv,
        "f1": f1,
        "balanced_accuracy": balanced_accuracy,
    }

In [108]:
baseline_years_by_case = {
    ("MG", 2025): [
        2019,
        2022,
        2023,
        2024,
    ],
    ("SP", 2026): [
        2019,
        2022,
        2023,
        2024,
        2025,
    ],
}

In [109]:
persistent_results = []

for threshold in [
    1.00,
    1.05,
    1.10,
    1.15,
    1.20,
    1.25,
    1.30,
]:
    for persistence in [
        1,
        2,
        3,
        4,
    ]:
        persistent_results.append(
            evaluate_persistent_ratio(
                cases=evaluation_cases,
                ground_truth=ground_truth,
                baseline_years_by_case=(
                    baseline_years_by_case
                ),
                ratio_threshold=threshold,
                persistence=persistence,
            )
        )

persistent_results_df = (
    pd.DataFrame(
        persistent_results
    )
    .sort_values(
        [
            "balanced_accuracy",
            "f1",
            "sensitivity",
            "specificity",
        ],
        ascending=False,
    )
    .reset_index(drop=True)
)

display(
    persistent_results_df.head(20)
)

,ratio_threshold,persistence,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,1.00,1,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
1,1.00,2,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
2,1.00,3,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
3,1.05,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
4,1.05,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
5,1.10,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
6,1.10,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
7,1.15,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
8,1.15,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
9,1.00,4,15,0,2,8,0.882353,1.0,1.0,0.800000,0.937500,0.941176


In [110]:
display(
    persistent_results_df.loc[
        persistent_results_df[
            "specificity"
        ] >= 0.90
    ].head(20)
)

,ratio_threshold,persistence,TP,FP,FN,TN,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,1.00,1,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
1,1.00,2,17,0,0,8,1.000000,1.0,1.0,1.000000,1.000000,1.000000
2,1.00,3,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
3,1.05,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
4,1.05,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
5,1.10,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
6,1.10,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
7,1.15,1,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
8,1.15,2,16,0,1,8,0.941176,1.0,1.0,0.888889,0.969697,0.970588
9,1.00,4,15,0,2,8,0.882353,1.0,1.0,0.800000,0.937500,0.941176


In [111]:
def predict_persistent_ratio(
    cases: list[dict],
    baseline_years_by_case: dict,
    ratio_threshold: float,
    persistence: int,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]
        key = (region, year)

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=case["historical_weekly"],
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count", 
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["above_ratio"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["sentinela_positive"] = (
            evaluated["above_ratio"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)
    
    return pd.concat(
        frames,
        ignore_index=True,
    )

In [112]:
candidate_predictions = (
    predict_persistent_ratio(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [113]:
candidate_validation = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions,
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

candidate_validation["ground_truth"] = (
    candidate_validation[
        "ground_truth"
    ].astype(int)
)

candidate_validation["outcome"] = (
    candidate_validation.apply(
        classify_outcome,
        axis=1,
    )
)

In [114]:
display(
    candidate_validation.loc[
        candidate_validation[
            "outcome"
        ].isin(["FN", "FP"]),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "above_ratio",
            "sentinela_positive",
            "outcome",
        ],
    ]
)

,region,year,epi_week,ground_truth,record_count,historical_median,ratio_to_median,above_ratio,sentinela_positive,outcome
18,SP,2026,28,1,1224.0,1206.0,1.014925,False,False,FN


In [115]:
display(
    pd.crosstab(
        candidate_validation["region"],
        candidate_validation["outcome"],
    )
)

outcome,FN,TN,TP
region,,,
MG,0,0,8
SP,1,8,8


### Análise do único falso negativo

O único falso negativo da configuração candidata ocorreu em SP/2026 na SE 28.

Nessa semana:

- casos observados: 1.224;
- mediana histórica: 1.206;
- razão observado/mediana: 1,015;
- limiar candidato: 1,15.

Embora a SE 28 tenha sido incluída na janela positiva externa, a contagem já havia retornado para um nível muito próximo da mediana histórica.

Esse resultado sugere que a divergência pode representar uma diferença entre:

- persistência institucional/epidemiológica do episódio;
- normalização estatística da série semanal.

A SE 28 deve, portanto, ser tratada como uma semana de borda do episódio durante análises posteriores.

In [116]:
sp_episode_phases = {
    20: "ONSET",
    21: "ACTIVE",
    22: "ACTIVE",
    23: "ACTIVE",
    24: "ACTIVE",
    25: "ACTIVE",
    26: "ACTIVE",
    27: "ACTIVE",
    28: "RESOLUTION",
}

In [117]:
mg_episode_phases = {
    18: "ONSET",
    19: "ACTIVE",
    20: "ACTIVE",
    21: "ACTIVE",
    22: "ACTIVE",
    23: "ACTIVE",
    24: "ACTIVE",
    25: "RESOLUTION",
}

In [118]:
ground_truth["episode_phase"] = pd.NA

for week, phase in mg_episode_phases.items():
    mask = (
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
        & ground_truth["epi_week"].eq(week)
    )

    ground_truth.loc[
        mask,
        "episode_phase",
    ] = phase


for week, phase in sp_episode_phases.items():
    mask = (
        ground_truth["region"].eq("SP")
        & ground_truth["year"].eq(2026)
        & ground_truth["epi_week"].eq(week)
    )

    ground_truth.loc[
        mask,
        "episode_phase",
    ] = phase

## 17. Validação fora da amostra — SP 2025

A configuração candidata foi definida utilizando apenas:

- MG/2025;
- SP/2026.

Nesta etapa, seus parâmetros são congelados:

- baseline excluindo 2020 e 2021;
- razão observado/mediana >= 1,15;
- persistência mínima de 2 semanas.

Nenhum parâmetro será modificado durante a avaliação de SP/2025.

Fontes oficiais do Estado de São Paulo mostram crescimento importante
das hospitalizações por SRAG ao longo do outono de 2025, com pico
aproximadamente nas SE 20–21 e queda posterior.

SP/2025 será utilizado como conjunto independente para avaliar a
generalização do modelo candidato.

In [119]:
weekly_sp_2025 = (
    historical_sp.loc[
        historical_sp["epi_year"].eq(2025)
    ][
        [
            "epi_week",
            "record_count",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

display(weekly_sp_2025.head())
display(weekly_sp_2025.tail())

,epi_week,record_count
0,1,927
1,2,817
2,3,812
3,4,791
4,5,700


,epi_week,record_count
48,49,1204
49,50,999
50,51,824
51,52,687
52,53,681


In [120]:
baseline_sp_2025_years = [
    2019,
    2022,
    2023,
    2024,
]

reference_sp_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=baseline_sp_2025_years,
        quantile=0.75,
    )
)

In [121]:
validation_sp_2025 = (
    weekly_sp_2025
    .merge(
        reference_sp_2025[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left"
    )
)

validation_sp_2025["ratio_to_median"] = (
    validation_sp_2025["record_count"]
    / validation_sp_2025["historical_median"]
)

validation_sp_2025["above_ratio"] = (
    validation_sp_2025["ratio_to_median"]
    >= 1.15
)

validation_sp_2025["sentinela_positive"] = (
    validation_sp_2025["above_ratio"]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

display(
    validation_sp_2025.loc[
        validation_sp_2025[
            "epi_week"
        ].between(10, 35)
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,above_ratio,sentinela_positive
9,10,1166,1806.0,0.645626,False,False
10,11,1113,1768.5,0.629347,False,False
11,12,1263,1794.5,0.703817,False,False
12,13,1462,1746.5,0.837103,False,False
13,14,1608,1873.0,0.858516,False,False
14,15,1791,1927.5,0.929183,False,False
15,16,1992,1796.0,1.109131,False,False
16,17,2313,1711.5,1.351446,True,False
17,18,2526,1755.0,1.439316,True,True
18,19,2728,1592.5,1.713030,True,True


In [122]:
ground_truth_sp_2025 = pd.DataFrame(
    {
        "epi_week": range(1, 53),
    }
)

ground_truth_sp_2025["region"] = "SP"
ground_truth_sp_2025["year"] = 2025

ground_truth_sp_2025["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth_sp_2025.index,
    dtype="Int64",
)

ground_truth_sp_2025["evidence_level"] = pd.NA

In [123]:
positive_mask = (
    ground_truth_sp_2025[
        "epi_week"
    ].between(17, 25)
)

ground_truth_sp_2025.loc[
    positive_mask,
    "ground_truth",
] = 1

ground_truth_sp_2025.loc[
    positive_mask,
    "evidence_level",
] = "STRONG"

In [124]:
negative_mask = (
    ground_truth_sp_2025[
        "epi_week"
    ].between(30, 40)
)

ground_truth_sp_2025.loc[
    negative_mask,
    "ground_truth",
] = 0

ground_truth_sp_2025.loc[
    negative_mask,
    "evidence_level",
] = "MODERATE"

In [125]:
sp_2025_evaluation = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_sp_2025[
            [
               "epi_week",
                "record_count",
                "historical_median",
                "ratio_to_median",
                "sentinela_positive", 
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

sp_2025_evaluation[
    "ground_truth"
] = (
    sp_2025_evaluation[
        "ground_truth"
    ].astype(int)
)

sp_2025_evaluation[
    "outcome"
] = sp_2025_evaluation.apply(
    classify_outcome,
    axis=1,
)

In [126]:
display(
    sp_2025_evaluation[
        [
            "epi_week",
            "ground_truth",
            "ratio_to_median",
            "sentinela_positive",
            "outcome",
        ]
    ]
)

display(
    sp_2025_evaluation[
        "outcome"
    ].value_counts()
)

,epi_week,ground_truth,ratio_to_median,sentinela_positive,outcome
0,17,1,1.351446,False,FN
1,18,1,1.439316,True,TP
2,19,1,1.713030,True,TP
3,20,1,1.821599,True,TP
4,21,1,1.940686,True,TP
5,22,1,1.813255,True,TP
6,23,1,1.745154,True,TP
7,24,1,1.517805,True,TP
8,25,1,1.492379,True,TP
9,30,0,1.096226,False,TN


outcome
TN    11
TP     8
FN     1
Name: count, dtype: int64

In [127]:
validation_sp_2025["signal_detected"] = (
    validation_sp_2025["ratio_to_median"]
    >= 1.15
)

validation_sp_2025["alert_confirmed"] = (
    validation_sp_2025["signal_detected"]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

In [128]:
display(
    validation_sp_2025.loc[
        validation_sp_2025["epi_week"].between(15, 27),
        [
            "epi_week",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ],
    ]
)

,epi_week,ratio_to_median,signal_detected,alert_confirmed
14,15,0.929183,False,False
15,16,1.109131,False,False
16,17,1.351446,True,False
17,18,1.439316,True,True
18,19,1.713030,True,True
19,20,1.821599,True,True
20,21,1.940686,True,True
21,22,1.813255,True,True
22,23,1.745154,True,True
23,24,1.517805,True,True


## 18. Validação fora da amostra — MG 2024

A configuração candidata permanece congelada:

- baseline histórico excluindo 2020 e 2021;
- `SIGNAL` quando observado/mediana histórica >= 1,15;
- `ALERT` após duas semanas consecutivas com `SIGNAL`.

Nenhum parâmetro será ajustado durante a avaliação de MG/2024.

In [129]:
weekly_mg_2024 = (
    historical_mg.loc[
        historical_mg["epi_year"].eq(2024)
    ][
        [
            "epi_week",
            "record_count",
        ]
    ]
    .sort_values("epi_week")
    .reset_index(drop=True)
)

display(weekly_mg_2024.head())
display(weekly_mg_2024.tail())

,epi_week,record_count
0,1,285
1,2,276
2,3,330
3,4,274
4,5,363


,epi_week,record_count
47,48,482
48,49,543
49,50,456
50,51,405
51,52,409


In [130]:
baseline_mg_2024_years = [
    2019,
    2022,
    2023,
]

In [131]:
reference_mg_2024 = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=baseline_mg_2024_years,
        quantile=0.75,
    )
)

In [132]:
validation_mg_2024 = (
    weekly_mg_2024
    .merge(
        reference_mg_2024[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

validation_mg_2024["ratio_to_median"] = (
    validation_mg_2024["record_count"]
    / validation_mg_2024["historical_median"]
)

validation_mg_2024["signal_detected"] = (
    validation_mg_2024[
        "ratio_to_median"
    ] >= 1.15
)

validation_mg_2024["alert_confirmed"] = (
    validation_mg_2024[
        "signal_detected"
    ]
    .rolling(
        window=2,
        min_periods=2,
    )
    .sum()
    .eq(2)
)

In [133]:
display(
    validation_mg_2024[
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,signal_detected,alert_confirmed
0,1,285,887.0,0.321308,False,False
1,2,276,667.0,0.413793,False,False
2,3,330,524.0,0.629771,False,False
3,4,274,452.0,0.606195,False,False
4,5,363,504.0,0.720238,False,False
5,6,363,612.0,0.593137,False,False
6,7,535,659.0,0.811836,False,False
7,8,650,657.0,0.989346,False,False
8,9,718,657.0,1.092846,False,False
9,10,603,648.0,0.930556,False,False


In [134]:
display(
    validation_mg_2024.loc[
        validation_mg_2024[
            "signal_detected"
        ],
        [
            "epi_week",
            "record_count",
            "historical_median",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
        ],
    ]
)

,epi_week,record_count,historical_median,ratio_to_median,signal_detected,alert_confirmed
13,14,717,565.0,1.269027,True,False
14,15,662,535.0,1.237383,True,True
15,16,735,561.0,1.310160,True,True
16,17,699,509.0,1.373281,True,True
17,18,713,605.0,1.178512,True,True
18,19,708,556.0,1.273381,True,True
19,20,722,618.0,1.168285,True,True
20,21,732,577.0,1.268631,True,True
22,23,631,533.0,1.183865,True,False
23,24,610,478.0,1.276151,True,True


## 19. Ground truth externo — MG 2024

A revisão de fontes independentes indica múltiplos períodos de aumento
de SRAG em Minas Gerais durante 2024.

Foram identificados três episódios principais:

### Episódio 1 — outono/inverno
- início externo: aproximadamente SE 14;
- crescimento documentado durante abril e maio;
- MG permaneceu com tendência de crescimento até pelo menos SE 21;
- encerramento exato não definido.

Rotulagem conservadora:
- SE 14–21 → positivo;
- SE 22–27 → transição / não rotulado.

### Episódio 2 — recrudescimento no segundo semestre
- crescimento documentado em setembro;
- manutenção de tendência de aumento até SE 39–40.

Rotulagem conservadora:
- SE 35–40 → positivo;
- SE 29–34 e SE 41–44 → transição / não rotulado.

### Episódio 3 — fim do ano
- nova elevação documentada nas SE 49–50;
- associação com circulação de Covid-19 e outros vírus respiratórios.

Rotulagem conservadora:
- SE 49–52 → positivo;
- SE 45–48 → transição / não rotulado.

Semanas fora dessas janelas não serão classificadas automaticamente como negativas
sem evidência externa suficiente.

In [135]:
ground_truth_mg_2024 = pd.DataFrame(
    {
        "epi_week": range(1, 53),
    }
)

ground_truth_mg_2024["region"] = "MG"
ground_truth_mg_2024["year"] = 2024

ground_truth_mg_2024["ground_truth"] = pd.Series(
    pd.NA,
    index=ground_truth_mg_2024.index,
    dtype="Int64",
)

ground_truth_mg_2024["evidence_level"] = pd.NA
ground_truth_mg_2024["episode_id"] = pd.NA

In [136]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(14, 21)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_001"

In [137]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(35, 40)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_002"

In [138]:
mask = ground_truth_mg_2024[
    "epi_week"
].between(49, 52)

ground_truth_mg_2024.loc[
    mask,
    "ground_truth",
] = 1

ground_truth_mg_2024.loc[
    mask,
    "evidence_level",
] = "STRONG"

ground_truth_mg_2024.loc[
    mask,
    "episode_id",
] = "MG2024_RESP_003"

In [139]:
negative_weeks = [
    *range(1, 9),
    *range(43, 46),
]

negative_mask = (
    ground_truth_mg_2024[
        "epi_week"
    ].isin(negative_weeks)
)

ground_truth_mg_2024.loc[
    negative_mask,
    "ground_truth",
] = 0

ground_truth_mg_2024.loc[
    negative_mask,
    "evidence_level",
] = "MODERATE"

In [140]:
mg_2024_evaluation = (
    ground_truth_mg_2024.loc[
        ground_truth_mg_2024[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_mg_2024[
            [
                "epi_week",
                "ratio_to_median",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

In [141]:
mg_2024_evaluation[
    "sentinela_positive"
] = (
    mg_2024_evaluation[
        "signal_detected"
    ]
)

mg_2024_evaluation[
    "ground_truth"
] = (
    mg_2024_evaluation[
        "ground_truth"
    ].astype(int)
)

mg_2024_evaluation[
    "outcome"
] = (
    mg_2024_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [142]:
display(
    mg_2024_evaluation[
        [
            "epi_week",
            "ground_truth",
            "ratio_to_median",
            "signal_detected",
            "alert_confirmed",
            "outcome",
        ]
    ]
)

display(
    mg_2024_evaluation[
        "outcome"
    ].value_counts()
)

,epi_week,ground_truth,ratio_to_median,signal_detected,alert_confirmed,outcome
0,1,0,0.321308,False,False,TN
1,2,0,0.413793,False,False,TN
2,3,0,0.629771,False,False,TN
3,4,0,0.606195,False,False,TN
4,5,0,0.720238,False,False,TN
5,6,0,0.593137,False,False,TN
6,7,0,0.811836,False,False,TN
7,8,0,0.989346,False,False,TN
8,14,1,1.269027,True,False,TP
9,15,1,1.237383,True,True,TP


outcome
TP    18
TN    11
Name: count, dtype: int64

## 20. Avaliação agregada do modelo candidato

A configuração candidata foi congelada em:

- baseline histórico excluindo 2020 e 2021;
- `SIGNAL` quando observado/mediana histórica >= 1,15;
- `ALERT` após 2 semanas consecutivas com `SIGNAL`.

A partir deste ponto, nenhum parâmetro será alterado.

Serão agregadas as avaliações independentes de:

- MG/2024;
- MG/2025;
- SP/2025;
- SP/2026.

O objetivo é estimar o desempenho global do modelo candidato antes de ampliar a validação para novas regiões.

In [143]:
eval_mg_2024 = (
    mg_2024_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "signal_detected",
        ]
    ]
    .copy()
)

eval_mg_2024["sentinela_positive"] = (
    eval_mg_2024["signal_detected"]
)

In [144]:
eval_sp_2025 = (
    sp_2025_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "sentinela_positive",
        ]
    ]
    .copy()
)

In [145]:
candidate_predictions_all = (
    predict_persistent_ratio(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [146]:
def predict_signal_and_alert(
    cases: list[dict],
    baseline_years_by_case: dict,
    ratio_threshold: float = 1.15,
    persistence: int = 2,
) -> pd.DataFrame:
    frames = []

    for case in cases:
        region = case["region"]
        year = case["year"]
        key = (region, year)

        weekly = (
            case["weekly_observed"]
            .sort_values("epi_week")
            .copy()
        )

        reference = (
            build_reference_selected_years(
                historical_weekly=case["historical_weekly"],
                years=baseline_years_by_case[key],
                quantile=0.75,
            )
        )

        evaluated = (
            weekly[
                [
                    "epi_week",
                    "record_count",
                ]
            ]
            .merge(
                reference[
                    [
                        "epi_week",
                        "historical_median",
                    ]
                ],
                on="epi_week",
                how="left",
            )
            .sort_values("epi_week")
        )

        evaluated["ratio_to_median"] = (
            evaluated["record_count"]
            / evaluated["historical_median"]
        )

        evaluated["signal_detected"] = (
            evaluated["ratio_to_median"]
            >= ratio_threshold
        )

        evaluated["alert_confirmed"] = (
            evaluated["signal_detected"]
            .rolling(
                window=persistence,
                min_periods=persistence,
            )
            .sum()
            .eq(persistence)
        )

        evaluated["region"] = region
        evaluated["year"] = year

        frames.append(evaluated)

    return pd.concat(
        frames,
        ignore_index=True,
    )

In [147]:
candidate_predictions_all = (
    predict_signal_and_alert(
        cases=evaluation_cases,
        baseline_years_by_case=baseline_years_by_case,
        ratio_threshold=1.15,
        persistence=2,
    )
)

In [148]:
eval_base_cases = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

eval_base_cases["sentinela_positive"] = (
    eval_base_cases["signal_detected"]
)

In [149]:
aggregate_evaluation = pd.concat(
    [
        eval_base_cases[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
        eval_sp_2025[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
        eval_mg_2024[
            [
                "region",
                "year",
                "epi_week",
                "ground_truth",
                "sentinela_positive",
            ]
        ],
    ],
    ignore_index=True,
)

In [150]:
aggregate_evaluation["ground_truth"] = (
    aggregate_evaluation[
        "ground_truth"
    ].astype(int)
)

aggregate_evaluation["outcome"] = (
    aggregate_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [151]:
display(
    pd.crosstab(
        [
            aggregate_evaluation["region"],
            aggregate_evaluation["year"],
        ],
        aggregate_evaluation["outcome"],
    )
)

outcome      FN  TN  TP
region year            
MG     2024   0  11  18
       2025   0   0   8
SP     2025   1  11   8
       2026   1   8   8

In [152]:
aggregate_counts = (
    aggregate_evaluation[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(aggregate_counts)

outcome
TP    42
FP     0
FN     2
TN    30
Name: count, dtype: int64

In [153]:
TP = int(aggregate_counts["TP"])
FP = int(aggregate_counts["FP"])
FN = int(aggregate_counts["FN"])
TN = int(aggregate_counts["TN"])

sensitivity = TP / (TP + FN)
specificity = TN / (TN + FP)
ppv = TP / (TP + FP)
npv = TN / (TN + FN)

f1 = (
    2 * TP
    / (
        2 * TP
        + FP
        + FN
    )
)

balanced_accuracy = (
    sensitivity
    + specificity
) / 2

In [154]:
aggregate_metrics = pd.DataFrame(
    [
        {
            "metric": "Sensitivity",
            "value": sensitivity,
        },
        {
            "metric": "Specificity",
            "value": specificity,
        },
        {
            "metric": "PPV",
            "value": ppv,
        },
        {
            "metric": "NPV",
            "value": npv,
        },
        {
            "metric": "F1",
            "value": f1,
        },
        {
            "metric": "Balanced Accuracy",
            "value": balanced_accuracy,
        },
    ]
)

display(aggregate_metrics)

,metric,value
0,Sensitivity,0.954545
1,Specificity,1.000000
2,PPV,1.000000
3,NPV,0.937500
4,F1,0.976744
5,Balanced Accuracy,0.977273


## 21. Desempenho agregado do modelo candidato

A avaliação agregada considerou quatro cenários independentes:

- MG/2024;
- MG/2025;
- SP/2025;
- SP/2026.

A configuração avaliada foi mantida fixa:

- exclusão de 2020 e 2021 do baseline;
- `SIGNAL` quando a razão observado/mediana histórica é >= 1,15;
- `ALERT` após duas semanas consecutivas com `SIGNAL`.

### Resultados agregados

- Sensibilidade: 95,45%
- Especificidade: 100,00%
- PPV: 100,00%
- NPV: 93,75%
- F1-score: 97,67%
- Balanced Accuracy: 97,73%

### Interpretação

O modelo candidato apresentou alta capacidade de detectar semanas
associadas a episódios epidemiológicos externos, mantendo ausência de
falsos positivos nas semanas negativas rotuladas.

A principal melhora em relação à regra original ocorreu na sensibilidade.

A regra original, baseada em Q75 e 1,5× a mediana histórica, apresentou
baixa sensibilidade para episódios de elevação moderada e persistente.

O modelo candidato mostrou melhor capacidade de detectar tanto:

- aumentos abruptos;
- elevações sustentadas de menor magnitude relativa.

### Limitações

As métricas ainda devem ser consideradas preliminares.

O conjunto de validação contém apenas duas regiões e quatro cenários
região/ano, e parte do ground truth foi construída por janelas
epidemiológicas derivadas de fontes externas.

Novas regiões e anos devem ser incorporados antes da adoção definitiva
dos parâmetros.

In [155]:
print("TP:", TP)
print("FP:", FP)
print("FN:", FN)
print("TN:", TN)
print("Total avaliado:", TP + FP + FN + TN)

TP: 42
FP: 0
FN: 2
TN: 30
Total avaliado: 74


In [156]:
alert_evaluation = aggregate_evaluation.copy()

In [158]:
alert_sp_2025 = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        validation_sp_2025[
            [
                "epi_week",
                "alert_confirmed",
            ]
        ],
        on="epi_week",
        how="inner",
    )
)

In [159]:
alert_mg_2024 = (
    mg_2024_evaluation[
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "alert_confirmed",
        ]
    ]
    .copy()
)

In [160]:
alert_base_cases = (
    ground_truth.loc[
        ground_truth["ground_truth"].notna()
    ][
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
        ]
    ]
    .merge(
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "alert_confirmed",
            ]
        ],
        on=[
            "region",
            "year",
            "epi_week",
        ],
        how="inner",
    )
)

In [161]:
aggregate_alert_evaluation = pd.concat(
    [
        alert_base_cases,
        alert_sp_2025,
        alert_mg_2024,
    ],
    ignore_index=True,
)

In [162]:
aggregate_alert_evaluation[
    "sentinela_positive"
] = (
    aggregate_alert_evaluation[
        "alert_confirmed"
    ].astype(bool)
)

aggregate_alert_evaluation[
    "ground_truth"
] = (
    aggregate_alert_evaluation[
        "ground_truth"
    ].astype(int)
)

In [163]:
aggregate_alert_evaluation[
    "outcome"
] = (
    aggregate_alert_evaluation.apply(
        classify_outcome,
        axis=1,
    )
)

In [164]:
alert_counts = (
    aggregate_alert_evaluation[
        "outcome"
    ]
    .value_counts()
    .reindex(
        [
            "TP",
            "FP",
            "FN",
            "TN",
        ],
        fill_value=0,
    )
)

display(alert_counts)

outcome
TP    41
FP     0
FN     3
TN    30
Name: count, dtype: int64

## 22. Desempenho agregado do estado ALERT

O estado `ALERT` foi avaliado separadamente do estado `SIGNAL`.

A regra utilizada foi:

- `SIGNAL`: razão observado/mediana histórica >= 1,15;
- `ALERT`: confirmação do excesso relativo em pelo menos 2 semanas consecutivas.

### Matriz de confusão

- TP: 41
- FP: 0
- FN: 3
- TN: 30

### Métricas

- Sensibilidade: 93,18%
- Especificidade: 100,00%
- PPV: 100,00%
- NPV: 90,91%
- F1-score: 96,47%
- Balanced Accuracy: 96,59%

### Interpretação

A exigência de persistência reduziu discretamente a sensibilidade em
relação ao estado `SIGNAL`, como esperado.

Entretanto, o estado `ALERT` manteve especificidade e valor preditivo
positivo máximos no conjunto avaliado.

Esse comportamento é compatível com a finalidade operacional dos
dois estados:

- `SIGNAL` prioriza detecção precoce;
- `ALERT` prioriza confirmação de uma elevação sustentada.

As métricas permanecem preliminares e devem ser reavaliadas em novas
regiões e períodos independentes.

In [165]:
signal_vs_alert = pd.DataFrame(
    [
        {
            "state": "SIGNAL",
            "sensitivity": 0.954545,
            "specificity": 1.0,
            "ppv": 1.0,
            "npv": 0.937500,
            "f1": 0.976744,
            "balanced_accuracy": 0.977273,
        },
        {
            "state": "ALERT",
            "sensitivity": 41 / 44,
            "specificity": 30 / 30,
            "ppv": 41 / 41,
            "npv": 30 / 33,
            "f1": 82 / 85,
            "balanced_accuracy": (
                (41 / 44) + 1.0
            ) / 2,
        },
    ]
)

display(signal_vs_alert)

,state,sensitivity,specificity,ppv,npv,f1,balanced_accuracy
0,SIGNAL,0.954545,1.0,1.0,0.937500,0.976744,0.977273
1,ALERT,0.931818,1.0,1.0,0.909091,0.964706,0.965909


In [166]:
display(
    aggregate_alert_evaluation.loc[
        aggregate_alert_evaluation[
            "outcome"
        ].eq("FN"),
        [
            "region",
            "year",
            "epi_week",
            "ground_truth",
            "alert_confirmed",
        ],
    ]
)

,region,year,epi_week,ground_truth,alert_confirmed
18,SP,2026,28,1,False
25,SP,2025,17,1,False
53,MG,2024,14,1,False


### Análise dos falsos negativos do ALERT

Foram identificados três falsos negativos na avaliação agregada:

- MG/2024 — SE 14;
- SP/2025 — SE 17;
- SP/2026 — SE 28.

Os casos de MG/2024 e SP/2025 correspondem às semanas iniciais dos respectivos episódios.

Nessas semanas, o estado `SIGNAL` já havia sido ativado, porém ainda não existiam duas semanas consecutivas de excesso relativo para confirmação de `ALERT`.

Portanto, esses casos representam o atraso operacional esperado da regra de persistência, e não ausência de detecção.

O terceiro caso, SP/2026 na SE 28, corresponde à fase de resolução do episódio. Nessa semana, a razão observado/mediana histórica já havia caído para aproximadamente 1,015, abaixo do limiar operacional de 1,15.

Assim, nenhum dos falsos negativos ocorreu durante a fase central de um episódio ativo.

In [167]:
episode_ground_truth = pd.DataFrame(
    [
        {
            "episode_id": "MG2024_RESP_001",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 14,
            "end_epi_week": 21,
        },
        {
            "episode_id": "MG2024_RESP_002",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 35,
            "end_epi_week": 40,
        },
        {
            "episode_id": "MG2024_RESP_003",
            "region": "MG",
            "year": 2024,
            "start_epi_week": 49,
            "end_epi_week": 52,
        },
        {
            "episode_id": "MG2025_RESP_001",
            "region": "MG",
            "year": 2025,
            "start_epi_week": 18,
            "end_epi_week": 25,
        },
        {
            "episode_id": "SP2025_RESP_001",
            "region": "SP",
            "year": 2025,
            "start_epi_week": 17,
            "end_epi_week": 25,
        },
        {
            "episode_id": "SP2026_RESP_001",
            "region": "SP",
            "year": 2026,
            "start_epi_week": 20,
            "end_epi_week": 28,
        },
    ]
)

In [168]:
predictions_mg_2024 = (
    validation_mg_2024[
        [
            "epi_week",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
    .copy()
)

predictions_mg_2024["region"] = "MG"
predictions_mg_2024["year"] = 2024


predictions_sp_2025 = (
    validation_sp_2025[
        [
            "epi_week",
            "signal_detected",
            "alert_confirmed",
        ]
    ]
    .copy()
)

predictions_sp_2025["region"] = "SP"
predictions_sp_2025["year"] = 2025

In [169]:
all_candidate_predictions = pd.concat(
    [
        candidate_predictions_all[
            [
                "region",
                "year",
                "epi_week",
                "signal_detected",
                "alert_confirmed",
            ]
        ],
        predictions_mg_2024,
        predictions_sp_2025,
    ],
    ignore_index=True,
)

In [170]:
episode_detection_rows = []

for episode in episode_ground_truth.itertuples(
    index=False
):
    episode_predictions = (
        all_candidate_predictions.loc[
            all_candidate_predictions[
                "region"
            ].eq(episode.region)
            & all_candidate_predictions[
                "year"
            ].eq(episode.year)
            & all_candidate_predictions[
                "epi_week"
            ].between(
                episode.start_epi_week,
                episode.end_epi_week,
            )
        ]
        .sort_values("epi_week")
    )

    signal_rows = episode_predictions.loc[
        episode_predictions[
            "signal_detected"
        ]
    ]

    alert_rows = episode_predictions.loc[
        episode_predictions[
            "alert_confirmed"
        ]
    ]

    first_signal_week = (
        int(signal_rows.iloc[0]["epi_week"])
        if not signal_rows.empty
        else None
    )

    first_alert_week = (
        int(alert_rows.iloc[0]["epi_week"])
        if not alert_rows.empty
        else None
    )

    signal_delay_weeks = (
        first_signal_week
        - episode.start_epi_week
        if first_signal_week is not None
        else None
    )

    alert_delay_weeks = (
        first_alert_week
        - episode.start_epi_week
        if first_alert_week is not None
        else None
    )

    episode_detection_rows.append(
        {
            "episode_id":
                episode.episode_id,
            "region":
                episode.region,
            "year":
                episode.year,
            "start_epi_week":
                episode.start_epi_week,
            "end_epi_week":
                episode.end_epi_week,
            "first_signal_week":
                first_signal_week,
            "signal_delay_weeks":
                signal_delay_weeks,
            "first_alert_week":
                first_alert_week,
            "alert_delay_weeks":
                alert_delay_weeks,
        }
    )

episode_detection = pd.DataFrame(
    episode_detection_rows
)

display(episode_detection)

,episode_id,region,year,start_epi_week,end_epi_week,first_signal_week,signal_delay_weeks,first_alert_week,alert_delay_weeks
0,MG2024_RESP_001,MG,2024,14,21,14,0,15,1
1,MG2024_RESP_002,MG,2024,35,40,35,0,35,0
2,MG2024_RESP_003,MG,2024,49,52,49,0,49,0
3,MG2025_RESP_001,MG,2025,18,25,18,0,18,0
4,SP2025_RESP_001,SP,2025,17,25,17,0,18,1
5,SP2026_RESP_001,SP,2026,20,28,20,0,20,0


In [171]:
print(
    "Episódios avaliados:",
    len(episode_detection),
)

print(
    "Episódios detectados por SIGNAL:",
    episode_detection[
        "first_signal_week"
    ].notna().sum(),
)

print(
    "Episódios confirmados por ALERT:",
    episode_detection[
        "first_alert_week"
    ].notna().sum(),
)

print(
    "Atraso médio SIGNAL:",
    episode_detection[
        "signal_delay_weeks"
    ].mean(),
    "semanas",
)

print(
    "Atraso médio ALERT:",
    episode_detection[
        "alert_delay_weeks"
    ].mean(),
    "semanas",
)

Episódios avaliados: 6
Episódios detectados por SIGNAL: 6
Episódios confirmados por ALERT: 6
Atraso médio SIGNAL: 0.0 semanas
Atraso médio ALERT: 0.3333333333333333 semanas


## 23. Avaliação pseudo-prospectiva do tempo de detecção

A avaliação semanal confirmou que todos os episódios externos foram
detectados pelo modelo candidato.

Entretanto, essa análise utiliza contagens semanais consolidadas.

Para avaliar capacidade de alerta precoce, é necessário reconstruir
o estado do Sentinela em diferentes datas após o encerramento de
cada semana epidemiológica.

Nesta etapa serão utilizados:

- registros conhecidos até cada snapshot;
- curva histórica regional de completude;
- nowcast regional;
- baseline histórico disponível antes do ano avaliado;
- limiar congelado de 1,15× a mediana histórica;
- duas semanas consecutivas para confirmação de `ALERT`.

Serão estimados:

- data do primeiro `SIGNAL`;
- data do primeiro `ALERT`;
- atraso em dias em relação ao início externo do episódio;
- antecedência em relação a respostas institucionais, quando disponível.

In [179]:
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_data_dir = (
    project_root
    / "data"
    / "raw"
)

historical_files = sorted(
    raw_data_dir.glob("INFLUD*.csv")
)

print(
    "Arquivos encontrados:",
    len(historical_files),
)

for path in historical_files:
    print(path.name)

Arquivos encontrados: 8
INFLUD19-23-03-2026.csv
INFLUD20-23-03-2026.csv
INFLUD21-23-03-2026.csv
INFLUD22-23-03-2026.csv
INFLUD23-23-03-2026.csv
INFLUD24-23-03-2026.csv
INFLUD25-28-09-2026.csv
INFLUD26-28-09-2026.csv


In [180]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

df_mg_2025_temporal = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    usecols=[
        "NU_NOTIFIC",
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_mg_2025_temporal["SG_UF"] = (
    df_mg_2025_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

mg_2025_temporal = (
    df_mg_2025_temporal.loc[
        df_mg_2025_temporal["SG_UF"].eq("MG")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    mg_2025_temporal[column] = pd.to_datetime(
        mg_2025_temporal[column],
        errors="coerce",
        dayfirst=True,
    )

/tmp/ipykernel_30676/397091653.py:38: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg_2025_temporal[column] = pd.to_datetime(
/tmp/ipykernel_30676/397091653.py:38: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg_2025_temporal[column] = pd.to_datetime(


In [182]:
model_years_mg = [
    2023,
    2024,
    2025,
]

delay_frames_mg = []

for path in historical_files:
    name = path.name.upper()

    year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if year not in model_years_mg:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    mg = df.loc[
        df["SG_UF"].eq("MG")
    ].copy()

    mg["DT_SIN_PRI"] = pd.to_datetime(
        mg["DT_SIN_PRI"],
        errors="coerce",
        dayfirst=True,
    )

    mg["DT_NOTIFIC"] = pd.to_datetime(
        mg["DT_NOTIFIC"],
        errors="coerce",
        dayfirst=True,
    )

    mg["delay_notification_days"] = (
        mg["DT_NOTIFIC"]
        - mg["DT_SIN_PRI"]
    ).dt.days

    mg["source_year"] = year

    delay_frames_mg.append(
        mg[
            [
                "source_year",
                "delay_notification_days",
            ]
        ]
    )

delay_mg = pd.concat(
    delay_frames_mg,
    ignore_index=True,
)

delay_mg = delay_mg.loc[
    delay_mg[
        "delay_notification_days"
    ].notna()
    & delay_mg[
        "delay_notification_days"
    ].ge(0)
].copy()

print(
    "Registros válidos MG:",
    len(delay_mg),
)

print(
    "Anos:",
    sorted(
        delay_mg[
            "source_year"
        ].unique()
    ),
)

/tmp/ipykernel_30676/142571632.py:48: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_NOTIFIC"] = pd.to_datetime(
/tmp/ipykernel_30676/142571632.py:42: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_SIN_PRI"] = pd.to_datetime(


Registros válidos MG: 57015
Anos: [np.int64(2023), np.int64(2024), np.int64(2025)]


/tmp/ipykernel_30676/142571632.py:42: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_SIN_PRI"] = pd.to_datetime(
/tmp/ipykernel_30676/142571632.py:48: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  mg["DT_NOTIFIC"] = pd.to_datetime(


In [195]:
mg_2025_delay_check = (
    mg_2025_temporal[
        "DT_NOTIFIC"
    ]
    - mg_2025_temporal[
        "DT_SIN_PRI"
    ]
).dt.days

mg_2025_delay_check = (
    mg_2025_delay_check.loc[
        mg_2025_delay_check.ge(0)
    ]
)

display(
    mg_2025_delay_check.describe(
        percentiles=[
            0.25,
            0.50,
            0.75,
            0.80,
            0.90,
            0.95,
        ]
    )
)

count    45655.000000
mean         8.226153
std         22.972289
min          0.000000
25%          2.000000
50%          4.000000
75%          7.000000
80%          7.000000
90%         12.000000
95%         22.000000
max        384.000000
dtype: float64

In [185]:
def build_week_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    min_completeness: float = 0.80,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(0, max_lag + 1):
        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue

        completeness = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        if completeness > 0:
            nowcast = (
                known_cases
                / completeness
            )
        else:
            nowcast = float("nan")

        ratio_to_median = (
            nowcast
            / historical_median
        )

        is_mature = (
            completeness
            >= min_completeness
        )

        signal_detected = (
            is_mature
            and ratio_to_median >= 1.15
        )

        rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "expected_completeness": completeness,
                "is_mature": is_mature,
                "nowcast": nowcast,
                "historical_median": historical_median,
                "ratio_to_median": ratio_to_median,
                "signal_detected": signal_detected,
            }
        )

    return pd.DataFrame(rows)

In [175]:
mg_2025_week_18_reference = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
    .loc[
        lambda df: df["epi_week"].eq(18)
    ]
    .iloc[0]
)

mg_2025_week_18_reference

epi_week                18.00
historical_median      659.00
historical_quantile    728.75
historical_years         4.00
Name: 17, dtype: float64

In [196]:
completion_model_mg = pd.DataFrame(
    {
        "lag_days": range(0, 29),
    }
)

completion_model_mg[
    "expected_completeness"
] = (
    completion_model_mg[
        "lag_days"
    ]
    .apply(
        lambda lag: (
            mg_2025_delay_check <= lag
        ).mean()
    )
)

completion_model_mg[
    "expected_completeness_pct"
] = (
    completion_model_mg[
        "expected_completeness"
    ]
    * 100
)

display(
    completion_model_mg.loc[
        completion_model_mg[
            "lag_days"
        ].isin(
            [0, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,expected_completeness_pct
0,0,0.066674,6.667397
7,7,0.803702,80.370168
14,14,0.920315,92.031541
21,21,0.949688,94.968788
28,28,0.959150,95.915015


In [197]:
MIN_COMPLETENESS = 0.80

mature_lag_mg = int(
    completion_model_mg.loc[
        completion_model_mg[
            "expected_completeness"
        ].ge(
            MIN_COMPLETENESS
        ),
        "lag_days",
    ].min()
)

print(
    "Primeiro lag maduro MG:",
    f"D+{mature_lag_mg}",
)

Primeiro lag maduro MG: D+7


In [198]:
timeline_mg_2025_se18 = (
    build_week_nowcast_timeline(
        temporal_df=mg_2025_temporal,
        week_start=week_start,
        week_end=week_end,
        epi_week=18,
        historical_median=float(
            mg_2025_week_18_reference[
                "historical_median"
            ]
        ),
        completion_model=completion_model_mg,
        min_completeness=0.80,
    )
)

display(
    timeline_mg_2025_se18
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,18,0,2025-05-03,477,0.066674,False,7154.216491,659.0,10.856171,False
1,18,1,2025-05-04,582,0.182214,False,3194.038947,659.0,4.846797,False
2,18,2,2025-05-05,778,0.325244,False,2392.052663,659.0,3.629822,False
3,18,3,2025-05-06,957,0.470858,False,2032.461971,659.0,3.084161,False
4,18,4,2025-05-07,1096,0.589815,False,1858.210042,659.0,2.819742,False
5,18,5,2025-05-08,1204,0.677297,False,1777.654097,659.0,2.697502,False
6,18,6,2025-05-09,1280,0.740445,False,1728.691022,659.0,2.623203,False
7,18,7,2025-05-10,1306,0.803702,True,1624.981059,659.0,2.465829,True
8,18,8,2025-05-11,1315,0.838112,True,1569.002849,659.0,2.380884,True
9,18,9,2025-05-12,1349,0.859424,True,1569.656064,659.0,2.381876,True


In [199]:
completion_years_mg_2025 = [
    2019,
    2022,
    2023,
    2024,
]

In [200]:
delay_frames = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_mg_2025:
        continue

    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq("MG")
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                errors="coerce",
                dayfirst=True,
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df = df.loc[
        df["delay_days"].notna()
        & df["delay_days"].ge(0)
    ].copy()

    df["source_year"] = source_year

    delay_frames.append(
        df[
            [
                "source_year",
                "delay_days",
            ]
        ]
    )

delay_historical_mg_2025 = pd.concat(
    delay_frames,
    ignore_index=True,
)

/tmp/ipykernel_30676/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_30676/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_30676/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_30676/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  pd.to_datetime(
/tmp/ipykernel_30676/713114665.py:41: UserWarning: Parsing dates in %Y-%m-%dT%H:%M:%S.%f%z format when dayfirst=True was specified. 

In [201]:
display(
    delay_historical_mg_2025
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
            0.95,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,95%,max
source_year,,,,,,,,,,
2019,4035.0,7.745725,24.045400,0.0,5.0,8.0,9.0,14.0,20.0,753.0
2022,11843.0,105.824960,87.679130,0.0,90.0,155.0,180.0,238.0,276.0,949.0
2023,5388.0,108.791574,88.010105,0.0,92.0,175.0,183.0,240.0,271.0,700.0
2024,5972.0,106.882954,81.295476,0.0,93.0,155.0,180.0,229.9,266.0,572.0


In [216]:
def load_notification_delays(
    path: Path,
    region: str,
    source_year: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq(region)
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df["source_year"] = source_year

    return df

In [233]:
notification_delay_frames = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    delays = load_notification_delays(
        path=path,
        region="MG",
        source_year=source_year,
    )

    notification_delay_frames.append(
        delays
    )

notification_delays_all = pd.concat(
    notification_delay_frames,
    ignore_index=True,
)

valid_notification_delays = (
    notification_delays_all.loc[
        notification_delays_all["delay_days"].notna()
        & notification_delays_all["delay_days"].ge(0)
    ]
    .copy()
)

print(
    valid_notification_delays
    .groupby("source_year")["delay_days"]
    .median()
)

source_year
2019    5.0
2020    5.0
2021    7.0
2022    4.0
2023    3.0
2024    4.0
2025    4.0
2026    4.0
Name: delay_days, dtype: float64


In [218]:
valid_notification_delays = (
    notification_delays_all.loc[
        notification_delays_all[
            "delay_days"
        ].notna()
        & notification_delays_all[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [219]:
delay_evolution = (
    valid_notification_delays
    .groupby("source_year")
    .agg(
        records=(
            "delay_days",
            "size",
        ),
        median_delay=(
            "delay_days",
            "median",
        ),
        p75_delay=(
            "delay_days",
            lambda x: x.quantile(0.75),
        ),
        p80_delay=(
            "delay_days",
            lambda x: x.quantile(0.80),
        ),
        p90_delay=(
            "delay_days",
            lambda x: x.quantile(0.90),
        ),
        completeness_d7=(
            "delay_days",
            lambda x: x.le(7).mean(),
        ),
        completeness_d14=(
            "delay_days",
            lambda x: x.le(14).mean(),
        ),
        completeness_d28=(
            "delay_days",
            lambda x: x.le(28).mean(),
        ),
    )
    .reset_index()
)

display(delay_evolution)

,source_year,records,median_delay,p75_delay,p80_delay,p90_delay,completeness_d7,completeness_d14,completeness_d28
0,2019,4035,5.0,8.0,9.0,14.0,0.712763,0.910781,0.967286
1,2020,113781,5.0,9.0,10.0,14.0,0.681379,0.913931,0.974732
2,2021,209248,7.0,10.0,11.0,15.0,0.550280,0.897437,0.973811
3,2022,69332,4.0,7.0,8.0,11.0,0.799660,0.936898,0.976807
4,2023,25613,3.0,6.0,7.0,11.0,0.821146,0.929450,0.964784
5,2024,27271,4.0,7.0,7.0,12.0,0.808698,0.924865,0.965714
6,2025,45655,4.0,7.0,7.0,12.0,0.803702,0.920315,0.959150
7,2026,29087,4.0,6.0,7.0,11.0,0.818235,0.935882,0.975866


In [234]:
completion_years_mg_2025 = [
    2019,
    2022,
    2023,
    2024,
]

historical_delays_mg_2025 = (
    valid_notification_delays.loc[
        valid_notification_delays[
            "source_year"
        ].isin(completion_years_mg_2025)
    ]
    .copy()
)

In [235]:
for year, group in (
    historical_delays_mg_2025
    .groupby("source_year")
):
    print(
        year,
        "D+7:",
        round(
            group["delay_days"]
            .le(7)
            .mean(),
            4,
        ),
    )

2019 D+7: 0.7128
2022 D+7: 0.7997
2023 D+7: 0.8211
2024 D+7: 0.8087


In [240]:
completion_rows = []

for year, group in (
    historical_delays_mg_2025
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_mg_2025 = pd.DataFrame(
    completion_rows
)

completion_model_mg_2025 = (
    completion_by_year_mg_2025
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_mg_2025[
    "expected_completeness_pct"
] = (
    completion_model_mg_2025[
        "expected_completeness"
    ]
    * 100
)

In [241]:
display(
    completion_model_mg_2025.loc[
        completion_model_mg_2025[
            "lag_days"
        ].isin(
            [0, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.072077,0.035440,0.077848,7.207700
7,7,0.804179,0.712763,0.821146,80.417875
14,14,0.927158,0.910781,0.936898,92.715757
21,21,0.956770,0.956107,0.967749,95.676998
28,28,0.966500,0.964784,0.976807,96.650037


In [242]:
timeline_mg_2025_se18 = (
    build_week_nowcast_timeline(
        temporal_df=mg_2025_temporal,
        week_start=week_start,
        week_end=week_end,
        epi_week=18,
        historical_median=float(
            mg_2025_week_18_reference[
                "historical_median"
            ]
        ),
        completion_model=completion_model_mg_2025,
        min_completeness=0.80,
    )
)

display(timeline_mg_2025_se18)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,18,0,2025-05-03,477,0.072077,False,6617.922334,659.0,10.042371,False
1,18,1,2025-05-04,582,0.188388,False,3089.373974,659.0,4.687973,False
2,18,2,2025-05-05,778,0.328573,False,2367.818055,659.0,3.593047,False
3,18,3,2025-05-06,957,0.476074,False,2010.192921,659.0,3.050369,False
4,18,4,2025-05-07,1096,0.594256,False,1844.323528,659.0,2.798670,False
5,18,5,2025-05-08,1204,0.679427,False,1772.082618,659.0,2.689048,False
6,18,6,2025-05-09,1280,0.742235,False,1724.521114,659.0,2.616876,False
7,18,7,2025-05-10,1306,0.804179,True,1624.017055,659.0,2.464366,True
8,18,8,2025-05-11,1315,0.842383,True,1561.047624,659.0,2.368813,True
9,18,9,2025-05-12,1349,0.866479,True,1556.876221,659.0,2.362483,True


In [243]:
first_signal_mg_2025_se18 = (
    timeline_mg_2025_se18.loc[
        timeline_mg_2025_se18[
            "signal_detected"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

display(first_signal_mg_2025_se18)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
7,18,7,2025-05-10,1306,0.804179,True,1624.017055,659.0,2.464366,True


In [244]:
if not first_signal_mg_2025_se18.empty:
    row = first_signal_mg_2025_se18.iloc[0]

    print(
        "Primeiro SIGNAL operacional:",
        row["snapshot_date"].date(),
    )

    print(
        "Lag:",
        f"D+{int(row['lag_days'])}",
    )

    print(
        "Nowcast:",
        round(row["nowcast"], 1),
    )

    print(
        "Razão para mediana:",
        round(row["ratio_to_median"], 3),
    )

Primeiro SIGNAL operacional: 2025-05-10
Lag: D+7
Nowcast: 1624.0
Razão para mediana: 2.464


In [245]:
weeks_mg_2025 = pd.DataFrame(
    {
        "epi_week": range(17, 26),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_mg_2025["week_start"] = (
    first_week_start
    + pd.to_timedelta(
        (weeks_mg_2025["epi_week"] - 1) * 7,
        unit="D",
    )
)

weeks_mg_2025["week_end"] = (
    weeks_mg_2025["week_start"]
    + pd.Timedelta(days=6)
)

display(weeks_mg_2025)

,epi_week,week_start,week_end
0,17,2025-04-20,2025-04-26
1,18,2025-04-27,2025-05-03
2,19,2025-05-04,2025-05-10
3,20,2025-05-11,2025-05-17
4,21,2025-05-18,2025-05-24
5,22,2025-05-25,2025-05-31
6,23,2025-06-01,2025-06-07
7,24,2025-06-08,2025-06-14
8,25,2025-06-15,2025-06-21


In [246]:
reference_mg_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_mg,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
)

reference_mg_2025 = (
    reference_mg_2025.loc[
        reference_mg_2025[
            "epi_week"
        ].between(17, 25)
    ]
    .copy()
)

display(reference_mg_2025)

,epi_week,historical_median,historical_quantile,historical_years
16,17,600.0,693.00,4
17,18,659.0,728.75,4
18,19,632.0,777.50,4
19,20,670.0,854.00,4
20,21,654.5,880.75,4
21,22,598.0,884.50,4
22,23,582.0,903.75,4
23,24,544.0,910.50,4
24,25,588.5,973.00,4


In [247]:
timeline_frames = []

for week in weeks_mg_2025.itertuples(
    index=False
):
    reference_row = (
        reference_mg_2025.loc[
            reference_mg_2025[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=mg_2025_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_mg_2025
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames.append(timeline)

timeline_mg_2025 = pd.concat(
    timeline_frames,
    ignore_index=True,
)

display(
    timeline_mg_2025.head()
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,is_mature,nowcast,historical_median,ratio_to_median,signal_detected
0,17,0,2025-04-26,412,0.072077,False,5716.109018,600.0,9.526848,False
1,17,1,2025-04-27,483,0.188388,False,2563.861906,600.0,4.273103,False
2,17,2,2025-04-28,636,0.328573,False,1935.645608,600.0,3.226076,False
3,17,3,2025-04-29,727,0.476074,False,1527.074455,600.0,2.545124,False
4,17,4,2025-04-30,800,0.594256,False,1346.221553,600.0,2.243703,False


In [248]:
first_signal_by_week = (
    timeline_mg_2025.loc[
        timeline_mg_2025[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,17,7,2025-05-03,879,1093.040576,600.0,1.821734
1,18,7,2025-05-10,1306,1624.017055,659.0,2.464366
2,19,7,2025-05-17,1642,2041.834613,632.0,3.230751
3,20,7,2025-05-24,1908,2372.606847,670.0,3.541204
4,21,7,2025-05-31,2098,2608.872728,654.5,3.986055
5,22,7,2025-06-07,1945,2418.616518,598.0,4.044509
6,23,7,2025-06-14,2007,2495.713806,582.0,4.288168
7,24,7,2025-06-21,1608,1999.555456,544.0,3.675653
8,25,7,2025-06-28,1488,1850.334899,588.5,3.144154


In [249]:
weekly_signal_summary = (
    weeks_mg_2025[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary[
    "has_signal"
] = (
    weekly_signal_summary[
        "snapshot_date"
    ].notna()
)

display(weekly_signal_summary)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal
0,17,2025-04-20,2025-04-26,2025-05-03,7,1.821734,True
1,18,2025-04-27,2025-05-03,2025-05-10,7,2.464366,True
2,19,2025-05-04,2025-05-10,2025-05-17,7,3.230751,True
3,20,2025-05-11,2025-05-17,2025-05-24,7,3.541204,True
4,21,2025-05-18,2025-05-24,2025-05-31,7,3.986055,True
5,22,2025-05-25,2025-05-31,2025-06-07,7,4.044509,True
6,23,2025-06-01,2025-06-07,2025-06-14,7,4.288168,True
7,24,2025-06-08,2025-06-14,2025-06-21,7,3.675653,True
8,25,2025-06-15,2025-06-21,2025-06-28,7,3.144154,True


In [250]:
weekly_signal_summary[
    "previous_week_signal"
] = (
    weekly_signal_summary[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary[
    "alert_candidate"
] = (
    weekly_signal_summary[
        "has_signal"
    ]
    & weekly_signal_summary[
        "previous_week_signal"
    ]
)

In [251]:
weekly_signal_summary[
    "previous_signal_date"
] = (
    weekly_signal_summary[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary[
        "alert_candidate"
    ]
)

weekly_signal_summary.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

In [252]:
display(
    weekly_signal_summary[
        [
            "epi_week",
            "has_signal",
            "snapshot_date",
            "previous_week_signal",
            "alert_candidate",
            "alert_date",
        ]
    ]
)

,epi_week,has_signal,snapshot_date,previous_week_signal,alert_candidate,alert_date
0,17,True,2025-05-03,False,False,NaT
1,18,True,2025-05-10,True,True,2025-05-10
2,19,True,2025-05-17,True,True,2025-05-17
3,20,True,2025-05-24,True,True,2025-05-24
4,21,True,2025-05-31,True,True,2025-05-31
5,22,True,2025-06-07,True,True,2025-06-07
6,23,True,2025-06-14,True,True,2025-06-14
7,24,True,2025-06-21,True,True,2025-06-21
8,25,True,2025-06-28,True,True,2025-06-28


In [253]:
first_signal = (
    weekly_signal_summary.loc[
        weekly_signal_summary[
            "has_signal"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

In [254]:
first_alert = (
    weekly_signal_summary.loc[
        weekly_signal_summary[
            "alert_candidate"
        ]
    ]
    .sort_values("alert_date")
    .head(1)
)

In [255]:
print("=== PRIMEIRO SIGNAL ===")

display(
    first_signal[
        [
            "epi_week",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
        ]
    ]
)

print("\n=== PRIMEIRO ALERT ===")

display(
    first_alert[
        [
            "epi_week",
            "alert_date",
        ]
    ]
)

=== PRIMEIRO SIGNAL ===


,epi_week,snapshot_date,lag_days,ratio_to_median
0,17,2025-05-03,7,1.821734



=== PRIMEIRO ALERT ===


,epi_week,alert_date
1,18,2025-05-10


In [256]:
official_events = {
    "Decreto estadual": pd.Timestamp(
        "2025-05-02"
    ),
    "Ativação do COE": pd.Timestamp(
        "2025-05-08"
    ),
}

In [257]:
first_signal_date = (
    pd.Timestamp(
        first_signal[
            "snapshot_date"
        ].iloc[0]
    )
    if not first_signal.empty
    else pd.NaT
)

first_alert_date = (
    pd.Timestamp(
        first_alert[
            "alert_date"
        ].iloc[0]
    )
    if not first_alert.empty
    else pd.NaT
)

In [258]:
lead_time_rows = []

for event_name, event_date in (
    official_events.items()
):
    lead_time_rows.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "SIGNAL",
            "sentinela_date": first_signal_date,
            "lead_days": (
                event_date
                - first_signal_date
            ).days,
        }
    )

    lead_time_rows.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "ALERT",
            "sentinela_date": first_alert_date,
            "lead_days": (
                event_date
                - first_alert_date
            ).days,
        }
    )

lead_time_mg_2025 = pd.DataFrame(
    lead_time_rows
)

display(lead_time_mg_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days
0,Decreto estadual,2025-05-02,SIGNAL,2025-05-03,-1
1,Decreto estadual,2025-05-02,ALERT,2025-05-10,-8
2,Ativação do COE,2025-05-08,SIGNAL,2025-05-03,5
3,Ativação do COE,2025-05-08,ALERT,2025-05-10,-2


## 24. Lead time pseudo-prospectivo — MG/2025

A reconstrução pseudo-prospectiva utilizou exclusivamente informações
que estariam disponíveis até cada data de snapshot.

Foram mantidos fixos:

- baseline histórico anterior a 2025;
- exclusão dos anos pandêmicos do baseline epidemiológico;
- razão observado/mediana >= 1,15;
- completude mínima de 80%;
- duas semanas consecutivas para confirmação de `ALERT`.

### Resultado

A SE 17 atingiu os critérios operacionais em 03/05/2025,
quando alcançou maturidade suficiente dos dados.

Esse foi o primeiro `SIGNAL` pseudo-prospectivo.

A SE 18 atingiu os mesmos critérios em 10/05/2025.
Como a SE 17 também permanecia sinalizada, essa data representa a
primeira confirmação de `ALERT`.

### Comparação com respostas institucionais

- Decreto estadual de emergência: 02/05/2025
  - SIGNAL: 1 dia depois;
  - ALERT: 8 dias depois.

- Ativação do Centro de Operações de Emergência: 08/05/2025
  - SIGNAL: 5 dias antes;
  - ALERT: 2 dias depois.

### Interpretação

O estado `SIGNAL` apresentou capacidade de detecção próxima ao primeiro
marco institucional e antecedeu a ativação do COE.

O estado `ALERT`, por exigir persistência em duas semanas epidemiológicas
consecutivas, apresentou atraso adicional esperado.

Esses resultados apoiam a utilização de `SIGNAL` como mecanismo de
detecção precoce e `ALERT` como confirmação operacional.

In [259]:
lead_time_mg_2025["interpretation"] = (
    lead_time_mg_2025["lead_days"]
    .apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else f"{abs(days)} dias depois"
            )
        )
    )
)

display(lead_time_mg_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,Decreto estadual,2025-05-02,SIGNAL,2025-05-03,-1,1 dias depois
1,Decreto estadual,2025-05-02,ALERT,2025-05-10,-8,8 dias depois
2,Ativação do COE,2025-05-08,SIGNAL,2025-05-03,5,5 dias antes
3,Ativação do COE,2025-05-08,ALERT,2025-05-10,-2,2 dias depois


In [260]:
completion_years_sp_2025 = [
    2019,
    2022,
    2023,
    2024,
]

In [261]:
def load_notification_delays_sp(
    path: Path,
    source_year: int,
) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=";",
        encoding="latin-1",
        usecols=[
            "SG_UF",
            "DT_SIN_PRI",
            "DT_NOTIFIC",
        ],
        dtype="string",
        low_memory=False,
    )

    df["SG_UF"] = (
        df["SG_UF"]
        .str.strip()
        .str.upper()
    )

    df = df.loc[
        df["SG_UF"].eq("SP")
    ].copy()

    for column in [
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ]:
        df[column] = (
            pd.to_datetime(
                df[column],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

    df["delay_days"] = (
        df["DT_NOTIFIC"]
        - df["DT_SIN_PRI"]
    ).dt.days

    df["source_year"] = source_year

    return df

In [262]:
delay_frames_sp = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_sp_2025:
        continue

    delays = load_notification_delays_sp(
        path=path,
        source_year=source_year,
    )

    delay_frames_sp.append(delays)

historical_delays_sp_2025 = pd.concat(
    delay_frames_sp,
    ignore_index=True,
)

historical_delays_sp_2025 = (
    historical_delays_sp_2025.loc[
        historical_delays_sp_2025[
            "delay_days"
        ].notna()
        & historical_delays_sp_2025[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [263]:
display(
    historical_delays_sp_2025
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,max
source_year,,,,,,,,,
2019,12331.0,9.048009,19.966451,0.0,5.0,8.0,10.0,17.0,597.0
2022,160930.0,7.628292,21.866400,0.0,4.0,8.0,9.0,14.0,1104.0
2023,78490.0,7.521506,21.935172,0.0,4.0,7.0,8.0,13.0,750.0
2024,69006.0,8.874025,22.105549,0.0,4.0,7.0,8.0,14.0,445.0


In [264]:
completion_rows_sp = []

for year, group in (
    historical_delays_sp_2025
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_sp.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_sp_2025 = pd.DataFrame(
    completion_rows_sp
)

completion_model_sp_2025 = (
    completion_by_year_sp_2025
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_sp_2025[
    "expected_completeness_pct"
] = (
    completion_model_sp_2025[
        "expected_completeness"
    ]
    * 100
)

In [265]:
display(
    completion_model_sp_2025.loc[
        completion_model_sp_2025[
            "lag_days"
        ].isin(
            [0, 7, 8, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.072473,0.032682,0.076073,7.247270
7,7,0.753612,0.722001,0.786024,75.361182
8,8,0.796093,0.763523,0.824729,79.609336
14,14,0.905568,0.875193,0.919098,90.556751
21,21,0.946681,0.924337,0.955752,94.668143
28,28,0.961084,0.944692,0.969308,96.108387


In [266]:
mature_rows_sp = (
    completion_model_sp_2025.loc[
        completion_model_sp_2025[
            "expected_completeness"
        ].ge(0.80)
    ]
)

mature_lag_sp = int(
    mature_rows_sp[
        "lag_days"
    ].min()
)

print(
    "Primeiro lag maduro SP/2025:",
    f"D+{mature_lag_sp}",
)

Primeiro lag maduro SP/2025: D+9


In [267]:
path_2025 = next(
    path
    for path in historical_files
    if "INFLUD25" in path.name.upper()
)

df_sp_2025_temporal = pd.read_csv(
    path_2025,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_sp_2025_temporal["SG_UF"] = (
    df_sp_2025_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

sp_2025_temporal = (
    df_sp_2025_temporal.loc[
        df_sp_2025_temporal[
            "SG_UF"
        ].eq("SP")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    sp_2025_temporal[column] = (
        pd.to_datetime(
            sp_2025_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

sp_2025_temporal = (
    sp_2025_temporal.loc[
        sp_2025_temporal[
            "DT_SIN_PRI"
        ].notna()
        & sp_2025_temporal[
            "DT_NOTIFIC"
        ].notna()
    ]
    .copy()
)

print(
    "Registros SP/2025:",
    len(sp_2025_temporal),
)

Registros SP/2025: 79295


In [268]:
weeks_sp_2025 = pd.DataFrame(
    {
        "epi_week": range(16, 27),
    }
)

first_week_start_sp_2025 = pd.Timestamp(
    "2024-12-29"
)

weeks_sp_2025["week_start"] = (
    first_week_start_sp_2025
    + pd.to_timedelta(
        (
            weeks_sp_2025["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2025["week_end"] = (
    weeks_sp_2025["week_start"]
    + pd.Timedelta(days=6)
)

In [269]:
reference_sp_2025 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
        quantile=0.75,
    )
)

In [270]:
timeline_frames_sp = []

for week in weeks_sp_2025.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2025.loc[
            reference_sp_2025[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=sp_2025_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_sp_2025
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames_sp.append(timeline)

timeline_sp_2025 = pd.concat(
    timeline_frames_sp,
    ignore_index=True,
)

In [271]:
first_signal_by_week_sp = (
    timeline_sp_2025.loc[
        timeline_sp_2025[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week_sp[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,16,9,2025-04-28,1715,2077.494511,1796.0,1.156734
1,17,9,2025-05-05,2013,2438.481895,1711.5,1.424763
2,18,9,2025-05-12,2144,2597.170980,1755.0,1.479870
3,19,9,2025-05-19,2338,2832.176190,1592.5,1.778447
4,20,9,2025-05-26,2749,3330.048053,1676.0,1.986902
5,21,9,2025-06-02,2815,3409.998279,1618.5,2.106888
6,22,9,2025-06-09,2511,3041.742692,1531.5,1.986120
7,23,9,2025-06-16,2504,3033.263123,1573.5,1.927717
8,24,9,2025-06-23,2110,2555.984500,1530.5,1.670032
9,25,9,2025-06-30,2198,2662.584802,1607.5,1.656351


In [274]:
weekly_signal_summary_sp = (
    weeks_sp_2025[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week_sp[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary_sp[
    "has_signal"
] = (
    weekly_signal_summary_sp[
        "snapshot_date"
    ].notna()
)

weekly_signal_summary_sp[
    "previous_week_signal"
] = (
    weekly_signal_summary_sp[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary_sp[
    "alert_candidate"
] = (
    weekly_signal_summary_sp[
        "has_signal"
    ]
    & weekly_signal_summary_sp[
        "previous_week_signal"
    ]
)

weekly_signal_summary_sp[
    "previous_signal_date"
] = (
    weekly_signal_summary_sp[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary_sp[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary_sp[
        "alert_candidate"
    ]
)

weekly_signal_summary_sp.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary_sp.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

display(weekly_signal_summary_sp)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,16,2025-04-13,2025-04-19,2025-04-28,9,1.156734,True,False,False,NaT,NaT
1,17,2025-04-20,2025-04-26,2025-05-05,9,1.424763,True,True,True,2025-04-28,2025-05-05
2,18,2025-04-27,2025-05-03,2025-05-12,9,1.479870,True,True,True,2025-05-05,2025-05-12
3,19,2025-05-04,2025-05-10,2025-05-19,9,1.778447,True,True,True,2025-05-12,2025-05-19
4,20,2025-05-11,2025-05-17,2025-05-26,9,1.986902,True,True,True,2025-05-19,2025-05-26
5,21,2025-05-18,2025-05-24,2025-06-02,9,2.106888,True,True,True,2025-05-26,2025-06-02
6,22,2025-05-25,2025-05-31,2025-06-09,9,1.986120,True,True,True,2025-06-02,2025-06-09
7,23,2025-06-01,2025-06-07,2025-06-16,9,1.927717,True,True,True,2025-06-09,2025-06-16
8,24,2025-06-08,2025-06-14,2025-06-23,9,1.670032,True,True,True,2025-06-16,2025-06-23
9,25,2025-06-15,2025-06-21,2025-06-30,9,1.656351,True,True,True,2025-06-23,2025-06-30


In [273]:
first_signal_sp = (
    weekly_signal_summary_sp.loc[
        weekly_signal_summary_sp[
            "has_signal"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

first_alert_sp = (
    weekly_signal_summary_sp.loc[
        weekly_signal_summary_sp[
            "alert_candidate"
        ]
    ]
    .sort_values("alert_date")
    .head(1)
)

print("=== PRIMEIRO SIGNAL SP/2025 ===")
display(first_signal_sp)

print("\n=== PRIMEIRO ALERT SP/2025 ===")
display(first_alert_sp)

=== PRIMEIRO SIGNAL SP/2025 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,16,2025-04-13,2025-04-19,2025-04-28,9,1.156734,True,False,False,NaT,NaT



=== PRIMEIRO ALERT SP/2025 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
1,17,2025-04-20,2025-04-26,2025-05-05,9,1.424763,True,True,True,2025-04-28,2025-05-05


In [275]:
first_signal_sp_date = pd.Timestamp(
    first_signal_sp[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_date = pd.Timestamp(
    first_alert_sp[
        "alert_date"
    ].iloc[0]
)

print(
    "Primeiro SIGNAL SP/2025:",
    first_signal_sp_date.date(),
)

print(
    "Primeiro ALERT SP/2025:",
    first_alert_sp_date.date(),
)

Primeiro SIGNAL SP/2025: 2025-04-28
Primeiro ALERT SP/2025: 2025-05-05


## 25. Reconstrução pseudo-prospectiva — SP/2025

Mantendo os mesmos parâmetros definidos anteriormente:

- razão observado/mediana >= 1,15;
- completude mínima de 80%;
- `ALERT` após duas semanas consecutivas com `SIGNAL`;
- baseline construído apenas com anos anteriores ao ano avaliado;
- sem recalibração específica para São Paulo.

### Resultado

O primeiro `SIGNAL` operacional ocorreu em 28/04/2025, referente à SE 16.

Naquele momento:

- lag de maturidade: D+9;
- razão nowcast/mediana histórica: 1,157.

A SE 17 atingiu o critério operacional em 05/05/2025.

Como a SE 16 também havia sido sinalizada, essa foi a primeira data de
confirmação do estado `ALERT`.

### Interpretação preliminar

O sistema detectou uma elevação ainda durante a fase de transição
epidemiológica, antes do início da janela positiva forte utilizada no
ground truth externo.

Esse resultado sugere capacidade de antecipação do episódio, sem alteração
dos parâmetros previamente definidos.

In [276]:
official_events_sp_2025 = {
    "Incentivo excepcional para SRAG":
        pd.Timestamp("2025-05-06"),

    "Boletim estadual atualizado até SE20":
        pd.Timestamp("2025-05-12"),
}

In [277]:
first_signal_sp_date = pd.Timestamp(
    first_signal_sp[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_date = pd.Timestamp(
    first_alert_sp[
        "alert_date"
    ].iloc[0]
)

lead_time_rows_sp = []

for event_name, event_date in (
    official_events_sp_2025.items()
):
    lead_time_rows_sp.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "SIGNAL",
            "sentinela_date":
                first_signal_sp_date,
            "lead_days": (
                event_date
                - first_signal_sp_date
            ).days,
        }
    )

    lead_time_rows_sp.append(
        {
            "reference_event": event_name,
            "official_date": event_date,
            "sentinela_state": "ALERT",
            "sentinela_date":
                first_alert_sp_date,
            "lead_days": (
                event_date
                - first_alert_sp_date
            ).days,
        }
    )

lead_time_sp_2025 = pd.DataFrame(
    lead_time_rows_sp
)

lead_time_sp_2025["interpretation"] = (
    lead_time_sp_2025[
        "lead_days"
    ].apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else (
                    f"{abs(days)} dias depois"
                )
            )
        )
    )
)

display(lead_time_sp_2025)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,Incentivo excepcional para SRAG,2025-05-06,SIGNAL,2025-04-28,8,8 dias antes
1,Incentivo excepcional para SRAG,2025-05-06,ALERT,2025-05-05,1,1 dias antes
2,Boletim estadual atualizado até SE20,2025-05-12,SIGNAL,2025-04-28,14,14 dias antes
3,Boletim estadual atualizado até SE20,2025-05-12,ALERT,2025-05-05,7,7 dias antes


### Evidência preliminar de capacidade de alerta precoce

Na reconstrução pseudo-prospectiva de SP/2025, o Sentinela teria emitido
seu primeiro `SIGNAL` em 28/04/2025 e confirmado `ALERT` em 05/05/2025.

Essas datas antecedem marcos externos utilizados como referência:

- o `SIGNAL` antecedeu em 8 dias o incentivo excepcional relacionado à SRAG;
- o `ALERT` antecedeu esse marco em 1 dia;
- em relação ao boletim estadual de 12/05, o `SIGNAL` apresentou 14 dias
  de antecedência e o `ALERT`, 7 dias.

Os parâmetros não foram recalibrados especificamente para SP/2025.

Esse resultado fornece evidência preliminar de que o modelo pode identificar
uma elevação epidemiológica antes de sua consolidação em marcos externos,
embora novas regiões, anos e episódios ainda sejam necessários para estimar
formalmente a capacidade de alerta precoce.

In [278]:
completion_years_sp_2026 = [
    2019,
    2022,
    2023,
    2024,
    2025,
]

In [279]:
delay_frames_sp_2026 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_sp_2026:
        continue

    delays = load_notification_delays_sp(
        path=path,
        source_year=source_year,
    )

    delay_frames_sp_2026.append(delays)

historical_delays_sp_2026 = pd.concat(
    delay_frames_sp_2026,
    ignore_index=True,
)

historical_delays_sp_2026 = (
    historical_delays_sp_2026.loc[
        historical_delays_sp_2026[
            "delay_days"
        ].notna()
        & historical_delays_sp_2026[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [280]:
display(
    historical_delays_sp_2026
    .groupby("source_year")[
        "delay_days"
    ]
    .describe(
        percentiles=[
            0.50,
            0.75,
            0.80,
            0.90,
        ]
    )
)

,count,mean,std,min,50%,75%,80%,90%,max
source_year,,,,,,,,,
2019,12331.0,9.048009,19.966451,0.0,5.0,8.0,10.0,17.0,597.0
2022,160930.0,7.628292,21.866400,0.0,4.0,8.0,9.0,14.0,1104.0
2023,78490.0,7.521506,21.935172,0.0,4.0,7.0,8.0,13.0,750.0
2024,69006.0,8.874025,22.105549,0.0,4.0,7.0,8.0,14.0,445.0
2025,79295.0,6.997730,16.288858,0.0,4.0,7.0,8.0,12.0,573.0


In [281]:
completion_rows_sp_2026 = []

for year, group in (
    historical_delays_sp_2026
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_sp_2026.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_sp_2026 = pd.DataFrame(
    completion_rows_sp_2026
)

completion_model_sp_2026 = (
    completion_by_year_sp_2026
    .groupby(
        "lag_days",
        as_index=False,
    )
    .agg(
        expected_completeness=(
            "completeness",
            "median",
        ),
        min_completeness=(
            "completeness",
            "min",
        ),
        max_completeness=(
            "completeness",
            "max",
        ),
    )
)

completion_model_sp_2026[
    "expected_completeness_pct"
] = (
    completion_model_sp_2026[
        "expected_completeness"
    ]
    * 100
)

In [282]:
display(
    completion_model_sp_2026.loc[
        completion_model_sp_2026[
            "lag_days"
        ].isin(
            [0, 7, 8, 9, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,min_completeness,max_completeness,expected_completeness_pct
0,0,0.070690,0.032682,0.076073,7.068951
7,7,0.759905,0.722001,0.791525,75.990494
8,8,0.802278,0.763523,0.829876,80.227806
9,9,0.831609,0.793123,0.855174,83.160885
14,14,0.905866,0.875193,0.921912,90.586590
21,21,0.951532,0.924337,0.956441,95.153172
28,28,0.967265,0.944692,0.969308,96.726527


In [283]:
mature_rows_sp_2026 = (
    completion_model_sp_2026.loc[
        completion_model_sp_2026[
            "expected_completeness"
        ].ge(0.80)
    ]
)

mature_lag_sp_2026 = int(
    mature_rows_sp_2026[
        "lag_days"
    ].min()
)

print(
    "Primeiro lag maduro SP/2026:",
    f"D+{mature_lag_sp_2026}",
)

Primeiro lag maduro SP/2026: D+8


In [284]:
path_2026 = next(
    path
    for path in historical_files
    if "INFLUD26" in path.name.upper()
)

df_sp_2026_temporal = pd.read_csv(
    path_2026,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_sp_2026_temporal["SG_UF"] = (
    df_sp_2026_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

sp_2026_temporal = (
    df_sp_2026_temporal.loc[
        df_sp_2026_temporal[
            "SG_UF"
        ].eq("SP")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    sp_2026_temporal[column] = (
        pd.to_datetime(
            sp_2026_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

sp_2026_temporal = (
    sp_2026_temporal.loc[
        sp_2026_temporal[
            "DT_SIN_PRI"
        ].notna()
        & sp_2026_temporal[
            "DT_NOTIFIC"
        ].notna()
    ]
    .copy()
)

print(
    "Registros SP/2026:",
    len(sp_2026_temporal),
)

Registros SP/2026: 48978


In [285]:
weeks_sp_2026 = pd.DataFrame(
    {
        "epi_week": range(18, 31),
    }
)

first_week_start_sp_2026 = pd.Timestamp(
    "2025-12-28"
)

weeks_sp_2026["week_start"] = (
    first_week_start_sp_2026
    + pd.to_timedelta(
        (
            weeks_sp_2026["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2026["week_end"] = (
    weeks_sp_2026["week_start"]
    + pd.Timedelta(days=6)
)

display(weeks_sp_2026)

,epi_week,week_start,week_end
0,18,2026-04-26,2026-05-02
1,19,2026-05-03,2026-05-09
2,20,2026-05-10,2026-05-16
3,21,2026-05-17,2026-05-23
4,22,2026-05-24,2026-05-30
5,23,2026-05-31,2026-06-06
6,24,2026-06-07,2026-06-13
7,25,2026-06-14,2026-06-20
8,26,2026-06-21,2026-06-27
9,27,2026-06-28,2026-07-04


In [286]:
reference_sp_2026 = (
    build_reference_selected_years(
        historical_weekly=historical_sp,
        years=[
            2019,
            2022,
            2023,
            2024,
            2025,
        ],
        quantile=0.75,
    )
)

In [287]:
timeline_frames_sp_2026 = []

for week in weeks_sp_2026.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2026.loc[
            reference_sp_2026[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_week_nowcast_timeline(
            temporal_df=sp_2026_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_model_sp_2026
            ),
            min_completeness=0.80,
        )
    )

    timeline_frames_sp_2026.append(
        timeline
    )

timeline_sp_2026 = pd.concat(
    timeline_frames_sp_2026,
    ignore_index=True,
)

In [288]:
first_signal_by_week_sp_2026 = (
    timeline_sp_2026.loc[
        timeline_sp_2026[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_by_week_sp_2026[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "known_cases",
            "nowcast",
            "historical_median",
            "ratio_to_median",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,known_cases,nowcast,historical_median,ratio_to_median
0,19,8,2026-05-17,1562,1946.955890,1688.0,1.153410
1,20,8,2026-05-24,1932,2408.142625,1869.0,1.288466
2,21,8,2026-05-31,2045,2548.991547,1872.0,1.361641
3,22,8,2026-06-07,1968,2453.014848,1692.0,1.449772
4,23,8,2026-06-14,2116,2637.489542,1574.0,1.675660
5,24,8,2026-06-21,2007,2501.626422,1629.0,1.535682
6,25,8,2026-06-28,1972,2458.000650,1637.0,1.501528
7,26,8,2026-07-05,1880,2343.327192,1527.0,1.534595
8,27,8,2026-07-12,1800,2243.611141,1457.0,1.539884
9,28,8,2026-07-19,1550,1931.998483,1206.0,1.601989


In [289]:
weekly_signal_summary_sp_2026 = (
    weeks_sp_2026[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_by_week_sp_2026[
            [
                "epi_week",
                "snapshot_date",
                "lag_days",
                "ratio_to_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

weekly_signal_summary_sp_2026[
    "has_signal"
] = (
    weekly_signal_summary_sp_2026[
        "snapshot_date"
    ].notna()
)

weekly_signal_summary_sp_2026[
    "previous_week_signal"
] = (
    weekly_signal_summary_sp_2026[
        "has_signal"
    ]
    .shift(1)
    .fillna(False)
)

weekly_signal_summary_sp_2026[
    "alert_candidate"
] = (
    weekly_signal_summary_sp_2026[
        "has_signal"
    ]
    & weekly_signal_summary_sp_2026[
        "previous_week_signal"
    ]
)

weekly_signal_summary_sp_2026[
    "previous_signal_date"
] = (
    weekly_signal_summary_sp_2026[
        "snapshot_date"
    ]
    .shift(1)
)

weekly_signal_summary_sp_2026[
    "alert_date"
] = pd.NaT

alert_mask = (
    weekly_signal_summary_sp_2026[
        "alert_candidate"
    ]
)

weekly_signal_summary_sp_2026.loc[
    alert_mask,
    "alert_date",
] = (
    weekly_signal_summary_sp_2026.loc[
        alert_mask,
        [
            "snapshot_date",
            "previous_signal_date",
        ],
    ]
    .max(axis=1)
)

display(
    weekly_signal_summary_sp_2026
)

,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
0,18,2026-04-26,2026-05-02,NaT,NaN,NaN,False,False,False,NaT,NaT
1,19,2026-05-03,2026-05-09,2026-05-17,8.0,1.153410,True,False,False,NaT,NaT
2,20,2026-05-10,2026-05-16,2026-05-24,8.0,1.288466,True,True,True,2026-05-17,2026-05-24
3,21,2026-05-17,2026-05-23,2026-05-31,8.0,1.361641,True,True,True,2026-05-24,2026-05-31
4,22,2026-05-24,2026-05-30,2026-06-07,8.0,1.449772,True,True,True,2026-05-31,2026-06-07
5,23,2026-05-31,2026-06-06,2026-06-14,8.0,1.675660,True,True,True,2026-06-07,2026-06-14
6,24,2026-06-07,2026-06-13,2026-06-21,8.0,1.535682,True,True,True,2026-06-14,2026-06-21
7,25,2026-06-14,2026-06-20,2026-06-28,8.0,1.501528,True,True,True,2026-06-21,2026-06-28
8,26,2026-06-21,2026-06-27,2026-07-05,8.0,1.534595,True,True,True,2026-06-28,2026-07-05
9,27,2026-06-28,2026-07-04,2026-07-12,8.0,1.539884,True,True,True,2026-07-05,2026-07-12


In [290]:
first_signal_sp_2026 = (
    weekly_signal_summary_sp_2026.loc[
        weekly_signal_summary_sp_2026[
            "has_signal"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(1)
)

first_alert_sp_2026 = (
    weekly_signal_summary_sp_2026.loc[
        weekly_signal_summary_sp_2026[
            "alert_candidate"
        ]
    ]
    .sort_values(
        "alert_date"
    )
    .head(1)
)

print(
    "=== PRIMEIRO SIGNAL SP/2026 ==="
)

display(first_signal_sp_2026)

print(
    "\n=== PRIMEIRO ALERT SP/2026 ==="
)

display(first_alert_sp_2026)

=== PRIMEIRO SIGNAL SP/2026 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
1,19,2026-05-03,2026-05-09,2026-05-17,8.0,1.15341,True,False,False,NaT,NaT



=== PRIMEIRO ALERT SP/2026 ===


,epi_week,week_start,week_end,snapshot_date,lag_days,ratio_to_median,has_signal,previous_week_signal,alert_candidate,previous_signal_date,alert_date
2,20,2026-05-10,2026-05-16,2026-05-24,8.0,1.288466,True,True,True,2026-05-17,2026-05-24


In [291]:
official_events_sp_2026 = {
    "InfoGripe indica crescimento em SP":
        pd.Timestamp("2026-05-14"),

    "Ampliação vacinação influenza SP":
        pd.Timestamp("2026-05-18"),

    "Informe nacional SE20":
        pd.Timestamp("2026-05-23"),
}

In [292]:
first_signal_sp_2026_date = pd.Timestamp(
    first_signal_sp_2026[
        "snapshot_date"
    ].iloc[0]
)

first_alert_sp_2026_date = pd.Timestamp(
    first_alert_sp_2026[
        "alert_date"
    ].iloc[0]
)

lead_time_rows_sp_2026 = []

for event_name, event_date in (
    official_events_sp_2026.items()
):
    for state, sentinela_date in [
        (
            "SIGNAL",
            first_signal_sp_2026_date,
        ),
        (
            "ALERT",
            first_alert_sp_2026_date,
        ),
    ]:
        lead_days = (
            event_date
            - sentinela_date
        ).days

        lead_time_rows_sp_2026.append(
            {
                "reference_event":
                    event_name,
                "official_date":
                    event_date,
                "sentinela_state":
                    state,
                "sentinela_date":
                    sentinela_date,
                "lead_days":
                    lead_days,
            }
        )

lead_time_sp_2026 = pd.DataFrame(
    lead_time_rows_sp_2026
)

lead_time_sp_2026[
    "interpretation"
] = (
    lead_time_sp_2026[
        "lead_days"
    ].apply(
        lambda days: (
            f"{days} dias antes"
            if days > 0
            else (
                "mesma data"
                if days == 0
                else (
                    f"{abs(days)} dias depois"
                )
            )
        )
    )
)

display(lead_time_sp_2026)

,reference_event,official_date,sentinela_state,sentinela_date,lead_days,interpretation
0,InfoGripe indica crescimento em SP,2026-05-14,SIGNAL,2026-05-17,-3,3 dias depois
1,InfoGripe indica crescimento em SP,2026-05-14,ALERT,2026-05-24,-10,10 dias depois
2,Ampliação vacinação influenza SP,2026-05-18,SIGNAL,2026-05-17,1,1 dias antes
3,Ampliação vacinação influenza SP,2026-05-18,ALERT,2026-05-24,-6,6 dias depois
4,Informe nacional SE20,2026-05-23,SIGNAL,2026-05-17,6,6 dias antes
5,Informe nacional SE20,2026-05-23,ALERT,2026-05-24,-1,1 dias depois


In [293]:
completion_uncertainty_sp_2026 = (
    completion_by_year_sp_2026
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=lambda x: x.quantile(0.25),
        completeness_q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

display(
    completion_uncertainty_sp_2026.loc[
        completion_uncertainty_sp_2026[
            "lag_days"
        ].between(0, 14)
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.070690,0.070496,0.074256
1,1,0.182839,0.181824,0.187654
2,2,0.303567,0.299177,0.319289
3,3,0.433195,0.422731,0.456132
4,4,0.541329,0.535417,0.576165
5,5,0.626974,0.623395,0.664607
6,6,0.697881,0.687535,0.729481
7,7,0.759905,0.747319,0.786024
8,8,0.802278,0.789909,0.824729
9,9,0.831609,0.819418,0.850529


In [294]:
def build_uncertainty_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    ratio_threshold: float = 1.15,
    mature_completeness: float = 0.80,
    min_early_completeness: float = 0.30,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df["DT_SIN_PRI"].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(max_lag + 1):
        completion_row = completion_model.loc[
            completion_model["lag_days"].eq(lag)
        ]

        if completion_row.empty:
            continue

        expected = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        q75 = float(
            completion_row[
                "completeness_q75"
            ].iloc[0]
        )

        snapshot_date = (
            week_end
            + pd.Timedelta(days=lag)
        )

        known_cases = int(
            (
                week_cases["DT_NOTIFIC"]
                <= snapshot_date
            ).sum()
        )

        if expected <= 0 or q75 <= 0:
            continue

        nowcast = (
            known_cases
            / expected
        )

        conservative_nowcast = (
            known_cases
            / q75
        )

        ratio = (
            nowcast
            / historical_median
        )

        conservative_ratio = (
            conservative_nowcast
            / historical_median
        )

        is_mature = (
            expected
            >= mature_completeness
        )

        has_minimum_information = (
            expected
            >= min_early_completeness
        )

        # Regra:
        # - se maduro, usa estimativa central;
        # - se ainda imaturo, exige que até
        #   o cenário conservador ultrapasse
        #   o threshold.
        if is_mature:
            signal_detected = (
                ratio >= ratio_threshold
            )

            signal_mode = (
                "MATURE"
                if signal_detected
                else "NONE"
            )

        elif has_minimum_information:
            signal_detected = (
                conservative_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "EARLY_CONSERVATIVE"
                if signal_detected
                else "NONE"
            )

        else:
            signal_detected = False
            signal_mode = "INSUFFICIENT_DATA"

        rows.append(
            {
                "epi_week": epi_week,
                "lag_days": lag,
                "snapshot_date": snapshot_date,
                "known_cases": known_cases,
                "expected_completeness": expected,
                "completeness_q75": q75,
                "is_mature": is_mature,
                "nowcast": nowcast,
                "conservative_nowcast":
                    conservative_nowcast,
                "historical_median":
                    historical_median,
                "ratio_to_median": ratio,
                "conservative_ratio":
                    conservative_ratio,
                "signal_detected":
                    signal_detected,
                "signal_mode":
                    signal_mode,
            }
        )

    return pd.DataFrame(rows)

In [295]:
week_19 = weeks_sp_2026.loc[
    weeks_sp_2026[
        "epi_week"
    ].eq(19)
].iloc[0]

reference_week_19 = (
    reference_sp_2026.loc[
        reference_sp_2026[
            "epi_week"
        ].eq(19),
        "historical_median",
    ]
    .iloc[0]
)

In [296]:
timeline_sp_2026_se19_v2 = (
    build_uncertainty_nowcast_timeline(
        temporal_df=sp_2026_temporal,
        week_start=week_19["week_start"],
        week_end=week_19["week_end"],
        epi_week=19,
        historical_median=float(
            reference_week_19
        ),
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ratio_threshold=1.15,
        mature_completeness=0.80,
        min_early_completeness=0.30,
    )
)

display(
    timeline_sp_2026_se19_v2[
        [
            "lag_days",
            "snapshot_date",
            "known_cases",
            "expected_completeness",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
        ]
    ]
)

,lag_days,snapshot_date,known_cases,expected_completeness,ratio_to_median,conservative_ratio,signal_mode,signal_detected
0,0,2026-05-09,723,0.070690,6.059139,5.768129,INSUFFICIENT_DATA,False
1,1,2026-05-10,807,0.182839,2.614760,2.547675,INSUFFICIENT_DATA,False
2,2,2026-05-11,1086,0.303567,2.119352,2.014994,EARLY_CONSERVATIVE,True
3,3,2026-05-12,1248,0.433195,1.706708,1.620882,EARLY_CONSERVATIVE,True
4,4,2026-05-13,1368,0.541329,1.497107,1.406588,EARLY_CONSERVATIVE,True
5,5,2026-05-14,1463,0.626974,1.382363,1.304088,EARLY_CONSERVATIVE,True
6,6,2026-05-15,1544,0.697881,1.310670,1.253893,EARLY_CONSERVATIVE,True
7,7,2026-05-16,1551,0.759905,1.209150,1.168971,EARLY_CONSERVATIVE,True
8,8,2026-05-17,1562,0.802278,1.153410,1.122011,MATURE,True
9,9,2026-05-18,1599,0.831609,1.139087,1.113748,NONE,False


In [297]:
first_early_signal = (
    timeline_sp_2026_se19_v2.loc[
        timeline_sp_2026_se19_v2[
            "signal_detected"
        ]
    ]
    .sort_values("snapshot_date")
    .head(1)
)

display(first_early_signal)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,completeness_q75,is_mature,nowcast,conservative_nowcast,historical_median,ratio_to_median,conservative_ratio,signal_detected,signal_mode
2,19,2,2026-05-11,1086,0.303567,0.319289,False,3577.466686,3401.310135,1688.0,2.119352,2.014994,True,EARLY_CONSERVATIVE


In [298]:
def get_completion_at_age(
    completion_model: pd.DataFrame,
    age_days: int,
    column: str,
) -> float:
    age_days = max(age_days, 0)

    max_lag = int(
        completion_model["lag_days"].max()
    )

    age_days = min(
        age_days,
        max_lag,
    )

    row = completion_model.loc[
        completion_model[
            "lag_days"
        ].eq(age_days)
    ]

    if row.empty:
        return float("nan")

    return float(
        row[column].iloc[0]
    )

In [299]:
def calculate_age_adjusted_nowcast(
    week_cases: pd.DataFrame,
    snapshot_date: pd.Timestamp,
    completion_model: pd.DataFrame,
    completion_column: str,
) -> float:
    known_cases = week_cases.loc[
        week_cases[
            "DT_NOTIFIC"
        ].le(snapshot_date)
    ].copy()

    if known_cases.empty:
        return 0.0

    known_cases["age_days"] = (
        snapshot_date
        - known_cases["DT_SIN_PRI"]
    ).dt.days

    known_cases[
        "completion_probability"
    ] = (
        known_cases["age_days"]
        .apply(
            lambda age: get_completion_at_age(
                completion_model=completion_model,
                age_days=int(age),
                column=completion_column,
            )
        )
    )

    known_cases = known_cases.loc[
        known_cases[
            "completion_probability"
        ].notna()
        & known_cases[
            "completion_probability"
        ].gt(0)
    ]

    return (
        1
        / known_cases[
            "completion_probability"
        ]
    ).sum()

In [300]:
def build_age_adjusted_nowcast_timeline(
    temporal_df: pd.DataFrame,
    week_start: pd.Timestamp,
    week_end: pd.Timestamp,
    epi_week: int,
    historical_median: float,
    completion_model: pd.DataFrame,
    ratio_threshold: float = 1.15,
    min_early_completeness: float = 0.30,
    mature_completeness: float = 0.80,
    max_lag: int = 28,
) -> pd.DataFrame:
    week_cases = temporal_df.loc[
        temporal_df[
            "DT_SIN_PRI"
        ].between(
            week_start,
            week_end,
        )
    ].copy()

    rows = []

    for lag in range(
        max_lag + 1
    ):
        snapshot_date = (
            week_end
            + pd.Timedelta(
                days=lag
            )
        )

        known = week_cases.loc[
            week_cases[
                "DT_NOTIFIC"
            ].le(snapshot_date)
        ]

        known_cases = len(known)

        central_nowcast = (
            calculate_age_adjusted_nowcast(
                week_cases=week_cases,
                snapshot_date=snapshot_date,
                completion_model=completion_model,
                completion_column=(
                    "expected_completeness"
                ),
            )
        )

        conservative_nowcast = (
            calculate_age_adjusted_nowcast(
                week_cases=week_cases,
                snapshot_date=snapshot_date,
                completion_model=completion_model,
                completion_column=(
                    "completeness_q75"
                ),
            )
        )

        central_ratio = (
            central_nowcast
            / historical_median
        )

        conservative_ratio = (
            conservative_nowcast
            / historical_median
        )

        completion_row = (
            completion_model.loc[
                completion_model[
                    "lag_days"
                ].eq(lag)
            ]
        )

        expected_completion = float(
            completion_row[
                "expected_completeness"
            ].iloc[0]
        )

        if (
            expected_completion
            >= mature_completeness
        ):
            signal_detected = (
                central_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "MATURE"
                if signal_detected
                else "NONE"
            )

        elif (
            expected_completion
            >= min_early_completeness
        ):
            signal_detected = (
                conservative_ratio
                >= ratio_threshold
            )

            signal_mode = (
                "EARLY_CONSERVATIVE"
                if signal_detected
                else "NONE"
            )

        else:
            signal_detected = False
            signal_mode = (
                "INSUFFICIENT_DATA"
            )

        rows.append(
            {
                "epi_week":
                    epi_week,
                "lag_days":
                    lag,
                "snapshot_date":
                    snapshot_date,
                "known_cases":
                    known_cases,
                "expected_completeness":
                    expected_completion,
                "age_adjusted_nowcast":
                    central_nowcast,
                "conservative_nowcast":
                    conservative_nowcast,
                "historical_median":
                    historical_median,
                "ratio_to_median":
                    central_ratio,
                "conservative_ratio":
                    conservative_ratio,
                "signal_detected":
                    signal_detected,
                "signal_mode":
                    signal_mode,
            }
        )

    return pd.DataFrame(rows)

In [301]:
timeline_sp_2026_se19_v3 = (
    build_age_adjusted_nowcast_timeline(
        temporal_df=sp_2026_temporal,
        week_start=week_19[
            "week_start"
        ],
        week_end=week_19[
            "week_end"
        ],
        epi_week=19,
        historical_median=float(
            reference_week_19
        ),
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ratio_threshold=1.15,
        mature_completeness=0.80,
        min_early_completeness=0.30,
    )
)

display(
    timeline_sp_2026_se19_v3[
        [
            "lag_days",
            "snapshot_date",
            "known_cases",
            "expected_completeness",
            "age_adjusted_nowcast",
            "conservative_nowcast",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
        ]
    ]
)

,lag_days,snapshot_date,known_cases,expected_completeness,age_adjusted_nowcast,conservative_nowcast,ratio_to_median,conservative_ratio,signal_mode,signal_detected
0,0,2026-05-09,723,0.070690,1699.547527,1618.366131,1.006841,0.958748,INSUFFICIENT_DATA,False
1,1,2026-05-10,807,0.182839,1548.903258,1479.767022,0.917597,0.876639,INSUFFICIENT_DATA,False
2,2,2026-05-11,1086,0.303567,1837.197309,1753.499269,1.088387,1.038803,NONE,False
3,3,2026-05-12,1248,0.433195,1887.431231,1806.949582,1.118146,1.070468,NONE,False
4,4,2026-05-13,1368,0.541329,1899.289623,1824.871057,1.125172,1.081085,NONE,False
5,5,2026-05-14,1463,0.626974,1902.521863,1839.349898,1.127086,1.089662,NONE,False
6,6,2026-05-15,1544,0.697881,1910.987319,1859.233623,1.132101,1.101442,NONE,False
7,7,2026-05-16,1551,0.759905,1848.766498,1806.761022,1.095241,1.070356,NONE,False
8,8,2026-05-17,1562,0.802278,1813.598323,1777.688054,1.074407,1.053133,NONE,False
9,9,2026-05-18,1599,0.831609,1820.175750,1788.337912,1.078303,1.059442,NONE,False


In [302]:
display(
    timeline_sp_2026_se19_v3.loc[
        timeline_sp_2026_se19_v3[
            "signal_detected"
        ]
    ].head(1)
)

,epi_week,lag_days,snapshot_date,known_cases,expected_completeness,age_adjusted_nowcast,conservative_nowcast,historical_median,ratio_to_median,conservative_ratio,signal_detected,signal_mode


In [303]:
weeks_sp_2026_v3 = pd.DataFrame(
    {
        "epi_week": range(15, 38),
    }
)

first_week_start_sp_2026 = pd.Timestamp(
    "2025-12-28"
)

weeks_sp_2026_v3["week_start"] = (
    first_week_start_sp_2026
    + pd.to_timedelta(
        (
            weeks_sp_2026_v3["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2026_v3["week_end"] = (
    weeks_sp_2026_v3["week_start"]
    + pd.Timedelta(days=6)
)

In [305]:
timeline_frames_v3 = []

for week in weeks_sp_2026_v3.itertuples(
    index=False
):
    reference_row = (
        reference_sp_2026.loc[
            reference_sp_2026[
                "epi_week"
            ].eq(week.epi_week)
        ]
    )

    if reference_row.empty:
        continue

    historical_median = float(
        reference_row[
            "historical_median"
        ].iloc[0]
    )

    timeline = (
        build_age_adjusted_nowcast_timeline(
            temporal_df=sp_2026_temporal,
            week_start=week.week_start,
            week_end=week.week_end,
            epi_week=week.epi_week,
            historical_median=historical_median,
            completion_model=(
                completion_uncertainty_sp_2026
            ),
            ratio_threshold=1.15,
            mature_completeness=0.80,
            min_early_completeness=0.30,
        )
    )

    timeline_frames_v3.append(
        timeline
    )

timeline_sp_2026_v3 = pd.concat(
    timeline_frames_v3,
    ignore_index=True,
)

In [306]:
first_signal_v3 = (
    timeline_sp_2026_v3.loc[
        timeline_sp_2026_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        [
            "epi_week",
            "snapshot_date",
        ]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .first()
)

display(
    first_signal_v3[
        [
            "epi_week",
            "lag_days",
            "snapshot_date",
            "expected_completeness",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
        ]
    ]
)

,epi_week,lag_days,snapshot_date,expected_completeness,ratio_to_median,conservative_ratio,signal_mode
0,20,2,2026-05-18,0.303567,1.224398,1.168706,EARLY_CONSERVATIVE
1,21,2,2026-05-25,0.303567,1.230247,1.174377,EARLY_CONSERVATIVE
2,22,2,2026-06-01,0.303567,1.436357,1.371251,EARLY_CONSERVATIVE
3,23,2,2026-06-08,0.303567,1.444234,1.378293,EARLY_CONSERVATIVE
4,24,2,2026-06-15,0.303567,1.463082,1.396960,EARLY_CONSERVATIVE
5,25,2,2026-06-22,0.303567,1.344299,1.283813,EARLY_CONSERVATIVE
6,26,2,2026-06-29,0.303567,1.415843,1.351444,EARLY_CONSERVATIVE
7,27,2,2026-07-06,0.303567,1.480455,1.412560,EARLY_CONSERVATIVE
8,28,2,2026-07-13,0.303567,1.428030,1.363218,EARLY_CONSERVATIVE
9,29,3,2026-07-21,0.433195,1.231045,1.179974,EARLY_CONSERVATIVE


In [307]:
evaluation_sp_2026_v3 = (
    weeks_sp_2026_v3[
        [
            "epi_week",
            "week_start",
            "week_end",
        ]
    ]
    .merge(
        first_signal_v3[
            [
                "epi_week",
                "lag_days",
                "snapshot_date",
                "ratio_to_median",
                "conservative_ratio",
                "signal_mode",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

evaluation_sp_2026_v3[
    "signal_detected"
] = (
    evaluation_sp_2026_v3[
        "snapshot_date"
    ].notna()
)

In [308]:
evaluation_sp_2026_v3[
    "ground_truth"
] = pd.Series(
    pd.NA,
    index=evaluation_sp_2026_v3.index,
    dtype="Int64",
)

# Negativo forte antes do episódio
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(15, 16),
    "ground_truth",
] = 0

# Positivo forte
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(20, 28),
    "ground_truth",
] = 1

# Negativo forte depois do episódio
evaluation_sp_2026_v3.loc[
    evaluation_sp_2026_v3[
        "epi_week"
    ].between(32, 37),
    "ground_truth",
] = 0

In [309]:
def classify_early_outcome(row):
    if pd.isna(row["ground_truth"]):
        return "UNLABELED"

    truth = int(row["ground_truth"])
    predicted = bool(
        row["signal_detected"]
    )

    if truth == 1 and predicted:
        return "TP"

    if truth == 1 and not predicted:
        return "FN"

    if truth == 0 and predicted:
        return "FP"

    return "TN"


evaluation_sp_2026_v3[
    "outcome"
] = (
    evaluation_sp_2026_v3.apply(
        classify_early_outcome,
        axis=1,
    )
)

In [310]:
display(
    evaluation_sp_2026_v3[
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,ratio_to_median,conservative_ratio,signal_mode,signal_detected,outcome
0,15,0,NaT,NaN,NaN,NaN,NaN,False,TN
1,16,0,NaT,NaN,NaN,NaN,NaN,False,TN
2,17,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
3,18,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
4,19,<NA>,NaT,NaN,NaN,NaN,NaN,False,UNLABELED
5,20,1,2026-05-18,2.0,1.224398,1.168706,EARLY_CONSERVATIVE,True,TP
6,21,1,2026-05-25,2.0,1.230247,1.174377,EARLY_CONSERVATIVE,True,TP
7,22,1,2026-06-01,2.0,1.436357,1.371251,EARLY_CONSERVATIVE,True,TP
8,23,1,2026-06-08,2.0,1.444234,1.378293,EARLY_CONSERVATIVE,True,TP
9,24,1,2026-06-15,2.0,1.463082,1.396960,EARLY_CONSERVATIVE,True,TP


In [311]:
display(
    evaluation_sp_2026_v3.loc[
        evaluation_sp_2026_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    9
TN    8
Name: count, dtype: int64

In [312]:
positive_detection_sp_2026 = (
    evaluation_sp_2026_v3.loc[
        evaluation_sp_2026_v3[
            "ground_truth"
        ].eq(1)
    ][
        [
            "epi_week",
            "week_end",
            "snapshot_date",
            "lag_days",
            "signal_mode",
            "conservative_ratio",
        ]
    ]
)

display(positive_detection_sp_2026)

,epi_week,week_end,snapshot_date,lag_days,signal_mode,conservative_ratio
5,20,2026-05-16,2026-05-18,2.0,EARLY_CONSERVATIVE,1.168706
6,21,2026-05-23,2026-05-25,2.0,EARLY_CONSERVATIVE,1.174377
7,22,2026-05-30,2026-06-01,2.0,EARLY_CONSERVATIVE,1.371251
8,23,2026-06-06,2026-06-08,2.0,EARLY_CONSERVATIVE,1.378293
9,24,2026-06-13,2026-06-15,2.0,EARLY_CONSERVATIVE,1.396960
10,25,2026-06-20,2026-06-22,2.0,EARLY_CONSERVATIVE,1.283813
11,26,2026-06-27,2026-06-29,2.0,EARLY_CONSERVATIVE,1.351444
12,27,2026-07-04,2026-07-06,2.0,EARLY_CONSERVATIVE,1.412560
13,28,2026-07-11,2026-07-13,2.0,EARLY_CONSERVATIVE,1.363218


## 26. Nowcast ajustado pela idade do registro — SP/2026

A correção do nowcast passou a considerar individualmente o tempo
decorrido entre o início dos sintomas de cada registro e a data do
snapshot.

Essa abordagem evita tratar todos os registros de uma semana epidemiológica
como se possuíssem o mesmo atraso de notificação.

### Resultado

Nas semanas rotuladas de SP/2026:

- TP: 9
- FP: 0
- FN: 0
- TN: 8

O modelo não sinalizou as semanas negativas fortes SE15–16 e SE32–37.

A semana SE19, previamente sinalizada pelo nowcast simplificado,
deixou corretamente de atingir o limiar após a correção por idade,
permanecendo compatível com uma fase de transição.

Todas as semanas positivas SE20–28 foram identificadas através do modo
`EARLY_CONSERVATIVE`.

O primeiro sinal de cada uma ocorreu em D+2, quando a completude histórica
esperada era de aproximadamente 30%.

Apesar da baixa completude, o limite conservador do nowcast permaneceu
acima do limiar de 1,15 em todas as semanas positivas.

Esse comportamento sugere que a utilização explícita da incerteza permite
antecipar a detecção sem aumentar falsos sinais no conjunto avaliado.

In [313]:
def evaluate_early_signal_case(
    temporal_df: pd.DataFrame,
    weeks_df: pd.DataFrame,
    reference_df: pd.DataFrame,
    completion_model: pd.DataFrame,
    ground_truth_by_week: dict[int, int],
    ratio_threshold: float = 1.15,
    min_early_completeness: float = 0.30,
    mature_completeness: float = 0.80,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    timeline_frames = []

    for week in weeks_df.itertuples(
        index=False
    ):
        reference_row = (
            reference_df.loc[
                reference_df[
                    "epi_week"
                ].eq(week.epi_week)
            ]
        )

        if reference_row.empty:
            continue

        historical_median = float(
            reference_row[
                "historical_median"
            ].iloc[0]
        )

        timeline = (
            build_age_adjusted_nowcast_timeline(
                temporal_df=temporal_df,
                week_start=week.week_start,
                week_end=week.week_end,
                epi_week=week.epi_week,
                historical_median=(
                    historical_median
                ),
                completion_model=(
                    completion_model
                ),
                ratio_threshold=(
                    ratio_threshold
                ),
                min_early_completeness=(
                    min_early_completeness
                ),
                mature_completeness=(
                    mature_completeness
                ),
            )
        )

        timeline_frames.append(
            timeline
        )

    timeline_all = pd.concat(
        timeline_frames,
        ignore_index=True,
    )

    first_signal = (
        timeline_all.loc[
            timeline_all[
                "signal_detected"
            ]
        ]
        .sort_values(
            [
                "epi_week",
                "snapshot_date",
            ]
        )
        .groupby(
            "epi_week",
            as_index=False,
        )
        .first()
    )

    evaluation = (
        weeks_df[
            [
                "epi_week",
                "week_start",
                "week_end",
            ]
        ]
        .merge(
            first_signal[
                [
                    "epi_week",
                    "lag_days",
                    "snapshot_date",
                    "ratio_to_median",
                    "conservative_ratio",
                    "signal_mode",
                ]
            ],
            on="epi_week",
            how="left",
        )
    )

    evaluation[
        "signal_detected"
    ] = (
        evaluation[
            "snapshot_date"
        ].notna()
    )

    evaluation[
        "ground_truth"
    ] = (
        evaluation[
            "epi_week"
        ]
        .map(
            ground_truth_by_week
        )
        .astype("Int64")
    )

    evaluation[
        "outcome"
    ] = (
        evaluation.apply(
            classify_early_outcome,
            axis=1,
        )
    )

    return (
        timeline_all,
        evaluation,
    )

In [314]:
ground_truth_sp_2026_map = {
    15: 0,
    16: 0,

    20: 1,
    21: 1,
    22: 1,
    23: 1,
    24: 1,
    25: 1,
    26: 1,
    27: 1,
    28: 1,

    32: 0,
    33: 0,
    34: 0,
    35: 0,
    36: 0,
    37: 0,
}

timeline_test, evaluation_test = (
    evaluate_early_signal_case(
        temporal_df=sp_2026_temporal,
        weeks_df=weeks_sp_2026_v3,
        reference_df=reference_sp_2026,
        completion_model=(
            completion_uncertainty_sp_2026
        ),
        ground_truth_by_week=(
            ground_truth_sp_2026_map
        ),
    )
)

print(
    evaluation_test.loc[
        evaluation_test[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    9
TN    8
Name: count, dtype: int64


In [315]:
completion_uncertainty_sp_2025 = (
    completion_by_year_sp_2025
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=(
            lambda x:
                x.quantile(0.25)
        ),
        completeness_q75=(
            lambda x:
                x.quantile(0.75)
        ),
    )
    .reset_index()
)

In [316]:
weeks_sp_2025_v3 = pd.DataFrame(
    {
        "epi_week": range(1, 41),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_sp_2025_v3[
    "week_start"
] = (
    first_week_start
    + pd.to_timedelta(
        (
            weeks_sp_2025_v3[
                "epi_week"
            ]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_sp_2025_v3[
    "week_end"
] = (
    weeks_sp_2025_v3[
        "week_start"
    ]
    + pd.Timedelta(days=6)
)

In [317]:
ground_truth_sp_2025_map = (
    ground_truth_sp_2025.loc[
        ground_truth_sp_2025[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [318]:
timeline_sp_2025_v3, evaluation_sp_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=sp_2025_temporal,
        weeks_df=weeks_sp_2025_v3,
        reference_df=reference_sp_2025,
        completion_model=(
            completion_uncertainty_sp_2025
        ),
        ground_truth_by_week=(
            ground_truth_sp_2025_map
        ),
    )
)

In [319]:
display(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
16,17,1,2025-04-28,2.0,1.246167,EARLY_CONSERVATIVE,True,TP
17,18,1,2025-05-05,2.0,1.204644,EARLY_CONSERVATIVE,True,TP
18,19,1,2025-05-12,2.0,1.552929,EARLY_CONSERVATIVE,True,TP
19,20,1,2025-05-19,2.0,1.721005,EARLY_CONSERVATIVE,True,TP
20,21,1,2025-05-26,2.0,1.928592,EARLY_CONSERVATIVE,True,TP
21,22,1,2025-06-02,2.0,1.871216,EARLY_CONSERVATIVE,True,TP
22,23,1,2025-06-09,2.0,1.879823,EARLY_CONSERVATIVE,True,TP
23,24,1,2025-06-16,2.0,1.586937,EARLY_CONSERVATIVE,True,TP
24,25,1,2025-06-23,2.0,1.501169,EARLY_CONSERVATIVE,True,TP
29,30,0,2025-07-28,2.0,1.206609,EARLY_CONSERVATIVE,True,FP


In [320]:
print(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TN    10
TP     9
FP     1
Name: count, dtype: int64


In [321]:
display(
    evaluation_sp_2025_v3.loc[
        evaluation_sp_2025_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(10)
)

,epi_week,week_start,week_end,lag_days,snapshot_date,ratio_to_median,conservative_ratio,signal_mode,signal_detected,ground_truth,outcome
16,17,2025-04-20,2025-04-26,2.0,2025-04-28,1.272197,1.246167,EARLY_CONSERVATIVE,True,1,TP
17,18,2025-04-27,2025-05-03,2.0,2025-05-05,1.230040,1.204644,EARLY_CONSERVATIVE,True,1,TP
18,19,2025-05-04,2025-05-10,2.0,2025-05-12,1.585130,1.552929,EARLY_CONSERVATIVE,True,1,TP
19,20,2025-05-11,2025-05-17,2.0,2025-05-19,1.757212,1.721005,EARLY_CONSERVATIVE,True,1,TP
20,21,2025-05-18,2025-05-24,2.0,2025-05-26,1.968923,1.928592,EARLY_CONSERVATIVE,True,1,TP
21,22,2025-05-25,2025-05-31,2.0,2025-06-02,1.910381,1.871216,EARLY_CONSERVATIVE,True,1,TP
22,23,2025-06-01,2025-06-07,2.0,2025-06-09,1.918792,1.879823,EARLY_CONSERVATIVE,True,1,TP
23,24,2025-06-08,2025-06-14,2.0,2025-06-16,1.620246,1.586937,EARLY_CONSERVATIVE,True,1,TP
24,25,2025-06-15,2025-06-21,2.0,2025-06-23,1.532431,1.501169,EARLY_CONSERVATIVE,True,1,TP
25,26,2025-06-22,2025-06-28,2.0,2025-06-30,1.389987,1.361642,EARLY_CONSERVATIVE,True,<NA>,UNLABELED


In [322]:
completion_uncertainty_mg_2025 = (
    completion_by_year_mg_2025
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=lambda x: x.quantile(0.25),
        completeness_q75=lambda x: x.quantile(0.75),
    )
    .reset_index()
)

In [323]:
weeks_mg_2025_v3 = pd.DataFrame(
    {
        "epi_week": range(1, 41),
    }
)

first_week_start = pd.Timestamp(
    "2024-12-29"
)

weeks_mg_2025_v3[
    "week_start"
] = (
    first_week_start
    + pd.to_timedelta(
        (
            weeks_mg_2025_v3[
                "epi_week"
            ]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_mg_2025_v3[
    "week_end"
] = (
    weeks_mg_2025_v3[
        "week_start"
    ]
    + pd.Timedelta(days=6)
)

In [324]:
ground_truth_mg_2025_map = (
    ground_truth.loc[
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
        & ground_truth[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [325]:
timeline_mg_2025_v3, evaluation_mg_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=mg_2025_temporal,
        weeks_df=weeks_mg_2025_v3,
        reference_df=reference_mg_2025,
        completion_model=(
            completion_uncertainty_mg_2025
        ),
        ground_truth_by_week=(
            ground_truth_mg_2025_map
        ),
        ratio_threshold=1.15,
        min_early_completeness=0.30,
        mature_completeness=0.80,
    )
)

In [326]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
17,18,1,2025-05-05,2.0,1.862618,EARLY_CONSERVATIVE,True,TP
18,19,1,2025-05-12,2.0,2.575234,EARLY_CONSERVATIVE,True,TP
19,20,1,2025-05-19,2.0,2.982193,EARLY_CONSERVATIVE,True,TP
20,21,1,2025-05-26,2.0,3.284841,EARLY_CONSERVATIVE,True,TP
21,22,1,2025-06-02,2.0,3.497126,EARLY_CONSERVATIVE,True,TP
22,23,1,2025-06-09,2.0,3.685245,EARLY_CONSERVATIVE,True,TP
23,24,1,2025-06-16,2.0,3.250442,EARLY_CONSERVATIVE,True,TP
24,25,1,2025-06-23,2.0,2.620404,EARLY_CONSERVATIVE,True,TP


In [327]:
print(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

outcome
TP    8
Name: count, dtype: int64


In [328]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "signal_detected"
        ]
    ]
    .sort_values(
        "snapshot_date"
    )
    .head(10)
)

,epi_week,week_start,week_end,lag_days,snapshot_date,ratio_to_median,conservative_ratio,signal_mode,signal_detected,ground_truth,outcome
16,17,2025-04-20,2025-04-26,2.0,2025-04-28,1.634392,1.599876,EARLY_CONSERVATIVE,True,<NA>,UNLABELED
17,18,2025-04-27,2025-05-03,2.0,2025-05-05,1.907510,1.862618,EARLY_CONSERVATIVE,True,1,TP
18,19,2025-05-04,2025-05-10,2.0,2025-05-12,2.632161,2.575234,EARLY_CONSERVATIVE,True,1,TP
19,20,2025-05-11,2025-05-17,2.0,2025-05-19,3.050462,2.982193,EARLY_CONSERVATIVE,True,1,TP
20,21,2025-05-18,2025-05-24,2.0,2025-05-26,3.357593,3.284841,EARLY_CONSERVATIVE,True,1,TP
21,22,2025-05-25,2025-05-31,2.0,2025-06-02,3.570340,3.497126,EARLY_CONSERVATIVE,True,1,TP
22,23,2025-06-01,2025-06-07,2.0,2025-06-09,3.759457,3.685245,EARLY_CONSERVATIVE,True,1,TP
23,24,2025-06-08,2025-06-14,2.0,2025-06-16,3.320740,3.250442,EARLY_CONSERVATIVE,True,1,TP
24,25,2025-06-15,2025-06-21,2.0,2025-06-23,2.677731,2.620404,EARLY_CONSERVATIVE,True,1,TP


In [329]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "signal_detected"
        ]
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "ratio_to_median",
            "conservative_ratio",
            "signal_mode",
            "outcome",
        ]
    ]
)

,epi_week,ground_truth,snapshot_date,lag_days,ratio_to_median,conservative_ratio,signal_mode,outcome
16,17,<NA>,2025-04-28,2.0,1.634392,1.599876,EARLY_CONSERVATIVE,UNLABELED
17,18,1,2025-05-05,2.0,1.907510,1.862618,EARLY_CONSERVATIVE,TP
18,19,1,2025-05-12,2.0,2.632161,2.575234,EARLY_CONSERVATIVE,TP
19,20,1,2025-05-19,2.0,3.050462,2.982193,EARLY_CONSERVATIVE,TP
20,21,1,2025-05-26,2.0,3.357593,3.284841,EARLY_CONSERVATIVE,TP
21,22,1,2025-06-02,2.0,3.570340,3.497126,EARLY_CONSERVATIVE,TP
22,23,1,2025-06-09,2.0,3.759457,3.685245,EARLY_CONSERVATIVE,TP
23,24,1,2025-06-16,2.0,3.320740,3.250442,EARLY_CONSERVATIVE,TP
24,25,1,2025-06-23,2.0,2.677731,2.620404,EARLY_CONSERVATIVE,TP


In [330]:
print(
    evaluation_mg_2025_v3[
        "outcome"
    ].value_counts()
)

outcome
UNLABELED    32
TP            8
Name: count, dtype: int64


In [331]:
negative_weeks_before = list(
    range(1, 11)
)

positive_weeks = list(
    range(18, 26)
)

negative_weeks_after = list(
    range(33, 41)
)

In [332]:
ground_truth_mg_2025_v2 = (
    ground_truth.loc[
        ground_truth["region"].eq("MG")
        & ground_truth["year"].eq(2025)
    ]
    .copy()
)

ground_truth_mg_2025_v2[
    "ground_truth"
] = pd.Series(
    pd.NA,
    index=ground_truth_mg_2025_v2.index,
    dtype="Int64",
)

ground_truth_mg_2025_v2[
    "evidence_level"
] = "UNLABELED"

In [333]:
mask_negative_before = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        negative_weeks_before
    )
)

ground_truth_mg_2025_v2.loc[
    mask_negative_before,
    "ground_truth",
] = 0

ground_truth_mg_2025_v2.loc[
    mask_negative_before,
    "evidence_level",
] = "MODERATE"

In [334]:
mask_positive = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        positive_weeks
    )
)

ground_truth_mg_2025_v2.loc[
    mask_positive,
    "ground_truth",
] = 1

ground_truth_mg_2025_v2.loc[
    mask_positive,
    "evidence_level",
] = "STRONG"

In [335]:
mask_negative_after = (
    ground_truth_mg_2025_v2[
        "epi_week"
    ].isin(
        negative_weeks_after
    )
)

ground_truth_mg_2025_v2.loc[
    mask_negative_after,
    "ground_truth",
] = 0

ground_truth_mg_2025_v2.loc[
    mask_negative_after,
    "evidence_level",
] = "STRONG"

In [336]:
display(
    ground_truth_mg_2025_v2[
        [
            "epi_week",
            "ground_truth",
            "evidence_level",
        ]
    ]
)

,epi_week,ground_truth,evidence_level
0,1,0,MODERATE
1,2,0,MODERATE
2,3,0,MODERATE
3,4,0,MODERATE
4,5,0,MODERATE
5,6,0,MODERATE
6,7,0,MODERATE
7,8,0,MODERATE
8,9,0,MODERATE
9,10,0,MODERATE


In [337]:
ground_truth_mg_2025_map = (
    ground_truth_mg_2025_v2.loc[
        ground_truth_mg_2025_v2[
            "ground_truth"
        ].notna()
    ]
    .set_index("epi_week")[
        "ground_truth"
    ]
    .astype(int)
    .to_dict()
)

In [338]:
timeline_mg_2025_v3, evaluation_mg_2025_v3 = (
    evaluate_early_signal_case(
        temporal_df=mg_2025_temporal,
        weeks_df=weeks_mg_2025_v3,
        reference_df=reference_mg_2025,
        completion_model=(
            completion_uncertainty_mg_2025
        ),
        ground_truth_by_week=(
            ground_truth_mg_2025_map
        ),
        ratio_threshold=1.15,
        min_early_completeness=0.30,
        mature_completeness=0.80,
    )
)

In [339]:
display(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

print(
    evaluation_mg_2025_v3.loc[
        evaluation_mg_2025_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
0,1,0,NaT,NaN,NaN,NaN,False,TN
1,2,0,NaT,NaN,NaN,NaN,False,TN
2,3,0,NaT,NaN,NaN,NaN,False,TN
3,4,0,NaT,NaN,NaN,NaN,False,TN
4,5,0,NaT,NaN,NaN,NaN,False,TN
5,6,0,NaT,NaN,NaN,NaN,False,TN
6,7,0,NaT,NaN,NaN,NaN,False,TN
7,8,0,NaT,NaN,NaN,NaN,False,TN
8,9,0,NaT,NaN,NaN,NaN,False,TN
9,10,0,NaT,NaN,NaN,NaN,False,TN


outcome
TN    18
TP     8
Name: count, dtype: int64


In [340]:
SENTINELA_V3 = {
    "ratio_threshold": 1.15,
    "min_early_completeness": 0.30,
    "mature_completeness": 0.80,
    "uncertainty_bound": "q75",
    "nowcast": "age_adjusted",
}

In [343]:
def build_weekly_history_for_region(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        source_year = int(
            f"20{name.split('INFLUD')[1][:2]}"
        )

        if source_year not in years:
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df = df.loc[
            df["DT_SIN_PRI"].notna()
        ].copy()

        #
        # Importante:
        # usamos calendário epidemiológico
        # no padrão domingo-sábado.
        #
        df["epi_week"] = (
            df["DT_SIN_PRI"]
            .dt.strftime("%U")
            .astype(int)
            + 1
        )

        #
        # Mantém apenas as semanas
        # epidemiológicas válidas.
        #
        df = df.loc[
            df["epi_week"].between(
                1,
                53,
            )
        ].copy()

        weekly = (
            df.groupby(
                "epi_week",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size": "record_count",
                }
            )
        )

        weekly["source_year"] = (
            source_year
        )

        weekly["region"] = region

        frames.append(weekly)

    if not frames:
        raise ValueError(
            f"Nenhum dado encontrado "
            f"para {region} nos anos {years}."
        )

    result = pd.concat(
        frames,
        ignore_index=True,
    )

    result = (
        result[
            [
                "source_year",
                "epi_week",
                "record_count",
                "region",
            ]
        ]
        .sort_values(
            [
                "source_year",
                "epi_week",
            ]
        )
        .reset_index(drop=True)
    )

    return result

In [344]:
def get_epi_week(
    date: pd.Timestamp,
    epi_year: int,
) -> int | None:
    first_sunday = pd.Timestamp(
        f"{epi_year - 1}-12-28"
    )

    delta_days = (
        date.normalize()
        - first_sunday
    ).days

    epi_week = (
        delta_days // 7
        + 1
    )

    if 1 <= epi_week <= 53:
        return epi_week

    return None

In [345]:
def build_epi_calendar_for_year(
    year: int,
) -> pd.DataFrame:
    jan4 = pd.Timestamp(
        year=year,
        month=1,
        day=4,
    )

    #
    # Domingo imediatamente anterior
    # ou igual a 4 de janeiro.
    #
    days_since_sunday = (
        jan4.dayofweek + 1
    ) % 7

    first_week_start = (
        jan4
        - pd.Timedelta(
            days=days_since_sunday
        )
    )

    weeks = []

    for epi_week in range(1, 54):
        week_start = (
            first_week_start
            + pd.Timedelta(
                days=(epi_week - 1) * 7
            )
        )

        week_end = (
            week_start
            + pd.Timedelta(days=6)
        )

        weeks.append(
            {
                "epi_year": year,
                "epi_week": epi_week,
                "week_start": week_start,
                "week_end": week_end,
            }
        )

    return pd.DataFrame(weeks)

In [346]:
def assign_epi_week_from_calendar(
    dates: pd.Series,
    year: int,
) -> pd.Series:
    calendar = (
        build_epi_calendar_for_year(
            year
        )
    )

    result = pd.Series(
        pd.NA,
        index=dates.index,
        dtype="Int64",
    )

    for row in calendar.itertuples(
        index=False
    ):
        mask = dates.between(
            row.week_start,
            row.week_end,
        )

        result.loc[mask] = (
            row.epi_week
        )

    return result

In [347]:
def build_weekly_history_for_region(
    historical_files,
    region: str,
    years: list[int],
) -> pd.DataFrame:
    frames = []

    for path in historical_files:
        name = path.name.upper()

        source_year = int(
            f"20{name.split('INFLUD')[1][:2]}"
        )

        if source_year not in years:
            continue

        df = pd.read_csv(
            path,
            sep=";",
            encoding="latin-1",
            usecols=[
                "SG_UF",
                "DT_SIN_PRI",
            ],
            dtype="string",
            low_memory=False,
        )

        df["SG_UF"] = (
            df["SG_UF"]
            .str.strip()
            .str.upper()
        )

        df = df.loc[
            df["SG_UF"].eq(region)
        ].copy()

        df["DT_SIN_PRI"] = (
            pd.to_datetime(
                df["DT_SIN_PRI"],
                format="ISO8601",
                errors="coerce",
                utc=True,
            )
            .dt.tz_localize(None)
        )

        df = df.loc[
            df["DT_SIN_PRI"].notna()
        ].copy()

        df["epi_week"] = (
            assign_epi_week_from_calendar(
                dates=df["DT_SIN_PRI"],
                year=source_year,
            )
        )

        df = df.loc[
            df["epi_week"].notna()
        ].copy()

        weekly = (
            df.groupby(
                "epi_week",
                as_index=False,
            )
            .size()
            .rename(
                columns={
                    "size":
                        "record_count",
                }
            )
        )

        weekly[
            "source_year"
        ] = source_year

        weekly[
            "region"
        ] = region

        frames.append(
            weekly
        )

    if not frames:
        raise ValueError(
            f"Nenhum dado encontrado "
            f"para região {region}."
        )

    return (
        pd.concat(
            frames,
            ignore_index=True,
        )
        [
            [
                "source_year",
                "epi_week",
                "record_count",
                "region",
            ]
        ]
        .sort_values(
            [
                "source_year",
                "epi_week",
            ]
        )
        .reset_index(drop=True)
    )

In [348]:
historical_pr = (
    build_weekly_history_for_region(
        historical_files=historical_files,
        region="PR",
        years=[
            2019,
            2022,
            2023,
            2024,
        ],
    )
)

In [349]:
display(
    historical_pr.head(20)
)

,source_year,epi_week,record_count,region
0,2019,1,33,PR
1,2019,2,48,PR
2,2019,3,37,PR
3,2019,4,42,PR
4,2019,5,35,PR
5,2019,6,25,PR
6,2019,7,42,PR
7,2019,8,44,PR
8,2019,9,64,PR
9,2019,10,61,PR


In [350]:
print(
    historical_pr[
        "source_year"
    ].value_counts().sort_index()
)

source_year
2019    52
2022    52
2023    52
2024    52
Name: count, dtype: int64


In [351]:
summary_pr = (
    historical_pr
    .groupby("source_year")
    .agg(
        weeks=(
            "epi_week",
            "nunique",
        ),
        total_cases=(
            "record_count",
            "sum",
        ),
        first_week=(
            "epi_week",
            "min",
        ),
        last_week=(
            "epi_week",
            "max",
        ),
    )
)

display(summary_pr)

,weeks,total_cases,first_week,last_week
source_year,,,,
2019,52,6805,1,52
2022,52,55377,1,52
2023,52,28153,1,52
2024,52,26531,1,52


In [352]:
assert historical_pr[
    "source_year"
].isin(
    [
        2019,
        2022,
        2023,
        2024,
    ]
).all()

assert historical_pr[
    "epi_week"
].between(
    1,
    53,
).all()

assert (
    historical_pr[
        [
            "source_year",
            "epi_week",
        ]
    ]
    .duplicated()
    .sum()
    == 0
)

print(
    "✓ Histórico PR construído."
)

✓ Histórico PR construído.


In [355]:
historical_pr = historical_pr.rename(
    columns={
        "source_year": "epi_year",
    }
)

In [356]:
baseline_years_pr_2024 = [
    2019,
    2022,
    2023,
]

reference_pr_2024 = (
    build_reference_selected_years(
        historical_weekly=historical_pr,
        years=baseline_years_pr_2024,
        quantile=0.75,
    )
)

In [357]:
completion_years_pr_2024 = [
    2019,
    2022,
    2023,
]

In [358]:
delay_frames_pr_2024 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_pr_2024:
        continue

    delays = load_notification_delays(
        path=path,
        region="PR",
        source_year=source_year,
    )

    delay_frames_pr_2024.append(delays)

historical_delays_pr_2024 = pd.concat(
    delay_frames_pr_2024,
    ignore_index=True,
)

historical_delays_pr_2024 = (
    historical_delays_pr_2024.loc[
        historical_delays_pr_2024[
            "delay_days"
        ].notna()
        & historical_delays_pr_2024[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [359]:
delay_frames_pr_2024 = []

for path in historical_files:
    name = path.name.upper()

    source_year = int(
        f"20{name.split('INFLUD')[1][:2]}"
    )

    if source_year not in completion_years_pr_2024:
        continue

    delays = load_notification_delays(
        path=path,
        region="PR",
        source_year=source_year,
    )

    delay_frames_pr_2024.append(delays)

historical_delays_pr_2024 = pd.concat(
    delay_frames_pr_2024,
    ignore_index=True,
)

historical_delays_pr_2024 = (
    historical_delays_pr_2024.loc[
        historical_delays_pr_2024[
            "delay_days"
        ].notna()
        & historical_delays_pr_2024[
            "delay_days"
        ].ge(0)
    ]
    .copy()
)

In [361]:
completion_rows_pr_2024 = []

for year, group in (
    historical_delays_pr_2024
    .groupby("source_year")
):
    for lag in range(0, 29):
        completion_rows_pr_2024.append(
            {
                "source_year": int(year),
                "lag_days": lag,
                "completeness": (
                    group["delay_days"]
                    .le(lag)
                    .mean()
                ),
            }
        )

completion_by_year_pr_2024 = pd.DataFrame(
    completion_rows_pr_2024
)

display(
    completion_by_year_pr_2024.head()
)

,source_year,lag_days,completeness
0,2019,0,0.053343
1,2019,1,0.184423
2,2019,2,0.333578
3,2019,3,0.481264
4,2019,4,0.611609


In [362]:
completion_uncertainty_pr_2024 = (
    completion_by_year_pr_2024
    .groupby("lag_days")[
        "completeness"
    ]
    .agg(
        expected_completeness="median",
        completeness_q25=(
            lambda x: x.quantile(0.25)
        ),
        completeness_q75=(
            lambda x: x.quantile(0.75)
        ),
    )
    .reset_index()
)

In [363]:
display(
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "lag_days"
        ].isin(
            [0, 2, 7, 14, 21, 28]
        )
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.068376,0.060860,0.078837
2,2,0.333578,0.332526,0.339968
7,7,0.821937,0.796950,0.841534
14,14,0.945938,0.926036,0.957466
21,21,0.975598,0.957109,0.981700
28,28,0.983803,0.966828,0.987787


In [364]:
path_2024 = next(
    path
    for path in historical_files
    if "INFLUD24" in path.name.upper()
)

df_pr_2024_temporal = pd.read_csv(
    path_2024,
    sep=";",
    encoding="latin-1",
    usecols=[
        "SG_UF",
        "DT_SIN_PRI",
        "DT_NOTIFIC",
    ],
    dtype="string",
    low_memory=False,
)

df_pr_2024_temporal["SG_UF"] = (
    df_pr_2024_temporal["SG_UF"]
    .str.strip()
    .str.upper()
)

pr_2024_temporal = (
    df_pr_2024_temporal.loc[
        df_pr_2024_temporal[
            "SG_UF"
        ].eq("PR")
    ]
    .copy()
)

for column in [
    "DT_SIN_PRI",
    "DT_NOTIFIC",
]:
    pr_2024_temporal[column] = (
        pd.to_datetime(
            pr_2024_temporal[column],
            format="ISO8601",
            errors="coerce",
            utc=True,
        )
        .dt.tz_localize(None)
    )

In [365]:
weeks_pr_2024 = pd.DataFrame(
    {
        "epi_week": range(1, 27),
    }
)

first_week_start_pr_2024 = pd.Timestamp(
    "2023-12-31"
)

weeks_pr_2024["week_start"] = (
    first_week_start_pr_2024
    + pd.to_timedelta(
        (
            weeks_pr_2024["epi_week"]
            - 1
        )
        * 7,
        unit="D",
    )
)

weeks_pr_2024["week_end"] = (
    weeks_pr_2024["week_start"]
    + pd.Timedelta(days=6)
)

In [366]:
ground_truth_pr_2024_map = {
    # Negativos externos fortes
    1: 0,
    2: 0,
    3: 0,
    4: 0,

    # Positivos externos fortes
    13: 1,
    14: 1,
    15: 1,
    16: 1,
    17: 1,
    18: 1,
    19: 1,
    20: 1,
    21: 1,
}

In [368]:
display(
    evaluation_pr_2024_v3.loc[
        evaluation_pr_2024_v3[
            "ground_truth"
        ].notna()
    ][
        [
            "epi_week",
            "ground_truth",
            "snapshot_date",
            "lag_days",
            "conservative_ratio",
            "signal_mode",
            "signal_detected",
            "outcome",
        ]
    ]
)

print(
    evaluation_pr_2024_v3.loc[
        evaluation_pr_2024_v3[
            "ground_truth"
        ].notna(),
        "outcome",
    ].value_counts()
)

,epi_week,ground_truth,snapshot_date,lag_days,conservative_ratio,signal_mode,signal_detected,outcome
0,1,0,NaT,NaN,NaN,NaN,False,TN
1,2,0,NaT,NaN,NaN,NaN,False,TN
2,3,0,NaT,NaN,NaN,NaN,False,TN
3,4,0,NaT,NaN,NaN,NaN,False,TN
12,13,1,NaT,NaN,NaN,NaN,False,FN
13,14,1,NaT,NaN,NaN,NaN,False,FN
14,15,1,NaT,NaN,NaN,NaN,False,FN
15,16,1,NaT,NaN,NaN,NaN,False,FN
16,17,1,NaT,NaN,NaN,NaN,False,FN
17,18,1,NaT,NaN,NaN,NaN,False,FN


outcome
FN    9
TN    4
Name: count, dtype: int64


In [369]:
pr_2024_weekly_observed = (
    pr_2024_temporal
    .assign(
        epi_week=assign_epi_week_from_calendar(
            dates=pr_2024_temporal["DT_SIN_PRI"],
            year=2024,
        )
    )
    .dropna(
        subset=["epi_week"]
    )
    .groupby(
        "epi_week",
        as_index=False,
    )
    .size()
    .rename(
        columns={
            "size": "record_count",
        }
    )
)

pr_2024_weekly_observed[
    "epi_week"
] = (
    pr_2024_weekly_observed[
        "epi_week"
    ].astype(int)
)

pr_2024_diagnostic = (
    pr_2024_weekly_observed
    .merge(
        reference_pr_2024[
            [
                "epi_week",
                "historical_median",
            ]
        ],
        on="epi_week",
        how="left",
    )
)

pr_2024_diagnostic[
    "final_ratio"
] = (
    pr_2024_diagnostic[
        "record_count"
    ]
    / pr_2024_diagnostic[
        "historical_median"
    ]
)

display(
    pr_2024_diagnostic.loc[
        pr_2024_diagnostic[
            "epi_week"
        ].between(1, 26)
    ]
)

,epi_week,record_count,historical_median,final_ratio
0,1,292,632.0,0.462025
1,2,280,469.0,0.597015
2,3,245,388.0,0.631443
3,4,224,346.0,0.647399
4,5,247,350.0,0.705714
5,6,281,415.0,0.677108
6,7,289,634.0,0.455836
7,8,438,625.0,0.700800
8,9,475,670.0,0.708955
9,10,492,635.0,0.774803


In [370]:
display(
    pr_2024_diagnostic.loc[
        pr_2024_diagnostic[
            "epi_week"
        ].between(13, 21),
        [
            "epi_week",
            "record_count",
            "historical_median",
            "final_ratio",
        ],
    ]
)

,epi_week,record_count,historical_median,final_ratio
12,13,419,677.0,0.618907
13,14,498,680.0,0.732353
14,15,535,788.0,0.678934
15,16,605,746.0,0.810992
16,17,650,682.0,0.953079
17,18,635,751.0,0.845539
18,19,608,639.0,0.951487
19,20,582,783.0,0.743295
20,21,681,688.0,0.989826


In [371]:
pr_2024_max_ratio = (
    timeline_pr_2024_v3
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        max_ratio=(
            "ratio_to_median",
            "max",
        ),
        max_conservative_ratio=(
            "conservative_ratio",
            "max",
        ),
    )
)

display(
    pr_2024_max_ratio.loc[
        pr_2024_max_ratio[
            "epi_week"
        ].between(13, 21)
    ]
)

,epi_week,max_ratio,max_conservative_ratio
12,13,0.624270,0.620269
13,14,0.735752,0.725245
14,15,0.686062,0.678001
15,16,0.816547,0.812957
16,17,0.995113,0.976799
17,18,0.859134,0.843923
18,19,0.964500,0.959128
19,20,0.754423,0.747379
20,21,1.007979,0.990934


In [372]:
display(
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "lag_days"
        ].between(0, 14)
    ]
)

,lag_days,expected_completeness,completeness_q25,completeness_q75
0,0,0.068376,0.060860,0.078837
1,1,0.197919,0.191171,0.200644
2,2,0.333578,0.332526,0.339968
3,3,0.481264,0.471621,0.486254
4,4,0.606117,0.588204,0.608863
5,5,0.696871,0.674266,0.707142
6,6,0.761446,0.737830,0.775359
7,7,0.821937,0.796950,0.841534
8,8,0.861400,0.836717,0.881104
9,9,0.885376,0.861671,0.902571


In [374]:
first_30 = (
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "expected_completeness"
        ].ge(0.30)
    ]
)

first_80 = (
    completion_uncertainty_pr_2024.loc[
        completion_uncertainty_pr_2024[
            "expected_completeness"
        ].ge(0.80)
    ]
)

print(
    "Primeiro >= 30%:",
    (
        int(first_30["lag_days"].min())
        if not first_30.empty
        else None
    ),
)

print(
    "Primeiro >= 80%:",
    (
        int(first_80["lag_days"].min())
        if not first_80.empty
        else None
    ),
)

Primeiro >= 30%: 2
Primeiro >= 80%: 7


In [375]:
import numpy as np

pr_trend = (
    pr_2024_diagnostic[
        [
            "epi_week",
            "record_count",
            "historical_median",
            "final_ratio",
        ]
    ]
    .copy()
)

pr_trend["log_count"] = np.log1p(
    pr_trend["record_count"]
)

In [376]:
def rolling_log_slope(
    series: pd.Series,
    window: int = 4,
) -> pd.Series:
    values = series.to_numpy(
        dtype=float
    )

    result = np.full(
        len(values),
        np.nan,
    )

    x = np.arange(
        window,
        dtype=float,
    )

    for i in range(
        window - 1,
        len(values),
    ):
        y = values[
            i - window + 1:
            i + 1
        ]

        slope = np.polyfit(
            x,
            y,
            1,
        )[0]

        result[i] = slope

    return pd.Series(
        result,
        index=series.index,
    )


pr_trend["slope_4w"] = (
    rolling_log_slope(
        pr_trend["log_count"],
        window=4,
    )
)

In [377]:
pr_trend[
    "weekly_growth_rate"
] = (
    np.exp(
        pr_trend["slope_4w"]
    )
    - 1
)

In [378]:
display(
    pr_trend.loc[
        pr_trend[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "slope_4w",
            "weekly_growth_rate",
        ],
    ]
)

,epi_week,record_count,final_ratio,slope_4w,weekly_growth_rate
0,1,292,0.462025,NaN,NaN
1,2,280,0.597015,NaN,NaN
2,3,245,0.631443,NaN,NaN
3,4,224,0.647399,-0.092524,-0.088373
4,5,247,0.705714,-0.046401,-0.045341
5,6,281,0.677108,0.050705,0.052013
6,7,289,0.455836,0.088982,0.093061
7,8,438,0.700800,0.174119,0.190197
8,9,475,0.708955,0.198515,0.219590
9,10,492,0.774803,0.167280,0.182086


In [379]:
pr_trend[
    "week_over_week_growth"
] = (
    pr_trend[
        "record_count"
    ]
    .pct_change()
)

pr_trend[
    "positive_growth"
] = (
    pr_trend[
        "week_over_week_growth"
    ] > 0
)

pr_trend[
    "positive_growth_3of4"
] = (
    pr_trend[
        "positive_growth"
    ]
    .rolling(
        4,
        min_periods=4,
    )
    .sum()
    >= 3
)

In [380]:
display(
    pr_trend.loc[
        pr_trend[
            "epi_week"
        ].between(8, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4
7,8,438,0.700800,0.190197,True
8,9,475,0.708955,0.219590,True
9,10,492,0.774803,0.182086,True
10,11,490,0.768025,0.037789,True
11,12,474,0.679083,-0.001037,False
12,13,419,0.618907,-0.050091,False
13,14,498,0.732353,-0.007430,False
14,15,535,0.678934,0.054937,False
15,16,605,0.810992,0.124277,True
16,17,650,0.953079,0.096415,True


In [381]:
def add_trend_features(
    weekly_df: pd.DataFrame,
    window: int = 4,
) -> pd.DataFrame:
    df = weekly_df.sort_values(
        "epi_week"
    ).copy()

    df["log_count"] = np.log1p(
        df["record_count"]
    )

    df["slope_4w"] = (
        rolling_log_slope(
            df["log_count"],
            window=window,
        )
    )

    df["weekly_growth_rate"] = (
        np.exp(df["slope_4w"]) - 1
    )

    df["week_over_week_growth"] = (
        df["record_count"]
        .pct_change()
    )

    df["positive_growth"] = (
        df["week_over_week_growth"] > 0
    )

    df["positive_growth_3of4"] = (
        df["positive_growth"]
        .rolling(
            window,
            min_periods=window,
        )
        .sum()
        >= 3
    )

    return df

In [382]:
historical_pr_trend_frames = []

for year in [
    2019,
    2022,
    2023,
]:
    yearly = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].eq(year)
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    yearly = add_trend_features(
        yearly,
        window=4,
    )

    historical_pr_trend_frames.append(
        yearly
    )

historical_pr_trends = pd.concat(
    historical_pr_trend_frames,
    ignore_index=True,
)

In [383]:
trend_reference_pr_2024 = (
    historical_pr_trends
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        historical_slope_median=(
            "slope_4w",
            "median",
        ),
        historical_slope_q75=(
            "slope_4w",
            lambda x:
                x.quantile(0.75),
        ),
        historical_slope_q90=(
            "slope_4w",
            lambda x:
                x.quantile(0.90),
        ),
        historical_slope_max=(
            "slope_4w",
            "max",
        ),
    )
)

In [384]:
pr_2024_trend_eval = (
    pr_trend
    .merge(
        trend_reference_pr_2024,
        on="epi_week",
        how="left",
    )
)

In [385]:
pr_2024_trend_eval[
    "slope_excess_q90"
] = (
    pr_2024_trend_eval[
        "slope_4w"
    ]
    - pr_2024_trend_eval[
        "historical_slope_q90"
    ]
)

In [386]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
            "slope_4w",
            "historical_slope_q90",
            "slope_excess_q90",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4,slope_4w,historical_slope_q90,slope_excess_q90
0,1,292,0.462025,NaN,False,NaN,NaN,NaN
1,2,280,0.597015,NaN,False,NaN,NaN,NaN
2,3,245,0.631443,NaN,False,NaN,NaN,NaN
3,4,224,0.647399,-0.088373,False,-0.092524,0.063564,-0.156088
4,5,247,0.705714,-0.045341,False,-0.046401,-0.048873,0.002472
5,6,281,0.677108,0.052013,False,0.050705,-0.009301,0.060006
6,7,289,0.455836,0.093061,True,0.088982,0.152116,-0.063134
7,8,438,0.700800,0.190197,True,0.174119,0.196141,-0.022023
8,9,475,0.708955,0.219590,True,0.198515,0.251946,-0.053431
9,10,492,0.774803,0.182086,True,0.167280,0.118725,0.048555


In [387]:
pr_2024_trend_eval[
    "trend_signal"
] = (
    pr_2024_trend_eval[
        "positive_growth_3of4"
    ]
    & (
        pr_2024_trend_eval[
            "slope_4w"
        ]
        >
        pr_2024_trend_eval[
            "historical_slope_q90"
        ]
    )
)

In [388]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(1, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "weekly_growth_rate",
            "positive_growth_3of4",
            "trend_signal",
        ],
    ]
)

,epi_week,record_count,final_ratio,weekly_growth_rate,positive_growth_3of4,trend_signal
0,1,292,0.462025,NaN,False,False
1,2,280,0.597015,NaN,False,False
2,3,245,0.631443,NaN,False,False
3,4,224,0.647399,-0.088373,False,False
4,5,247,0.705714,-0.045341,False,False
5,6,281,0.677108,0.052013,False,False
6,7,289,0.455836,0.093061,True,False
7,8,438,0.700800,0.190197,True,False
8,9,475,0.708955,0.219590,True,False
9,10,492,0.774803,0.182086,True,True


In [389]:
pr_2024_trend_eval[
    "log_ratio"
] = np.log(
    pr_2024_trend_eval[
        "final_ratio"
    ]
)

In [390]:
pr_2024_trend_eval[
    "ratio_slope_4w"
] = rolling_log_slope(
    pr_2024_trend_eval[
        "log_ratio"
    ],
    window=4,
)

In [391]:
pr_2024_trend_eval[
    "relative_growth_rate"
] = (
    np.exp(
        pr_2024_trend_eval[
            "ratio_slope_4w"
        ]
    )
    - 1
)

In [392]:
display(
    pr_2024_trend_eval.loc[
        pr_2024_trend_eval[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "ratio_slope_4w",
            "relative_growth_rate",
        ],
    ]
)

,epi_week,record_count,final_ratio,ratio_slope_4w,relative_growth_rate
5,6,281,0.677108,0.029572,0.030013
6,7,289,0.455836,-0.109387,-0.103616
7,8,438,0.700800,-0.041666,-0.040810
8,9,475,0.708955,0.056797,0.058441
9,10,492,0.774803,0.160300,0.173863
10,11,490,0.768025,0.036362,0.037031
11,12,474,0.679083,-0.013793,-0.013699
12,13,419,0.618907,-0.079704,-0.076610
13,14,498,0.732353,-0.023547,-0.023272
14,15,535,0.678934,0.016765,0.016906


In [393]:
pr_history_years = [
    2019,
    2022,
    2023,
]

In [394]:
historical_ratio_trend_frames = []

for target_year in pr_history_years:
    baseline_years = [
        year
        for year in pr_history_years
        if year != target_year
    ]

    target = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].eq(target_year)
        ][
            [
                "epi_year",
                "epi_week",
                "record_count",
            ]
        ]
        .copy()
    )

    baseline = (
        historical_pr.loc[
            historical_pr[
                "epi_year"
            ].isin(
                baseline_years
            )
        ]
        .groupby(
            "epi_week",
            as_index=False,
        )
        .agg(
            historical_median=(
                "record_count",
                "median",
            )
        )
    )

    target = target.merge(
        baseline,
        on="epi_week",
        how="left",
    )

    target["ratio"] = (
        target["record_count"]
        / target[
            "historical_median"
        ]
    )

    target["log_ratio"] = (
        np.log(target["ratio"])
    )

    target["ratio_slope_4w"] = (
        rolling_log_slope(
            target["log_ratio"],
            window=4,
        )
    )

    historical_ratio_trend_frames.append(
        target
    )

historical_ratio_trends = pd.concat(
    historical_ratio_trend_frames,
    ignore_index=True,
)

In [395]:
ratio_trend_reference_pr = (
    historical_ratio_trends
    .groupby(
        "epi_week",
        as_index=False,
    )
    .agg(
        ratio_slope_median=(
            "ratio_slope_4w",
            "median",
        ),
        ratio_slope_q75=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.75),
        ),
        ratio_slope_q90=(
            "ratio_slope_4w",
            lambda x:
                x.quantile(0.90),
        ),
        ratio_slope_max=(
            "ratio_slope_4w",
            "max",
        ),
    )
)

In [396]:
pr_2024_trend_v2 = (
    pr_2024_trend_eval
    .merge(
        ratio_trend_reference_pr,
        on="epi_week",
        how="left",
    )
)

In [397]:
pr_2024_trend_v2[
    "trend_signal_v2"
] = (
    pr_2024_trend_v2[
        "ratio_slope_4w"
    ]
    >
    pr_2024_trend_v2[
        "ratio_slope_q90"
    ]
)

In [398]:
pr_2024_trend_v2[
    "trend_signal_v2"
] = (
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ]
        >
        pr_2024_trend_v2[
            "ratio_slope_q90"
        ]
    )
    &
    (
        pr_2024_trend_v2[
            "ratio_slope_4w"
        ] > 0
    )
)

In [399]:
display(
    pr_2024_trend_v2.loc[
        pr_2024_trend_v2[
            "epi_week"
        ].between(6, 26),
        [
            "epi_week",
            "record_count",
            "final_ratio",
            "ratio_slope_4w",
            "ratio_slope_q90",
            "trend_signal_v2",
        ],
    ]
)

,epi_week,record_count,final_ratio,ratio_slope_4w,ratio_slope_q90,trend_signal_v2
5,6,281,0.677108,0.029572,0.148106,False
6,7,289,0.455836,-0.109387,0.334611,False
7,8,438,0.700800,-0.041666,0.406444,False
8,9,475,0.708955,0.056797,0.387962,False
9,10,492,0.774803,0.160300,0.276777,False
10,11,490,0.768025,0.036362,0.182304,False
11,12,474,0.679083,-0.013793,0.113195,False
12,13,419,0.618907,-0.079704,0.164356,False
13,14,498,0.732353,-0.023547,0.200871,False
14,15,535,0.678934,0.016765,0.128977,False
